# Data read

In [ ]:
"""
benchmark_fscore_corr_plots.py
绘制三层 F-score 和相关系数图
数据列名（已确认）:
  TF_region_per      → TF / Method / Precision / Recall / fscore
  TF_region_all      → Method / Precision / Recall / F_score
  Region_gene_per_corr → Method / Gene / Spearman_Rho
  Region_gene_total_precision → Method / Precision / Recall / F_score / AUC / F-beta
  TF_gene_per_corr   → Method / TF / Target / Correlation
  TF_gene_per_precision → Method / TF / Precision / Recall / F-beta
  TF_gene_total_precision → Method / Precision / Recall / AUC / F-beta
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/fscore_corr/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878", "HepG2", "IMR90", "K562", "MCF7", "A549", "H1", "HELA", "SK"]
METHOD_LIST = ["DyGMamba", "CellOracle", "FigR", "GLUE", "LINGER", "GRaNIE", "Pando"]
FOCAL       = "DyGMamba"

COLORS = {
    "DyGMamba":  "#9467BD",
    "GLUE":      "#00A087",
    "FigR":      "#3C5488",
    "CellOracle":"#F39B7F",
    "LINGER":    "#8491B4",
    "GRaNIE":    "#91D1C2",
    "Pando":     "#DC0000",
}
METHOD_ORDER = [m for m in METHOD_LIST]  # DyGMamba 在最前

FONT = dict(fontsize=11)
TITLE_FONT = dict(fontsize=12, fontweight='bold')

######################################################################
# 读取数据
######################################################################
def load_all(cell_types, data_root):
    store = {}
    for ct in cell_types:
        path = os.path.join(data_root, ct, "benchmarkV7", "benchmark_all_results.xlsx")
        if not os.path.exists(path):
            print(f"  [跳过] {ct}")
            continue
        xl = pd.ExcelFile(path)
        d  = {}
        for s in xl.sheet_names:
            try:
                df = xl.parse(s)
                # 统一 Method 列的大小写
                if 'Method' in df.columns:
                    df['Method'] = df['Method'].str.strip()
                d[s] = df
            except Exception as e:
                print(f"  [{ct}/{s}] {e}")
        store[ct] = d
        print(f"  ✓ {ct}: {list(d.keys())}")
    return store

print("读取数据...")
ALL = load_all(CELL_TYPES, data_root)
avail_ct = list(ALL.keys())
print(f"可用细胞系: {avail_ct}\n")

######################################################################
# 工具函数
######################################################################
def collect(sheet, col, all_data=ALL):
    """汇总所有细胞系某 sheet 的某列，返回 (CellType, Method, Value) DataFrame"""
    rows = []
    for ct, d in all_data.items():
        if sheet not in d:
            continue
        df = d[sheet]
        if 'Method' not in df.columns or col not in df.columns:
            continue
        for _, row in df[['Method', col]].dropna().iterrows():
            rows.append({'CellType': ct, 'Method': row['Method'], 'Value': row[col]})
    return pd.DataFrame(rows)

def method_mean(df):
    """每个 CellType × Method 取均值"""
    return df.groupby(['CellType','Method'])['Value'].mean().reset_index()

def style_ax(ax, title='', xlabel='', ylabel='', rotate_x=True):
    ax.set_title(title, **TITLE_FONT)
    ax.set_xlabel(xlabel, **FONT)
    ax.set_ylabel(ylabel, **FONT)
    ax.spines[['top','right']].set_visible(False)
    if rotate_x:
        ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

def highlight_focal(ax, order):
    """高亮 DyGMamba 的柱子背景"""
    if FOCAL in order:
        idx = list(order).index(FOCAL)
        ax.axvspan(idx - 0.45, idx + 0.45, alpha=0.07,
                   color=COLORS[FOCAL], zorder=0)

def add_sig_stars(ax, focal_vals, comp_vals_dict, order, y_offset=0.02):
    """在显著优于对比方法的位置标星"""
    y_max = ax.get_ylim()[1]
    for i, method in enumerate(order):
        if method == FOCAL or method not in comp_vals_dict:
            continue
        a = np.array(focal_vals)
        b = np.array(comp_vals_dict[method])
        n = min(len(a), len(b))
        if n < 4:
            continue
        try:
            _, p = stats.wilcoxon(a[:n], b[:n], alternative='greater')
            if p < 0.05:
                ax.text(i, y_max * (1 - y_offset), '*',
                        ha='center', fontsize=14,
                        color='#C00000', fontweight='bold')
        except Exception:
            pass

def get_order(df, method_col='Method'):
    present = df[method_col].unique()
    return [m for m in METHOD_ORDER if m in present]

######################################################################
# ── 收集数据 ──
######################################################################

# TF-Region per-TF fscore
tfr_per   = collect('TF_region_per',   'fscore')
tfr_all   = collect('TF_region_all',   'F_score')

# Region-Gene Spearman + precision
rg_corr   = collect('Region_gene_per_corr',        'Spearman_Rho')
rg_prec   = collect('Region_gene_total_precision',  'F_score')   # 整体 F-score
rg_prec_b = collect('Region_gene_total_precision',  'F-beta')    # F-beta

# TF-Gene correlation + precision
tfg_corr  = collect('TF_gene_per_corr',       'Correlation')
tfg_prec  = collect('TF_gene_per_precision',  'F-beta')         # per-TF F-beta
tfg_total = collect('TF_gene_total_precision','F-beta')          # 整体 F-beta


In [ ]:
for ct, d in ALL.items():
    print(f"\n=== {ct} ===")
    for sheet, df in d.items():
        print(f"  {sheet}: columns={list(df.columns)[:8]}, shape={df.shape}")
    break  # 只看第一个细胞系就够了

## correct 

In [ ]:
######################################################################
# 修正版数据收集（基于实际列名）
######################################################################

def collect(sheet, col, all_data=ALL):
    """汇总所有细胞系某 sheet 的某列"""
    rows = []
    for ct, d in all_data.items():
        if sheet not in d:
            continue
        df = d[sheet]
        if 'Method' not in df.columns or col not in df.columns:
            continue
        for _, row in df[['Method', col]].dropna().iterrows():
            rows.append({'CellType': ct, 'Method': row['Method'], 'Value': row[col]})
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=['CellType','Method','Value'])

def load_pkl_dict(pkl_path, method_key, value_col):
    """从字典型 pkl 加载数据，返回 (Method, Value) 列表"""
    rows = []
    if not os.path.exists(pkl_path):
        return rows
    try:
        obj = pd.read_pickle(pkl_path)
        if isinstance(obj, dict):
            for method, df in obj.items():
                if not isinstance(df, pd.DataFrame) or len(df) == 0:
                    continue
                if value_col in df.columns:
                    for v in df[value_col].dropna():
                        rows.append({'Method': method, 'Value': float(v)})
        elif isinstance(obj, pd.DataFrame):
            if 'Method' in obj.columns and value_col in obj.columns:
                for _, r in obj[['Method', value_col]].dropna().iterrows():
                    rows.append({'Method': r['Method'], 'Value': float(r[value_col])})
    except Exception as e:
        print(f"  [pkl 错误] {pkl_path}: {e}")
    return rows

# ── TF-Gene（列名已修正）──────────────────────────────────────────
# per-gene 相关系数：列名是 Correlation
tfg_corr  = collect('TF_gene_per_corr',       'Correlation')

# per-TF F-score：列名是 F_score（不是 F-beta）
tfg_prec  = collect('TF_gene_per_precision',  'F_score')

# 整体 F-beta
tfg_total = collect('TF_gene_total_precision', 'F-beta')

# 取绝对值
if len(tfg_corr) > 0:
    tfg_corr['Value'] = tfg_corr['Value'].abs()

# ── TF-Region（从 pkl 加载）─────────────────────────────────────────
tfr_per_rows, tfr_all_rows = [], []

for ct in avail_ct:
    base = f"/home/wuyan/dygmamba_project/data/cell_line/{ct}/benchmarkV7/"

    # per-TF fscore（字典：method → DataFrame[TF, Precision, Recall, fscore]）
    for fname in ["tf_region_result.pkl", "tf_region_per.pkl", "method_result.pkl"]:
        path = base + fname
        rows = load_pkl_dict(path, method_key='Method', value_col='fscore')
        if rows:
            for r in rows:
                tfr_per_rows.append({'CellType': ct, **r})
            print(f"  ✓ {ct} TF-Region per-TF 从 {fname}")
            break

    # 整体 F_score（DataFrame[Method, Precision, Recall, F_score]）
    for fname in ["tf_region_all.pkl", "all_tf_region_result.pkl", "overall_metrics.pkl"]:
        path = base + fname
        rows = load_pkl_dict(path, method_key='Method', value_col='F_score')
        if rows:
            for r in rows:
                tfr_all_rows.append({'CellType': ct, **r})
            print(f"  ✓ {ct} TF-Region overall 从 {fname}")
            break

tfr_per = pd.DataFrame(tfr_per_rows) if tfr_per_rows else \
          pd.DataFrame(columns=['CellType','Method','Value'])
tfr_all = pd.DataFrame(tfr_all_rows) if tfr_all_rows else \
          pd.DataFrame(columns=['CellType','Method','Value'])

# ── Region-Gene（从 pkl 加载）────────────────────────────────────────
rg_corr_rows, rg_prec_rows = [], []

for ct in avail_ct:
    base = f"/home/wuyan/dygmamba_project/data/cell_line/{ct}/benchmarkV7/"

    # per-gene Spearman（DataFrame[Method, Gene, Spearman_Rho]）
    for fname in ["region_gene_per_corr.pkl", "rg_corr.pkl",
                  "region_gene_corr_df.pkl", "per_corr_df.pkl"]:
        path = base + fname
        if not os.path.exists(path):
            continue
        try:
            df = pd.read_pickle(path)
            if isinstance(df, pd.DataFrame) and \
               'Method' in df.columns and 'Spearman_Rho' in df.columns:
                for _, r in df[['Method','Spearman_Rho']].dropna().iterrows():
                    rg_corr_rows.append({'CellType': ct,
                                         'Method': r['Method'],
                                         'Value': abs(float(r['Spearman_Rho']))})
                print(f"  ✓ {ct} Region-Gene corr 从 {fname}")
                break
        except Exception as e:
            print(f"  [pkl 错误] {path}: {e}")

    # 整体 precision/F-score
    for fname in ["region_gene_total_precision.pkl", "rg_precision.pkl",
                  "region_gene_precision.pkl"]:
        path = base + fname
        for col in ['F_score', 'F-beta', 'fscore']:
            rows = load_pkl_dict(path, method_key='Method', value_col=col)
            if rows:
                for r in rows:
                    rg_prec_rows.append({'CellType': ct, **r})
                print(f"  ✓ {ct} Region-Gene prec 从 {fname}[{col}]")
                break
        if rg_prec_rows:
            break

rg_corr = pd.DataFrame(rg_corr_rows) if rg_corr_rows else \
          pd.DataFrame(columns=['CellType','Method','Value'])
rg_prec = pd.DataFrame(rg_prec_rows) if rg_prec_rows else \
          pd.DataFrame(columns=['CellType','Method','Value'])

# ── 状态汇总 ────────────────────────────────────────────────────────
print("\n=== 数据状态 ===")
for name, df in [
    ('TF-Region per-TF fscore',   tfr_per),
    ('TF-Region overall F_score', tfr_all),
    ('Region-Gene |Spearman ρ|',  rg_corr),
    ('Region-Gene F-score',       rg_prec),
    ('TF-Gene Correlation',       tfg_corr),
    ('TF-Gene per-TF F_score',    tfg_prec),
    ('TF-Gene overall F-beta',    tfg_total),
]:
    if len(df) > 0:
        methods = sorted(df['Method'].unique().tolist())
        cts     = sorted(df['CellType'].unique().tolist())
        print(f"  {'✓':2s} {name:35s}: {len(df):5d} 行 | methods={methods}")
    else:
        print(f"  {'⚠':2s} {name:35s}: 空（pkl 未找到）")

In [ ]:
import os
ct = "A549"
base = f"/home/wuyan/dygmamba_project/data/cell_line/{ct}/benchmarkV7/"
print(f"benchmarkV7 目录下所有文件:")
for f in sorted(os.listdir(base)):
    size = os.path.getsize(os.path.join(base, f)) // 1024
    print(f"  {f}  ({size} KB)")

In [ ]:

# 取绝对值（相关系数）
rg_corr['Value']  = rg_corr['Value'].abs()
tfg_corr['Value'] = tfg_corr['Value'].abs()

for df, name in [(tfr_per,'TF-Region per'), (tfr_all,'TF-Region all'),
                 (rg_corr,'RG corr'), (rg_prec,'RG prec'),
                 (tfg_corr,'TFG corr'), (tfg_prec,'TFG prec')]:
    print(f"{name}: {len(df)} 行")

######################################################################
# ── Figure 1: TF-Region F-score (柱状图 + 箱线图，共 2 行) ──
######################################################################
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Region: F0.1 Score Comparison', **TITLE_FONT, fontsize=14)

# ---- A: 跨细胞系箱线图 (per-TF fscore) ----
ax = axes[0, 0]
order_a = get_order(tfr_per)
pal_a   = {m: COLORS.get(m,'#999') for m in order_a}
sns.boxplot(data=tfr_per, x='Method', y='Value', order=order_a,
            palette=pal_a, ax=ax, width=0.55,
            flierprops={'marker':'o','markersize':3,'alpha':0.4},
            linewidth=1.2)
sns.stripplot(data=tfr_per, x='Method', y='Value', order=order_a,
              palette=pal_a, ax=ax, size=3, alpha=0.35, jitter=True)
highlight_focal(ax, order_a)
style_ax(ax, 'Per-TF F0.1 (all cell types)', ylabel='F0.1 Score')

# 显著性星号
focal_per_vals = tfr_per[tfr_per['Method']==FOCAL]['Value'].values
comp_dict_per  = {m: tfr_per[tfr_per['Method']==m]['Value'].values
                  for m in order_a if m != FOCAL}
add_sig_stars(ax, focal_per_vals, comp_dict_per, order_a)

# ---- B: 每个细胞系 DyGMamba vs Best-Other 对比柱状图 ----
ax = axes[0, 1]
bar_data = []
for ct in avail_ct:
    sub = tfr_per[tfr_per['CellType']==ct].groupby('Method')['Value'].mean()
    if FOCAL not in sub.index:
        continue
    others = {m: sub[m] for m in order_a if m != FOCAL and m in sub.index}
    if not others:
        continue
    best_other_m = max(others, key=others.get)
    bar_data.append({
        'CellType': ct,
        FOCAL: sub[FOCAL],
        'Best Other': others[best_other_m],
        'Best Other Method': best_other_m,
    })
bar_df = pd.DataFrame(bar_data).set_index('CellType')
x = np.arange(len(bar_df))
w = 0.35
bars1 = ax.bar(x - w/2, bar_df[FOCAL], w, color=COLORS[FOCAL],
               label=FOCAL, alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + w/2, bar_df['Best Other'], w,
               color=[COLORS.get(bar_df.loc[ct,'Best Other Method'],'#999')
                      for ct in bar_df.index],
               alpha=0.75, edgecolor='white', label='Best Competitor')
ax.set_xticks(x)
ax.set_xticklabels(bar_df.index, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=9)
style_ax(ax, 'DyGMamba vs Best Competitor per Cell Type',
         ylabel='Mean F0.1', rotate_x=False)

# ---- C: 整体 F_score 跨细胞系箱线图 ----
ax = axes[1, 0]
order_c = get_order(tfr_all)
pal_c   = {m: COLORS.get(m,'#999') for m in order_c}
sns.boxplot(data=tfr_all, x='Method', y='Value', order=order_c,
            palette=pal_c, ax=ax, width=0.55, linewidth=1.2,
            flierprops={'marker':'o','markersize':4,'alpha':0.5})
sns.stripplot(data=tfr_all, x='Method', y='Value', order=order_c,
              palette=pal_c, ax=ax, size=5, alpha=0.6, jitter=0.15)
highlight_focal(ax, order_c)
style_ax(ax, 'Overall Macro F0.1 (per cell type)', ylabel='Macro F_score')
for ct_i, (ct, row) in enumerate(
    tfr_all.groupby('CellType').apply(
        lambda g: g.set_index('Method')['Value']).iterrows()
):
    for m in order_c:
        if m in row.index:
            pass  # 点已经在 stripplot 里了

# ---- D: 热图 (method × cell type, macro F_score) ----
ax = axes[1, 1]
pivot = tfr_all.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
order_d = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL] if m in pivot.index]
ct_ord  = [c for c in avail_ct if c in pivot.columns]
pivot   = pivot.loc[order_d, ct_ord]
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', annot=True, fmt='.3f',
            annot_kws={'size':8}, linewidths=0.3,
            cbar_kws={'label':'Macro F0.1','shrink':0.8})
for j in range(len(ct_ord)):
    ax.add_patch(plt.Rectangle((j, 0), 1, 1,
                 fill=False, edgecolor=COLORS[FOCAL], lw=2.2))
style_ax(ax, 'Macro F0.1 Heatmap (Method × Cell Type)',
         rotate_x=True)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f"{output_dir}Fig1_TF_Region_Fscore.png", dpi=200, bbox_inches='tight')
plt.close()
print("✓ Fig1_TF_Region_Fscore.png")

######################################################################
# ── Figure 2: Region-Gene 相关系数 + F-score ──
######################################################################
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Region-Gene: Spearman Correlation & F-score', **TITLE_FONT, fontsize=14)

# ---- A: 跨细胞系 Spearman 箱线图 ----
ax = axes[0, 0]
order_rg = get_order(rg_corr)
pal_rg   = {m: COLORS.get(m,'#999') for m in order_rg}
sns.boxplot(data=rg_corr, x='Method', y='Value', order=order_rg,
            palette=pal_rg, ax=ax, width=0.55, linewidth=1.2,
            flierprops={'marker':'o','markersize':3,'alpha':0.4})
sns.stripplot(data=rg_corr, x='Method', y='Value', order=order_rg,
              palette=pal_rg, ax=ax, size=3, alpha=0.3, jitter=True)
highlight_focal(ax, order_rg)
focal_rg = rg_corr[rg_corr['Method']==FOCAL]['Value'].values
comp_rg  = {m: rg_corr[rg_corr['Method']==m]['Value'].values for m in order_rg if m!=FOCAL}
add_sig_stars(ax, focal_rg, comp_rg, order_rg)
style_ax(ax, 'Per-Gene |Spearman ρ| (all cell types)', ylabel='|Spearman ρ|')

# ---- B: 每细胞系平均 Spearman 柱状图 ----
ax = axes[0, 1]
rg_mean = rg_corr.groupby(['CellType','Method'])['Value'].mean().unstack(fill_value=0)
ct_plot = [c for c in avail_ct if c in rg_mean.index]
x = np.arange(len(ct_plot))
n_m = len(order_rg)
w   = 0.8 / n_m
for i, method in enumerate(order_rg):
    if method not in rg_mean.columns:
        continue
    vals = [rg_mean.loc[ct, method] if ct in rg_mean.index else 0
            for ct in ct_plot]
    lw  = 1.5 if method == FOCAL else 0.8
    ec  = COLORS[FOCAL] if method == FOCAL else 'white'
    ax.bar(x + (i - n_m/2 + 0.5) * w, vals, w,
           color=COLORS.get(method,'#999'), alpha=0.85,
           edgecolor=ec, linewidth=lw, label=method)
ax.set_xticks(x)
ax.set_xticklabels(ct_plot, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=7, ncol=2)
style_ax(ax, 'Mean |Spearman ρ| per Cell Type',
         ylabel='Mean |ρ|', rotate_x=False)

# ---- C: Region-Gene F-score 箱线图（整体，跨细胞系）----
ax = axes[1, 0]
# 优先取 F-beta，没有就取 F_score
rg_f = rg_prec_b if len(rg_prec_b) > 0 else rg_prec
order_rgf = get_order(rg_f)
pal_rgf   = {m: COLORS.get(m,'#999') for m in order_rgf}
sns.boxplot(data=rg_f, x='Method', y='Value', order=order_rgf,
            palette=pal_rgf, ax=ax, width=0.55, linewidth=1.2)
sns.stripplot(data=rg_f, x='Method', y='Value', order=order_rgf,
              palette=pal_rgf, ax=ax, size=5, alpha=0.65, jitter=0.15)
highlight_focal(ax, order_rgf)
focal_rgf = rg_f[rg_f['Method']==FOCAL]['Value'].values
comp_rgf  = {m: rg_f[rg_f['Method']==m]['Value'].values for m in order_rgf if m!=FOCAL}
add_sig_stars(ax, focal_rgf, comp_rgf, order_rgf)
style_ax(ax, 'Region-Gene F-score (overall per cell type)', ylabel='F-score')

# ---- D: Spearman 热图 ----
ax = axes[1, 1]
pivot_rg = rg_corr.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
ord_d = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL] if m in pivot_rg.index]
ct_d  = [c for c in avail_ct if c in pivot_rg.columns]
pivot_rg = pivot_rg.loc[ord_d, ct_d]
sns.heatmap(pivot_rg, ax=ax, cmap='PuBuGn', annot=True, fmt='.3f',
            annot_kws={'size':8}, linewidths=0.3,
            cbar_kws={'label':'Mean |Spearman ρ|','shrink':0.8})
for j in range(len(ct_d)):
    ax.add_patch(plt.Rectangle((j, 0), 1, 1,
                 fill=False, edgecolor=COLORS[FOCAL], lw=2.2))
style_ax(ax, '|Spearman ρ| Heatmap (Method × Cell Type)', rotate_x=True)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f"{output_dir}Fig2_Region_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close()
print("✓ Fig2_Region_Gene_Corr_Fscore.png")

######################################################################
# ── Figure 3: TF-Gene 相关系数 + F-score ──
######################################################################
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Gene: Correlation & F-score', **TITLE_FONT, fontsize=14)

# ---- A: 跨细胞系 Correlation 箱线图 ----
ax = axes[0, 0]
order_tgc = get_order(tfg_corr)
pal_tgc   = {m: COLORS.get(m,'#999') for m in order_tgc}
sns.boxplot(data=tfg_corr, x='Method', y='Value', order=order_tgc,
            palette=pal_tgc, ax=ax, width=0.55, linewidth=1.2,
            flierprops={'marker':'o','markersize':3,'alpha':0.4})
sns.stripplot(data=tfg_corr, x='Method', y='Value', order=order_tgc,
              palette=pal_tgc, ax=ax, size=3, alpha=0.3, jitter=True)
highlight_focal(ax, order_tgc)
focal_tgc = tfg_corr[tfg_corr['Method']==FOCAL]['Value'].values
comp_tgc  = {m: tfg_corr[tfg_corr['Method']==m]['Value'].values
             for m in order_tgc if m!=FOCAL}
add_sig_stars(ax, focal_tgc, comp_tgc, order_tgc)
style_ax(ax, 'TF-Gene Correlation (all cell types)', ylabel='|Correlation|')

# ---- B: 每细胞系平均 Correlation 柱状图 ----
ax = axes[0, 1]
tgc_mean = tfg_corr.groupby(['CellType','Method'])['Value'].mean().unstack(fill_value=0)
ct_tgc   = [c for c in avail_ct if c in tgc_mean.index]
x = np.arange(len(ct_tgc))
n_m = len(order_tgc)
w   = 0.8 / n_m
for i, method in enumerate(order_tgc):
    if method not in tgc_mean.columns:
        continue
    vals = [tgc_mean.loc[ct, method] if ct in tgc_mean.index else 0
            for ct in ct_tgc]
    lw = 1.5 if method == FOCAL else 0.8
    ec = COLORS[FOCAL] if method == FOCAL else 'white'
    ax.bar(x + (i - n_m/2 + 0.5) * w, vals, w,
           color=COLORS.get(method,'#999'), alpha=0.85,
           edgecolor=ec, linewidth=lw, label=method)
ax.set_xticks(x)
ax.set_xticklabels(ct_tgc, rotation=30, ha='right', fontsize=9)
ax.legend(fontsize=7, ncol=2)
style_ax(ax, 'Mean Correlation per Cell Type',
         ylabel='Mean |Corr|', rotate_x=False)

# ---- C: TF-Gene F-score 箱线图 ----
ax = axes[1, 0]
order_tgf = get_order(tfg_prec)
pal_tgf   = {m: COLORS.get(m,'#999') for m in order_tgf}
sns.boxplot(data=tfg_prec, x='Method', y='Value', order=order_tgf,
            palette=pal_tgf, ax=ax, width=0.55, linewidth=1.2)
sns.stripplot(data=tfg_prec, x='Method', y='Value', order=order_tgf,
              palette=pal_tgf, ax=ax, size=3, alpha=0.4, jitter=True)
highlight_focal(ax, order_tgf)
focal_tgf = tfg_prec[tfg_prec['Method']==FOCAL]['Value'].values
comp_tgf  = {m: tfg_prec[tfg_prec['Method']==m]['Value'].values
             for m in order_tgf if m!=FOCAL}
add_sig_stars(ax, focal_tgf, comp_tgf, order_tgf)
style_ax(ax, 'Per-TF F-beta Score (all cell types)', ylabel='F-beta Score')

# ---- D: TF-Gene F-score 热图 ----
ax = axes[1, 1]
pivot_tgf = tfg_prec.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
ord_tgfd  = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL]
             if m in pivot_tgf.index]
ct_tgfd   = [c for c in avail_ct if c in pivot_tgf.columns]
pivot_tgf = pivot_tgf.loc[ord_tgfd, ct_tgfd]
sns.heatmap(pivot_tgf, ax=ax, cmap='OrRd', annot=True, fmt='.3f',
            annot_kws={'size':8}, linewidths=0.3,
            cbar_kws={'label':'Mean F-beta','shrink':0.8})
for j in range(len(ct_tgfd)):
    ax.add_patch(plt.Rectangle((j, 0), 1, 1,
                 fill=False, edgecolor=COLORS[FOCAL], lw=2.2))
style_ax(ax, 'TF-Gene F-beta Heatmap (Method × Cell Type)', rotate_x=True)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f"{output_dir}Fig3_TF_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close()
print("✓ Fig3_TF_Gene_Corr_Fscore.png")

######################################################################
# ── Figure 4: 三层合并大图 (论文主图) ──
# 布局: 3列(层) × 2行(箱线 + 热图)
######################################################################
fig = plt.figure(figsize=(24, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 行0: 三层的跨细胞系箱线图（F-score / Corr）
# 行1: 三层的热图

layer_configs = [
    # (sheet_data, col, title_box, title_heat, cmap, ylabel)
    (tfr_per,  'TF-Region F0.1',         'TF-Region F0.1 Heatmap',  'YlOrRd', 'F0.1 Score'),
    (rg_corr,  'Region-Gene |Spearman ρ|','Region-Gene |ρ| Heatmap', 'PuBuGn', '|Spearman ρ|'),
    (tfg_corr, 'TF-Gene Correlation',    'TF-Gene Corr Heatmap',    'YlGnBu', '|Correlation|'),
]

for col_i, (df, title_b, title_h, cmap, ylabel) in enumerate(layer_configs):
    order_i = get_order(df)
    pal_i   = {m: COLORS.get(m,'#999') for m in order_i}

    # ── 箱线图 ──
    ax_box = fig.add_subplot(gs[0, col_i])
    sns.boxplot(data=df, x='Method', y='Value', order=order_i,
                palette=pal_i, ax=ax_box, width=0.55, linewidth=1.3,
                flierprops={'marker':'o','markersize':3,'alpha':0.35})
    sns.stripplot(data=df, x='Method', y='Value', order=order_i,
                  palette=pal_i, ax=ax_box, size=3.5, alpha=0.35, jitter=True)
    highlight_focal(ax_box, order_i)

    focal_v = df[df['Method']==FOCAL]['Value'].values
    comp_v  = {m: df[df['Method']==m]['Value'].values for m in order_i if m!=FOCAL}
    add_sig_stars(ax_box, focal_v, comp_v, order_i, y_offset=0.03)
    style_ax(ax_box, title_b, ylabel=ylabel)

    # 中位数标注
    medians = df.groupby('Method')['Value'].median()
    for j, method in enumerate(order_i):
        if method in medians.index:
            ax_box.text(j, medians[method], f'{medians[method]:.3f}',
                        ha='center', va='bottom', fontsize=7,
                        color='#333333', fontweight='bold')

    # ── 热图 ──
    ax_heat = fig.add_subplot(gs[1, col_i])
    pivot_i = df.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
    ord_hi  = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL]
               if m in pivot_i.index]
    ct_hi   = [c for c in avail_ct if c in pivot_i.columns]
    if len(ord_hi) > 0 and len(ct_hi) > 0:
        pivot_i = pivot_i.loc[ord_hi, ct_hi]
        sns.heatmap(pivot_i, ax=ax_heat, cmap=cmap,
                    annot=True, fmt='.3f', annot_kws={'size':8},
                    linewidths=0.3,
                    cbar_kws={'label': ylabel, 'shrink':0.75})
        # 高亮 DyGMamba 行
        for j in range(len(ct_hi)):
            ax_heat.add_patch(plt.Rectangle((j, 0), 1, 1,
                              fill=False, edgecolor=COLORS[FOCAL], lw=2.0))
    style_ax(ax_heat, title_h, rotate_x=True)
    ax_heat.tick_params(axis='y', rotation=0, labelsize=9)

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS.get(m,'#999'), label=m,
          alpha=0.9 if m==FOCAL else 0.75,
          linewidth=2 if m==FOCAL else 0.5,
          edgecolor=COLORS[FOCAL] if m==FOCAL else 'none')
    for m in METHOD_ORDER
]
fig.legend(handles=legend_elements, loc='lower center',
           ncol=len(METHOD_ORDER), fontsize=10,
           bbox_to_anchor=(0.5, -0.03),
           title='Methods', title_fontsize=11)

fig.suptitle('DyGMamba Benchmark: F-score & Correlation Across 3 Regulatory Layers\n'
             f'({len(avail_ct)} cell types, * = Wilcoxon p < 0.05)',
             fontsize=14, fontweight='bold', y=1.01)

plt.savefig(f"{output_dir}Fig4_Main_Fscore_Corr_Combined.png",
            dpi=200, bbox_inches='tight')
plt.close()
print("✓ Fig4_Main_Fscore_Corr_Combined.png")

######################################################################
# 数字摘要
######################################################################
print("\n" + "="*60)
print("数字摘要（DyGMamba 中位数 vs 各方法）")
print("="*60)
for name, df in [('TF-Region F0.1', tfr_per),
                  ('Region-Gene |ρ|', rg_corr),
                  ('TF-Gene Corr',    tfg_corr)]:
    print(f"\n── {name} ──")
    summary = df.groupby('Method')['Value'].agg(['median','mean','std'])
    order_s = [m for m in METHOD_ORDER if m in summary.index]
    print(summary.loc[order_s].round(4).to_string())
    focal_med = summary.loc[FOCAL,'median'] if FOCAL in summary.index else np.nan
    for m in order_s:
        if m == FOCAL: continue
        delta = focal_med - summary.loc[m,'median']
        print(f"  {FOCAL} vs {m:>12s}: Δmedian = {delta:+.4f}")

print(f"\n图已保存到: {output_dir}")

# V2

In [ ]:
"""
benchmark_recompute_and_plot.py
直接调用分析函数重新计算 TF-Region 和 Region-Gene，不依赖 Excel
"""
import sys
sys.path.append('/home/wuyan/dygmamba_project/model/dygmamba/src/')

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from benchmark.benchmark_model_new import (
    analysis_tf_region, calc_overall_metrics,
    analysis_region_gene, analysis_region_gene_precision,
    analysis_tf_gene_data, dyg_tf_gene_result_inherit
)

######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/fscore_corr/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878", "HepG2", "IMR90", "K562", "MCF7", "A549", "H1", "HELA", "SK"]
METHOD_LIST = ["DyGMamba", "CellOracle", "FigR", "GLUE", "LINGER", "GRaNIE", "Pando"]
FOCAL       = "DyGMamba"
BETA        = 0.1

COLORS = {
    "DyGMamba":  "#9467BD",
    "GLUE":      "#00A087",
    "FigR":      "#3C5488",
    "CellOracle":"#F39B7F",
    "LINGER":    "#8491B4",
    "GRaNIE":    "#91D1C2",
    "Pando":     "#DC0000",
}
METHOD_ORDER = METHOD_LIST.copy()

######################################################################
# Step 1: 对每个细胞系重新计算并收集结果
######################################################################
# 存放跨细胞系的数据
tfr_per_rows  = []   # TF-Region per-TF fscore
tfr_all_rows  = []   # TF-Region macro F_score
rg_corr_rows  = []   # Region-Gene Spearman_Rho
rg_prec_rows  = []   # Region-Gene F-score
tfg_corr_rows = []   # TF-Gene Correlation
tfg_prec_rows = []   # TF-Gene per-TF F_score
tfg_tot_rows  = []   # TF-Gene overall F-beta

for ct in CELL_TYPES:
    data_path   = f"{data_root}{ct}/"
    bench_path  = f"{data_path}benchmarkV7/"
    unibind_file = f"{bench_path}unibind_df.pkl"

    if not os.path.exists(unibind_file):
        print(f"[跳过] {ct}: 找不到 {unibind_file}")
        continue

    print(f"\n{'='*50}")
    print(f"处理: {ct}")
    print(f"{'='*50}")

    # ── TF-Region ─────────────────────────────────────
    try:
        tf_region_result, _ = analysis_tf_region(
            unibind_file, data_path, BETA, METHOD_LIST)
        all_tf_region_result = calc_overall_metrics(tf_region_result, beta=BETA)

        for method, df in tf_region_result.items():
            if len(df) == 0: continue
            for v in df['fscore'].dropna():
                tfr_per_rows.append({'CellType': ct, 'Method': method, 'Value': float(v)})

        for _, row in all_tf_region_result.iterrows():
            tfr_all_rows.append({'CellType': ct,
                                  'Method':   row['Method'],
                                  'Value':    float(row['F_score'])})
        print(f"  ✓ TF-Region: {len(tf_region_result)} methods")

    except Exception as e:
        print(f"  [TF-Region 错误] {ct}: {e}")

    # ── Region-Gene ────────────────────────────────────
    try:
        rg_total_corr, rg_per_corr_df, rg_method_data, bench_peak_gene_df = \
            analysis_region_gene(data_path, bench_path, METHOD_LIST,
                                 hic_threshold=140, hic_threshold_lower=30,
                                 self_threshold=0.7)

        rg_total_prec, rg_per_prec, _ = analysis_region_gene_precision(
            bench_peak_gene_df, rg_method_data, bench_path, COLORS, beta=BETA)

        # per-gene Spearman
        for _, row in rg_per_corr_df[['Method','Spearman_Rho']].dropna().iterrows():
            rg_corr_rows.append({'CellType': ct,
                                  'Method':   row['Method'],
                                  'Value':    abs(float(row['Spearman_Rho']))})

        # overall F-score/F-beta
        fscore_col = None
        for c in ['F-beta', 'F_score', 'fscore']:
            if c in rg_total_prec.columns:
                fscore_col = c
                break
        if fscore_col:
            for _, row in rg_total_prec[['Method', fscore_col]].dropna().iterrows():
                rg_prec_rows.append({'CellType': ct,
                                      'Method':   row['Method'],
                                      'Value':    float(row[fscore_col])})

        print(f"  ✓ Region-Gene: {rg_per_corr_df['Method'].nunique()} methods, "
              f"{len(rg_per_corr_df)} gene-pairs")

    except Exception as e:
        print(f"  [Region-Gene 错误] {ct}: {e}")

    # ── TF-Gene ────────────────────────────────────────
    try:
        model_result_path = f"{data_path}data_dyg/"
        _ = dyg_tf_gene_result_inherit(data_path, model_result_path)

        corr_dict, _, prec_df, per_prec_dict = analysis_tf_gene_data(
            data_path, METHOD_LIST, flag_corr=True, flag_precision=True, beta=BETA)

        # per-gene correlation
        for method, df in corr_dict.items():
            if len(df) == 0: continue
            col = 'Correlation' if 'Correlation' in df.columns else df.columns[-1]
            for v in df[col].dropna():
                tfg_corr_rows.append({'CellType': ct, 'Method': method,
                                       'Value': abs(float(v))})

        # per-TF F_score
        for method, df in per_prec_dict.items():
            if len(df) == 0: continue
            for c in ['F_score', 'F-beta', 'fscore']:
                if c in df.columns:
                    for v in df[c].dropna():
                        tfg_prec_rows.append({'CellType': ct, 'Method': method,
                                               'Value': float(v)})
                    break

        # overall F-beta
        if prec_df is not None and len(prec_df) > 0:
            for c in ['F-beta', 'F_score', 'fscore']:
                if c in prec_df.columns:
                    for _, row in prec_df[['Method', c]].dropna().iterrows():
                        tfg_tot_rows.append({'CellType': ct,
                                              'Method':   row['Method'],
                                              'Value':    float(row[c])})
                    break

        print(f"  ✓ TF-Gene: {len(corr_dict)} methods")

    except Exception as e:
        print(f"  [TF-Gene 错误] {ct}: {e}")

# ── 转为 DataFrame ─────────────────────────────────
tfr_per  = pd.DataFrame(tfr_per_rows)  if tfr_per_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
tfr_all  = pd.DataFrame(tfr_all_rows)  if tfr_all_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
rg_corr  = pd.DataFrame(rg_corr_rows)  if rg_corr_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
rg_prec  = pd.DataFrame(rg_prec_rows)  if rg_prec_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_corr = pd.DataFrame(tfg_corr_rows) if tfg_corr_rows else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_prec = pd.DataFrame(tfg_prec_rows) if tfg_prec_rows else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_tot  = pd.DataFrame(tfg_tot_rows)  if tfg_tot_rows  else pd.DataFrame(columns=['CellType','Method','Value'])

avail_ct = sorted(set(
    tfr_per['CellType'].tolist() + rg_corr['CellType'].tolist() +
    tfg_corr['CellType'].tolist()
))

print(f"\n=== 数据状态 ===")
for name, df in [('TF-Region per fscore', tfr_per), ('TF-Region macro F', tfr_all),
                  ('Region-Gene |ρ|', rg_corr),     ('Region-Gene F', rg_prec),
                  ('TF-Gene Corr', tfg_corr),        ('TF-Gene per-TF F', tfg_prec),
                  ('TF-Gene overall F-beta', tfg_tot)]:
    status = f"{len(df)} 行" if len(df) > 0 else "⚠ 空"
    print(f"  {name:30s}: {status}")

# ── 同时保存修复后的 Excel（供下次直接使用）──────────────
save_excel = os.path.join(output_dir, "benchmark_all_results_fixed.xlsx")
with pd.ExcelWriter(save_excel, engine='openpyxl') as writer:
    for name, df in [
        ('TF_region_per',              tfr_per),
        ('TF_region_all',              tfr_all),
        ('Region_gene_per_corr',       rg_corr),
        ('Region_gene_prec',           rg_prec),
        ('TF_gene_per_corr',           tfg_corr),
        ('TF_gene_per_precision',      tfg_prec),
        ('TF_gene_total_precision',    tfg_tot),
    ]:
        if len(df) > 0:
            df.to_excel(writer, sheet_name=name, index=False)
print(f"✓ 汇总 Excel 已保存: {save_excel}")

######################################################################
# Step 2: 绘图工具函数
######################################################################
FONT       = dict(fontsize=11)
TITLE_FONT = dict(fontsize=12, fontweight='bold')

def get_order(df):
    present = df['Method'].unique()
    return [m for m in METHOD_ORDER if m in present]

def style_ax(ax, title='', xlabel='', ylabel='', rotate_x=True):
    ax.set_title(title, **TITLE_FONT)
    ax.set_xlabel(xlabel, **FONT)
    ax.set_ylabel(ylabel, **FONT)
    ax.spines[['top','right']].set_visible(False)
    if rotate_x:
        ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

def highlight_focal(ax, order):
    if FOCAL in order:
        ax.axvspan(list(order).index(FOCAL) - 0.45,
                   list(order).index(FOCAL) + 0.45,
                   alpha=0.07, color=COLORS[FOCAL], zorder=0)

def add_sig_stars(ax, focal_vals, comp_dict, order, y_offset=0.02):
    if len(focal_vals) < 4: return
    y_max = ax.get_ylim()[1]
    for i, method in enumerate(order):
        if method == FOCAL or method not in comp_dict: continue
        b = comp_dict[method]
        n = min(len(focal_vals), len(b))
        if n < 4: continue
        try:
            _, p = stats.wilcoxon(focal_vals[:n], b[:n], alternative='greater')
            if p < 0.05:
                ax.text(i, y_max*(1-y_offset), '*',
                        ha='center', fontsize=14, color='#C00000', fontweight='bold')
        except Exception:
            pass

def draw_boxplot(ax, df, title, ylabel):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title, ylabel=ylabel); return
    order = get_order(df)
    pal   = {m: COLORS.get(m,'#999') for m in order}
    sns.boxplot(data=df, x='Method', y='Value', order=order, palette=pal,
                ax=ax, width=0.55, linewidth=1.2,
                flierprops={'marker':'o','markersize':3,'alpha':0.4})
    sns.stripplot(data=df, x='Method', y='Value', order=order, palette=pal,
                  ax=ax, size=3.5, alpha=0.35, jitter=True)
    highlight_focal(ax, order)
    focal_v = df[df['Method']==FOCAL]['Value'].values
    comp_v  = {m: df[df['Method']==m]['Value'].values for m in order if m!=FOCAL}
    add_sig_stars(ax, focal_v, comp_v, order)
    medians = df.groupby('Method')['Value'].median()
    for j, m in enumerate(order):
        if m in medians.index:
            ax.text(j, medians[m], f'{medians[m]:.3f}',
                    ha='center', va='bottom', fontsize=7, color='#333')
    style_ax(ax, title, ylabel=ylabel)

def draw_grouped_bar(ax, df, title, ylabel):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title, ylabel=ylabel, rotate_x=False); return
    order  = get_order(df)
    mean_df = df.groupby(['CellType','Method'])['Value'].mean().unstack(fill_value=0)
    ct_list = [c for c in avail_ct if c in mean_df.index]
    x = np.arange(len(ct_list))
    n_m, w = len(order), 0.8/len(order)
    for i, method in enumerate(order):
        if method not in mean_df.columns: continue
        vals = [mean_df.loc[ct, method] if ct in mean_df.index else 0 for ct in ct_list]
        ax.bar(x+(i-n_m/2+0.5)*w, vals, w,
               color=COLORS.get(method,'#999'), alpha=0.85,
               edgecolor=COLORS[FOCAL] if method==FOCAL else 'white',
               linewidth=1.5 if method==FOCAL else 0.6, label=method)
    ax.set_xticks(x)
    ax.set_xticklabels(ct_list, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=7, ncol=2, loc='upper right')
    style_ax(ax, title, ylabel=ylabel, rotate_x=False)

def draw_heatmap(ax, df, title, ylabel, cmap='YlOrRd'):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title); return
    pivot = df.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
    ord_h = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL] if m in pivot.index]
    ct_h  = [c for c in avail_ct if c in pivot.columns]
    if not ord_h or not ct_h: return
    pivot = pivot.loc[ord_h, ct_h]
    sns.heatmap(pivot, ax=ax, cmap=cmap, annot=True, fmt='.3f',
                annot_kws={'size':8}, linewidths=0.3,
                cbar_kws={'label':ylabel,'shrink':0.8})
    for j in range(len(ct_h)):
        ax.add_patch(plt.Rectangle((j,0),1,1,fill=False,
                     edgecolor=COLORS[FOCAL],lw=2.2))
    style_ax(ax, title, rotate_x=True)
    ax.tick_params(axis='y', rotation=0)

######################################################################
# Step 3: 绘图
######################################################################

# ── Figure 1: TF-Region F-score ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Region: F0.1 Score Comparison', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], tfr_per, 'Per-TF F0.1 (all cell types)', 'F0.1 Score')
draw_grouped_bar(axes[0,1], tfr_per,
                 'Mean F0.1 per Cell Type', 'Mean F0.1')
draw_boxplot(axes[1,0], tfr_all, 'Overall Macro F0.1 (per cell type)', 'Macro F_score')
draw_heatmap(axes[1,1], tfr_all, 'Macro F0.1 Heatmap (Method × Cell Type)',
             'Macro F0.1', 'YlOrRd')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig1_TF_Region_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig1_TF_Region_Fscore.png")

# ── Figure 2: Region-Gene ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Region-Gene: |Spearman ρ| & F-score', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], rg_corr, 'Per-Gene |Spearman ρ| (all cell types)', '|Spearman ρ|')
draw_grouped_bar(axes[0,1], rg_corr, 'Mean |Spearman ρ| per Cell Type', 'Mean |ρ|')
draw_boxplot(axes[1,0], rg_prec, 'Overall F-score (per cell type)', 'F-score')
draw_heatmap(axes[1,1], rg_corr, '|Spearman ρ| Heatmap (Method × Cell Type)',
             'Mean |ρ|', 'PuBuGn')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig2_Region_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig2_Region_Gene_Corr_Fscore.png")

# ── Figure 3: TF-Gene ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Gene: Correlation & F-score', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], tfg_corr, 'Per-Gene Correlation (all cell types)', '|Correlation|')
draw_grouped_bar(axes[0,1], tfg_corr, 'Mean Correlation per Cell Type', 'Mean |Corr|')
draw_boxplot(axes[1,0], tfg_prec, 'Per-TF F_score (all cell types)', 'F_score')
draw_heatmap(axes[1,1], tfg_prec, 'Per-TF F_score Heatmap (Method × Cell Type)',
             'F_score', 'OrRd')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig3_TF_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig3_TF_Gene_Corr_Fscore.png")

# ── Figure 4: 三层合并主图 ───────────────────────────────────────────
from matplotlib.patches import Patch

fig = plt.figure(figsize=(24, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

layer_configs = [
    (tfr_per,  'TF-Region F0.1',          'TF-Region F0.1 Heatmap',       'YlOrRd',  'F0.1 Score'),
    (rg_corr,  'Region-Gene |Spearman ρ|', 'Region-Gene |ρ| Heatmap',      'PuBuGn',  '|Spearman ρ|'),
    (tfg_corr, 'TF-Gene Correlation',      'TF-Gene Correlation Heatmap',  'YlGnBu',  '|Correlation|'),
]

for col_i, (df, title_b, title_h, cmap, ylabel) in enumerate(layer_configs):
    ax_box  = fig.add_subplot(gs[0, col_i])
    ax_heat = fig.add_subplot(gs[1, col_i])
    draw_boxplot(ax_box, df, title_b, ylabel)
    draw_heatmap(ax_heat, df, title_h, ylabel, cmap)

legend_elements = [
    Patch(facecolor=COLORS.get(m,'#999'), label=m,
          alpha=0.9 if m==FOCAL else 0.75,
          edgecolor=COLORS[FOCAL] if m==FOCAL else 'none',
          linewidth=2 if m==FOCAL else 0)
    for m in METHOD_ORDER
]
fig.legend(handles=legend_elements, loc='lower center', ncol=len(METHOD_ORDER),
           fontsize=10, bbox_to_anchor=(0.5,-0.03),
           title='Methods', title_fontsize=11)
fig.suptitle(
    'DyGMamba Benchmark: F-score & Correlation Across 3 Regulatory Layers\n'
    f'({len(avail_ct)} cell types  |  * = Wilcoxon p < 0.05  |  β = {BETA})',
    fontsize=14, fontweight='bold', y=1.01)

plt.savefig(f"{output_dir}Fig4_Main_Combined.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig4_Main_Combined.png")

print(f"\n所有图已保存到: {output_dir}")

In [ ]:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             "'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/fscore_corr/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878", "HepG2", "IMR90", "K562", "MCF7", "A549", "H1", "HELA", "SK"]
METHOD_LIST = ["DyGMamba", "CellOracle", "FigR", "GLUE", "LINGER", "GRaNIE", "Pando"]
FOCAL       = "DyGMamba"
BETA        = 0.1

COLORS = {
    "DyGMamba":  "#9467BD",
    "GLUE":      "#00A087",
    "FigR":      "#3C5488",
    "CellOracle":"#F39B7F",
    "LINGER":    "#8491B4",
    "GRaNIE":    "#91D1C2",
    "Pando":     "#DC0000",
}
METHOD_ORDER = METHOD_LIST.copy()

######################################################################
# Step 1: 对每个细胞系重新计算并收集结果
######################################################################
# 存放跨细胞系的数据
tfr_per_rows  = []   # TF-Region per-TF fscore
tfr_all_rows  = []   # TF-Region macro F_score
rg_corr_rows  = []   # Region-Gene Spearman_Rho
rg_prec_rows  = []   # Region-Gene F-score
tfg_corr_rows = []   # TF-Gene Correlation
tfg_prec_rows = []   # TF-Gene per-TF F_score
tfg_tot_rows  = []   # TF-Gene overall F-beta

for ct in CELL_TYPES:
    data_path   = f"{data_root}{ct}/"
    bench_path  = f"{data_path}benchmarkV7/"
    unibind_file = f"{bench_path}unibind_df.pkl"

    if not os.path.exists(unibind_file):
        print(f"[跳过] {ct}: 找不到 {unibind_file}")
        continue

    print(f"\n{'='*50}")
    print(f"处理: {ct}")
    print(f"{'='*50}")

    # ── TF-Region ─────────────────────────────────────
    try:
        tf_region_result, _ = analysis_tf_region(
            unibind_file, data_path, BETA, METHOD_LIST)
        all_tf_region_result = calc_overall_metrics(tf_region_result, beta=BETA)

        for method, df in tf_region_result.items():
            if len(df) == 0: continue
            for v in df['fscore'].dropna():
                tfr_per_rows.append({'CellType': ct, 'Method': method, 'Value': float(v)})

        for _, row in all_tf_region_result.iterrows():
            tfr_all_rows.append({'CellType': ct,
                                  'Method':   row['Method'],
                                  'Value':    float(row['F_score'])})
        print(f"  ✓ TF-Region: {len(tf_region_result)} methods")

    except Exception as e:
        print(f"  [TF-Region 错误] {ct}: {e}")

    # ── Region-Gene ────────────────────────────────────
    try:
        rg_total_corr, rg_per_corr_df, rg_method_data, bench_peak_gene_df = \
            analysis_region_gene(data_path, bench_path, METHOD_LIST,
                                 hic_threshold=140, hic_threshold_lower=30,
                                 self_threshold=0.7)

        rg_total_prec, rg_per_prec, _ = analysis_region_gene_precision(
            bench_peak_gene_df, rg_method_data, bench_path, COLORS, beta=BETA)

        # per-gene Spearman
        for _, row in rg_per_corr_df[['Method','Spearman_Rho']].dropna().iterrows():
            rg_corr_rows.append({'CellType': ct,
                                  'Method':   row['Method'],
                                  'Value':    abs(float(row['Spearman_Rho']))})

        # overall F-score/F-beta
        fscore_col = None
        for c in ['F-beta', 'F_score', 'fscore']:
            if c in rg_total_prec.columns:
                fscore_col = c
                break
        if fscore_col:
            for _, row in rg_total_prec[['Method', fscore_col]].dropna().iterrows():
                rg_prec_rows.append({'CellType': ct,
                                      'Method':   row['Method'],
                                      'Value':    float(row[fscore_col])})

        print(f"  ✓ Region-Gene: {rg_per_corr_df['Method'].nunique()} methods, "
              f"{len(rg_per_corr_df)} gene-pairs")''

    except Exception as e:
        print(f"  [Region-Gene 错误] {ct}: {e}")'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''

    # ── TF-Gene ────────────────────────────────────────
    try:
        model_result_path = f"{data_path}data_dyg/"
        _ = dyg_tf_gene_result_inherit(data_path, model_result_path)

        corr_dict, _, prec_df, per_prec_dict = analysis_tf_gene_data(
            data_path, METHOD_LIST, flag_corr=True, flag_precision=True, beta=BETA)

        # per-gene correlation
        for method, df in corr_dict.items():
            if len(df) == 0: continue
            col = 'Correlation' if 'Correlation' in df.columns else df.columns[-1]
            for v in df[col].dropna():
                tfg_corr_rows.append({'CellType': ct, 'Method': method,
                                       'Value': abs(float(v))})

        # per-TF F_score
        for method, df in per_prec_dict.items():
            if len(df) == 0: continue
            for c in ['F_score', 'F-beta', 'fscore']:
                if c in df.columns:
                    for v in df[c].dropna():
                        tfg_prec_rows.append({'CellType': ct, 'Method': method,
                                               'Value': float(v)})
                    break

        # overall F-beta
        if prec_df is not None and len(prec_df) > 0:
            for c in ['F-beta', 'F_score', 'fscore']:
                if c in prec_df.columns:
                    for _, row in prec_df[['Method', c]].dropna().iterrows():
                        tfg_tot_rows.append({'CellType': ct,
                                              'Method':   row['Method'],
                                              'Value':    float(row[c])})
                    break

        print(f"  ✓ TF-Gene: {len(corr_dict)} methods")

    except Exception as e:
        print(f"  [TF-Gene 错误] {ct}: {e}")

# ── 转为 DataFrame ─────────────────────────────────
tfr_per  = pd.DataFrame(tfr_per_rows)  if tfr_per_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
tfr_all  = pd.DataFrame(tfr_all_rows)  if tfr_all_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
rg_corr  = pd.DataFrame(rg_corr_rows)  if rg_corr_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
rg_prec  = pd.DataFrame(rg_prec_rows)  if rg_prec_rows  else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_corr = pd.DataFrame(tfg_corr_rows) if tfg_corr_rows else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_prec = pd.DataFrame(tfg_prec_rows) if tfg_prec_rows else pd.DataFrame(columns=['CellType','Method','Value'])
tfg_tot  = pd.DataFrame(tfg_tot_rows)  if tfg_tot_rows  else pd.DataFrame(columns=['CellType','Method','Value'])

avail_ct = sorted(set(
    tfr_per['CellType'].tolist() + rg_corr['CellType'].tolist() +
    tfg_corr['CellType'].tolist()
))

print(f"\n=== 数据状态 ===")
for name, df in [('TF-Region per fscore', tfr_per), ('TF-Region macro F', tfr_all),
                  ('Region-Gene |ρ|', rg_corr),     ('Region-Gene F', rg_prec),
                  ('TF-Gene Corr', tfg_corr),        ('TF-Gene per-TF F', tfg_prec),
                  ('TF-Gene overall F-beta', tfg_tot)]:
    status = f"{len(df)} 行" if len(df) > 0 else "⚠ 空"
    print(f"  {name:30s}: {status}")

# ── 同时保存修复后的 Excel（供下次直接使用）──────────────
save_excel = os.path.join(output_dir, "benchmark_all_results_fixed.xlsx")
with pd.ExcelWriter(save_excel, engine='openpyxl') as writer:
    for name, df in [
        ('TF_region_per',              tfr_per),
        ('TF_region_all',              tfr_all),
        ('Region_gene_per_corr',       rg_corr),
        ('Region_gene_prec',           rg_prec),
        ('TF_gene_per_corr',           tfg_corr),
        ('TF_gene_per_precision',      tfg_prec),
        ('TF_gene_total_precision',    tfg_tot),
    ]:
        if len(df) > 0:
            df.to_excel(writer, sheet_name=name, index=False)
print(f"✓ 汇总 Excel 已保存: {save_excel}")

######################################################################
# Step 2: 绘图工具函数
######################################################################
FONT       = dict(fontsize=11)
TITLE_FONT = dict(fontsize=12, fontweight='bold')

def get_order(df):
    present = df['Method'].unique()
    return [m for m in METHOD_ORDER if m in present]

def style_ax(ax, title='', xlabel='', ylabel='', rotate_x=True):
    ax.set_title(title, **TITLE_FONT)
    ax.set_xlabel(xlabel, **FONT)
    ax.set_ylabel(ylabel, **FONT)
    ax.spines[['top','right']].set_visible(False)
    if rotate_x:
        ax.tick_params(axis='x', rotation=30, labelsize=9)
    ax.tick_params(axis='y', labelsize=9)

def highlight_focal(ax, order):
    if FOCAL in order:
        ax.axvspan(list(order).index(FOCAL) - 0.45,
                   list(order).index(FOCAL) + 0.45,
                   alpha=0.07, color=COLORS[FOCAL], zorder=0)

def add_sig_stars(ax, focal_vals, comp_dict, order, y_offset=0.02):
    if len(focal_vals) < 4: return
    y_max = ax.get_ylim()[1]
    for i, method in enumerate(order):
        if method == FOCAL or method not in comp_dict: continue
        b = comp_dict[method]
        n = min(len(focal_vals), len(b))
        if n < 4: continue
        try:
            _, p = stats.wilcoxon(focal_vals[:n], b[:n], alternative='greater')
            if p < 0.05:
                ax.text(i, y_max*(1-y_offset), '*',
                        ha='center', fontsize=14, color='#C00000', fontweight='bold')
        except Exception:
            pass

def draw_boxplot(ax, df, title, ylabel):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title, ylabel=ylabel); return
    order = get_order(df)
    pal   = {m: COLORS.get(m,'#999') for m in order}
    sns.boxplot(data=df, x='Method', y='Value', order=order, palette=pal,
                ax=ax, width=0.55, linewidth=1.2,
                flierprops={'marker':'o','markersize':3,'alpha':0.4})
    sns.stripplot(data=df, x='Method', y='Value', order=order, palette=pal,
                  ax=ax, size=3.5, alpha=0.35, jitter=True)
    highlight_focal(ax, order)
    focal_v = df[df['Method']==FOCAL]['Value'].values
    comp_v  = {m: df[df['Method']==m]['Value'].values for m in order if m!=FOCAL}
    add_sig_stars(ax, focal_v, comp_v, order)
    medians = df.groupby('Method')['Value'].median()
    for j, m in enumerate(order):
        if m in medians.index:
            ax.text(j, medians[m], f'{medians[m]:.3f}',
                    ha='center', va='bottom', fontsize=7, color='#333')
    style_ax(ax, title, ylabel=ylabel)

def draw_grouped_bar(ax, df, title, ylabel):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title, ylabel=ylabel, rotate_x=False); return
    order  = get_order(df)
    mean_df = df.groupby(['CellType','Method'])['Value'].mean().unstack(fill_value=0)
    ct_list = [c for c in avail_ct if c in mean_df.index]
    x = np.arange(len(ct_list))
    n_m, w = len(order), 0.8/len(order)
    for i, method in enumerate(order):
        if method not in mean_df.columns: continue
        vals = [mean_df.loc[ct, method] if ct in mean_df.index else 0 for ct in ct_list]
        ax.bar(x+(i-n_m/2+0.5)*w, vals, w,
               color=COLORS.get(method,'#999'), alpha=0.85,
               edgecolor=COLORS[FOCAL] if method==FOCAL else 'white',
               linewidth=1.5 if method==FOCAL else 0.6, label=method)
    ax.set_xticks(x)
    ax.set_xticklabels(ct_list, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=7, ncol=2, loc='upper right')
    style_ax(ax, title, ylabel=ylabel, rotate_x=False)

def draw_heatmap(ax, df, title, ylabel, cmap='YlOrRd'):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, fontsize=11, color='gray')
        style_ax(ax, title); return
    pivot = df.groupby(['Method','CellType'])['Value'].mean().unstack(fill_value=np.nan)
    ord_h = [m for m in [FOCAL]+[m for m in METHOD_ORDER if m!=FOCAL] if m in pivot.index]
    ct_h  = [c for c in avail_ct if c in pivot.columns]
    if not ord_h or not ct_h: return
    pivot = pivot.loc[ord_h, ct_h]
    sns.heatmap(pivot, ax=ax, cmap=cmap, annot=True, fmt='.3f',
                annot_kws={'size':8}, linewidths=0.3,
                cbar_kws={'label':ylabel,'shrink':0.8})
    for j in range(len(ct_h)):
        ax.add_patch(plt.Rectangle((j,0),1,1,fill=False,
                     edgecolor=COLORS[FOCAL],lw=2.2))
    style_ax(ax, title, rotate_x=True)
    ax.tick_params(axis='y', rotation=0)

######################################################################
# Step 3: 绘图
######################################################################

# ── Figure 1: TF-Region F-score ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Region: F0.1 Score Comparison', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], tfr_per, 'Per-TF F0.1 (all cell types)', 'F0.1 Score')
draw_grouped_bar(axes[0,1], tfr_per,
                 'Mean F0.1 per Cell Type', 'Mean F0.1')
draw_boxplot(axes[1,0], tfr_all, 'Overall Macro F0.1 (per cell type)', 'Macro F_score')
draw_heatmap(axes[1,1], tfr_all, 'Macro F0.1 Heatmap (Method × Cell Type)',
             'Macro F0.1', 'YlOrRd')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig1_TF_Region_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig1_TF_Region_Fscore.png")

# ── Figure 2: Region-Gene ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Region-Gene: |Spearman ρ| & F-score', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], rg_corr, 'Per-Gene |Spearman ρ| (all cell types)', '|Spearman ρ|')
draw_grouped_bar(axes[0,1], rg_corr, 'Mean |Spearman ρ| per Cell Type', 'Mean |ρ|')
draw_boxplot(axes[1,0], rg_prec, 'Overall F-score (per cell type)', 'F-score')
draw_heatmap(axes[1,1], rg_corr, '|Spearman ρ| Heatmap (Method × Cell Type)',
             'Mean |ρ|', 'PuBuGn')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig2_Region_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig2_Region_Gene_Corr_Fscore.png")

# ── Figure 3: TF-Gene ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('TF-Gene: Correlation & F-score', fontsize=14, fontweight='bold')

draw_boxplot(axes[0,0], tfg_corr, 'Per-Gene Correlation (all cell types)', '|Correlation|')
draw_grouped_bar(axes[0,1], tfg_corr, 'Mean Correlation per Cell Type', 'Mean |Corr|')
draw_boxplot(axes[1,0], tfg_prec, 'Per-TF F_score (all cell types)', 'F_score')
draw_heatmap(axes[1,1], tfg_prec, 'Per-TF F_score Heatmap (Method × Cell Type)',
             'F_score', 'OrRd')

plt.tight_layout()
plt.savefig(f"{output_dir}Fig3_TF_Gene_Corr_Fscore.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig3_TF_Gene_Corr_Fscore.png")

# ── Figure 4: 三层合并主图 ───────────────────────────────────────────
from matplotlib.patches import Patch

fig = plt.figure(figsize=(24, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

layer_configs = [
    (tfr_per,  'TF-Region F0.1',          'TF-Region F0.1 Heatmap',       'YlOrRd',  'F0.1 Score'),
    (rg_corr,  'Region-Gene |Spearman ρ|', 'Region-Gene |ρ| Heatmap',      'PuBuGn',  '|Spearman ρ|'),
    (tfg_corr, 'TF-Gene Correlation',      'TF-Gene Correlation Heatmap',  'YlGnBu',  '|Correlation|'),
]

for col_i, (df, title_b, title_h, cmap, ylabel) in enumerate(layer_configs):
    ax_box  = fig.add_subplot(gs[0, col_i])
    ax_heat = fig.add_subplot(gs[1, col_i])
    draw_boxplot(ax_box, df, title_b, ylabel)
    draw_heatmap(ax_heat, df, title_h, ylabel, cmap)

legend_elements = [
    Patch(facecolor=COLORS.get(m,'#999'), label=m,
          alpha=0.9 if m==FOCAL else 0.75,
          edgecolor=COLORS[FOCAL] if m==FOCAL else 'none',
          linewidth=2 if m==FOCAL else 0)
    for m in METHOD_ORDER
]
fig.legend(handles=legend_elements, loc='lower center', ncol=len(METHOD_ORDER),
           fontsize=10, bbox_to_anchor=(0.5,-0.03),
           title='Methods', title_fontsize=11)
fig.suptitle(
    'DyGMamba Benchmark: F-score & Correlation Across 3 Regulatory Layers\n'
    f'({len(avail_ct)} cell types  |  * = Wilcoxon p < 0.05  |  β = {BETA})',
    fontsize=14, fontweight='bold', y=1.01)

plt.savefig(f"{output_dir}Fig4_Main_Combined.png", dpi=200, bbox_inches='tight')
plt.close(); print("✓ Fig4_Main_Combined.png")

print(f"\n所有图已保存到: {output_dir}")

# V3

In [3]:
"""
benchmark_final_plots.py

评估指标参考文献：
────────────────────────────────────────────────────────────────────
层次          指标              参考文献
────────────────────────────────────────────────────────────────────
TF-Region    F0.1 (β=0.1)     Bravo González-Blas et al. Nature Methods 20(10):1355-1367 (2023) [SCENIC+]
             Precision/Recall  Wang et al. Nature Methods 20(9):1368-1378 (2023) [Dictys]
             Macro F0.1        Huynh-Thu et al. PLOS ONE 5(9):e12776 (2010) [GENIE3]

Region-Gene  Spearman ρ        Pliner et al. Molecular Cell 71(5):858-871 (2018) [Cicero]
             F0.1/Recall       Kartha et al. Cell Genomics 2(12):100237 (2022) [FigR]
             AUPRC             Yuan & Duren Nature Biotechnology 43:247-257 (2025) [LINGER]

TF-Gene      Correlation       Kamimoto et al. Nature 614(7949):742-751 (2023) [CellOracle]
             F-score           Pratapa et al. Nature Methods 17(2):147-154 (2020) [BEELINE]
             Precision         Omony et al. Brief. Bioinformatics 20(3):812-823 (2019)

TF-Recovery  Recovery Curve    Bravo González-Blas et al. Nature Methods 20(10):1355-1367 (2023)
────────────────────────────────────────────────────────────────────
"""
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/final/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878","HepG2","IMR90","K562","MCF7","A549","H1","HELA","SK"]
METHOD_LIST = ["DyGMamba","CellOracle","FigR","GLUE","LINGER","GRaNIE","Pando"]
FOCAL       = "DyGMamba"
TOP_N       = 50

COLORS = {
    "DyGMamba":  "#9467BD",
    "GLUE":      "#00A087",
    "FigR":      "#3C5488",
    "CellOracle":"#F39B7F",
    "LINGER":    "#8491B4",
    "GRaNIE":    "#91D1C2",
    "Pando":     "#DC0000",
}

######################################################################
# 数据读取
######################################################################
def load_excel(ct, sheet):
    path = f"{data_root}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    if not os.path.exists(path): return None
    try:
        xl = pd.ExcelFile(path)
        if sheet not in xl.sheet_names: return None
        df = xl.parse(sheet)
        if 'Method' in df.columns:
            df['Method'] = df['Method'].astype(str).str.strip()
        return df
    except Exception:
        return None

def load_all_cts(sheet, col, cell_types=CELL_TYPES):
    """汇总: {ct: {method: np.array(values)}}"""
    result = {}
    for ct in cell_types:
        df = load_excel(ct, sheet)
        if df is None or col not in df.columns: continue
        ct_data = {}
        for m in METHOD_LIST:
            sub = df[df['Method']==m][col].dropna().values.astype(float)
            if len(sub) > 0:
                ct_data[m] = sub
        if ct_data:
            result[ct] = ct_data
    return result

def top_n_data(ct_data_dict, n=TOP_N, sort_col_data=None):
    """取每个方法按值降序排列后的 top-N"""
    result = {}
    for ct, method_dict in ct_data_dict.items():
        result[ct] = {}
        for m, vals in method_dict.items():
            sorted_vals = np.sort(vals)[::-1]
            result[ct][m] = sorted_vals[:min(n, len(sorted_vals))]
    return result

print("加载数据...")
# ── TF-Region ──────────────────────────────────────────────────────
tfr_fscore_all  = load_all_cts('TF_region_per', 'fscore')
tfr_prec_all    = load_all_cts('TF_region_per', 'Precision')
tfr_recall_all  = load_all_cts('TF_region_per', 'Recall')
tfr_macro       = {}   # ct → {method: scalar}
for ct in CELL_TYPES:
    df = load_excel(ct, 'TF_region_all')
    if df is None: continue
    row_dict = {}
    for col_try in ['F_score','fscore','F-beta']:
        if col_try in df.columns:
            for _, r in df[['Method',col_try]].dropna().iterrows():
                row_dict[r['Method']] = float(r[col_try])
            break
    if row_dict: tfr_macro[ct] = row_dict

tfr_macro_prec = {}
for ct in CELL_TYPES:
    df = load_excel(ct, 'TF_region_all')
    if df is None: continue
    if 'Precision' in df.columns:
        tfr_macro_prec[ct] = {r['Method']: float(r['Precision'])
                              for _, r in df[['Method','Precision']].dropna().iterrows()}

tfr_macro_recall = {}
for ct in CELL_TYPES:
    df = load_excel(ct, 'TF_region_all')
    if df is None: continue
    if 'Recall' in df.columns:
        tfr_macro_recall[ct] = {r['Method']: float(r['Recall'])
                                for _, r in df[['Method','Recall']].dropna().iterrows()}

# ── Region-Gene ────────────────────────────────────────────────────
rg_spearman     = load_all_cts('Region_gene_per_corr',       'Abs_Spearman_Rho')
rg_fscore_per   = load_all_cts('Region_gene_per_precision',  'F_score')
rg_recall_per   = load_all_cts('Region_gene_per_precision',  'Recall')
rg_prec_per     = load_all_cts('Region_gene_per_precision',  'Precision')
rg_auprc_per    = load_all_cts('Region_gene_per_precision',  'AUPRC')
rg_total        = {}   # ct → {method: {metric: scalar}}
for ct in CELL_TYPES:
    df = load_excel(ct, 'Region_gene_total_precision')
    if df is None: continue
    d = {}
    for _, r in df.dropna(subset=['Method']).iterrows():
        m = str(r['Method']).strip()
        d[m] = {}
        for col in ['Precision','Recall','F-beta','F_score','AUPRC']:
            if col in r.index and pd.notna(r[col]):
                d[m][col] = float(r[col])
    if d: rg_total[ct] = d

# ── TF-Gene ────────────────────────────────────────────────────────
tfg_corr_per    = load_all_cts('TF_gene_per_corr',       'Correlation')
tfg_fscore_per  = load_all_cts('TF_gene_per_precision',  'F_score')
tfg_prec_per    = load_all_cts('TF_gene_per_precision',  'Precision')
tfg_recall_per  = load_all_cts('TF_gene_per_precision',  'Recall')
tfg_total       = {}
for ct in CELL_TYPES:
    df = load_excel(ct, 'TF_gene_total_precision')
    if df is None: continue
    d = {}
    for _, r in df.dropna(subset=['Method']).iterrows():
        m = str(r['Method']).strip()
        d[m] = {}
        for col in ['Precision','Recall','F-beta','AUC']:
            if col in r.index and pd.notna(r[col]):
                d[m][col] = float(r[col])
    if d: tfg_total[ct] = d

# ── TF-Recovery ────────────────────────────────────────────────────
tf_recovery = {}   # ct → {method: np.array of cumulative counts}
for ct in CELL_TYPES:
    df = load_excel(ct, 'TF_recovery_num')
    if df is None: continue
    d = {}
    for m in METHOD_LIST:
        if m in df.columns:
            d[m] = df[m].dropna().values.astype(float)
    if d: tf_recovery[ct] = d

avail_ct = sorted(set(
    list(tfr_fscore_all.keys()) +
    list(rg_spearman.keys()) +
    list(tfg_corr_per.keys())
))
print(f"可用细胞系: {avail_ct}  ({len(avail_ct)}个)")

######################################################################
# 通用绘图工具
######################################################################
plt.rcParams.update({
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
})

def get_present(ct_data):
    """返回在当前数据中存在的方法（保持顺序）"""
    all_m = set()
    for d in ct_data.values():
        all_m |= set(d.keys())
    return [m for m in METHOD_LIST if m in all_m]

def bar_colors(methods):
    return [COLORS.get(m,'#999') for m in methods]

def edge_colors(methods):
    return [COLORS[FOCAL] if m==FOCAL else '#444' for m in methods]

def edge_widths(methods):
    return [2.5 if m==FOCAL else 0.6 for m in methods]

def add_rank_badge(ax, rank, total):
    """右上角添加排名徽章"""
    color = '#006400' if rank == 1 else '#555'
    ax.text(0.97, 0.97, f'#{rank}/{total}',
            ha='right', va='top', transform=ax.transAxes,
            fontsize=7, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                      edgecolor=color, alpha=0.85, linewidth=1.2))

def focal_rank(ct_data, ct):
    """DyGMamba 在当前细胞系中的排名"""
    if ct not in ct_data or FOCAL not in ct_data[ct]:
        return None, None
    means = {m: np.mean(v) for m, v in ct_data[ct].items() if len(v)>0}
    sorted_m = sorted(means, key=means.get, reverse=True)
    rank = sorted_m.index(FOCAL)+1 if FOCAL in sorted_m else None
    return rank, len(sorted_m)

def plot_boxplot_panel(ax, ct_data, ct, title='', ylabel='',
                       show_xlabel=True, top_n=None):
    """在 ax 上绘制单个细胞系的箱线图"""
    if ct not in ct_data:
        ax.text(0.5,0.5,'N/A',ha='center',va='center',
                transform=ax.transAxes,color='#aaa',fontsize=9)
        ax.set_title(f'{ct}\n{title}',fontsize=8,fontweight='bold')
        return

    data  = ct_data[ct]
    if top_n:
        data = {m: np.sort(v)[::-1][:min(top_n,len(v))]
                for m,v in data.items()}
    methods = [m for m in METHOD_LIST if m in data and len(data[m])>0]
    if not methods:
        ax.text(0.5,0.5,'N/A',ha='center',va='center',transform=ax.transAxes)
        return

    plot_df = pd.DataFrame([
        {'Method':m,'Value':v}
        for m in methods for v in data[m]
    ])
    sns.boxplot(data=plot_df, x='Method', y='Value',
                order=methods,
                palette={m:COLORS.get(m,'#999') for m in methods},
                ax=ax, width=0.55, linewidth=0.9,
                flierprops={'marker':'o','markersize':2,'alpha':0.3},
                hue='Method', legend=False)

    # DyGMamba 背景高亮
    if FOCAL in methods:
        fi = methods.index(FOCAL)
        ax.axvspan(fi-0.4, fi+0.4, alpha=0.1,
                   color=COLORS[FOCAL], zorder=0)
        # 中位数标注
        med = np.median(data[FOCAL])
        ax.text(fi, med, f'{med:.3f}',
                ha='center', va='bottom', fontsize=6,
                color=COLORS[FOCAL], fontweight='bold')

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([m[:6] for m in methods],
                       rotation=40, ha='right', fontsize=6.5)
    ax.set_title(f'{ct}', fontsize=8.5, fontweight='bold')
    if ylabel: ax.set_ylabel(ylabel, fontsize=7.5)
    ax.set_xlabel('')
    ax.spines[['top','right']].set_visible(False)

    rank, total = focal_rank(ct_data, ct)
    if rank is not None:
        add_rank_badge(ax, rank, total)

def plot_bar_panel(ax, ct_data_scalar, ct, title='', ylabel='',
                   show_xlabel=True):
    """在 ax 上绘制单个细胞系的柱状图（scalar 值）"""
    if ct not in ct_data_scalar:
        ax.text(0.5,0.5,'N/A',ha='center',va='center',
                transform=ax.transAxes,color='#aaa',fontsize=9)
        ax.set_title(f'{ct}',fontsize=8.5,fontweight='bold')
        return

    data    = ct_data_scalar[ct]
    methods = [m for m in METHOD_LIST if m in data]
    if not methods: return

    values  = [data[m] for m in methods]
    x       = np.arange(len(methods))
    bars    = ax.bar(x, values,
                     color=bar_colors(methods),
                     edgecolor=edge_colors(methods),
                     linewidth=[ew for ew in edge_widths(methods)],
                     alpha=0.85, width=0.65, zorder=3)

    # DyGMamba 高亮
    if FOCAL in methods:
        fi = methods.index(FOCAL)
        ax.axvspan(fi-0.4, fi+0.4, alpha=0.1,
                   color=COLORS[FOCAL], zorder=0)
        ax.text(fi, values[fi] + max(values)*0.03,
                f'{values[fi]:.3f}',
                ha='center', va='bottom', fontsize=6.5,
                color=COLORS[FOCAL], fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels([m[:6] for m in methods],
                       rotation=40, ha='right', fontsize=6.5)
    ax.set_title(f'{ct}', fontsize=8.5, fontweight='bold')
    if ylabel: ax.set_ylabel(ylabel, fontsize=7.5)
    ax.set_xlabel('')
    ax.spines[['top','right']].set_visible(False)
    ax.set_ylim(0, max(values)*1.25 if max(values)>0 else 1)

    rank = sorted(methods, key=lambda m: data[m], reverse=True).index(FOCAL)+1 \
           if FOCAL in methods else None
    if rank is not None:
        add_rank_badge(ax, rank, len(methods))

def plot_bar_panel_from_per(ax, ct_data, ct, ylabel=''):
    """从 per-entry 数据计算均值，绘制柱状图"""
    if ct not in ct_data:
        ax.text(0.5,0.5,'N/A',ha='center',va='center',
                transform=ax.transAxes,color='#aaa',fontsize=9)
        ax.set_title(f'{ct}',fontsize=8.5,fontweight='bold'); return
    scalar = {m: np.mean(v) for m,v in ct_data[ct].items() if len(v)>0}
    plot_bar_panel(ax, {ct: scalar}, ct, ylabel=ylabel)

def add_reference(fig, text, y=0.01):
    fig.text(0.5, y, text, ha='center', va='bottom',
             fontsize=6.5, color='#444',
             style='italic', wrap=True)

def make_legend(fig, methods, y=-0.02):
    handles = [Patch(facecolor=COLORS.get(m,'#999'), label=m,
                     edgecolor=COLORS[FOCAL] if m==FOCAL else '#555',
                     linewidth=2 if m==FOCAL else 0.5)
               for m in methods]
    fig.legend(handles=handles, loc='lower center',
               ncol=len(methods), fontsize=8,
               bbox_to_anchor=(0.5, y),
               title='Methods  (★ = DyGMamba)',
               title_fontsize=8)

######################################################################
# Figure 1: TF-Region  (4行 × N_CT列)
# Row0: F0.1 Top-50 Boxplot  Row1: F0.1 All Boxplot
# Row2: Recall Top-50 Boxplot Row3: Macro F0.1 Barplot + Macro Prec bar
######################################################################
print("绘制 Figure 1: TF-Region...")
N = len(avail_ct)
fig = plt.figure(figsize=(2.8*N, 22))
gs  = gridspec.GridSpec(5, N, figure=fig,
                        hspace=0.55, wspace=0.38)

row_configs = [
    # (data, plot_func, kwargs, row_label, ref_short)
    (tfr_fscore_all,  'box', {'top_n':50},     'F₀.₁ Top-50 Boxplot',   'SCENIC+, Nat Methods 2023'),
    (tfr_fscore_all,  'box', {'top_n':None},   'F₀.₁ All Boxplot',      'Dictys, Nat Methods 2023'),
    (tfr_recall_all,  'box', {'top_n':50},     'Recall Top-50 Boxplot', 'scMTNI, Nat Commun 2023'),
    (tfr_macro,       'bar', {},               'Macro F₀.₁ Barplot',    'GENIE3, PLOS ONE 2010'),
    (tfr_macro_prec,  'bar', {},               'Macro Precision Barplot','Wang et al, Nat Methods 2023'),
]

for row_i, (data, ptype, kwargs, row_label, ref) in enumerate(row_configs):
    for col_i, ct in enumerate(avail_ct):
        ax = fig.add_subplot(gs[row_i, col_i])

        ylabel = row_label.split(' ')[0] if col_i == 0 else ''

        if ptype == 'box':
            plot_boxplot_panel(ax, data, ct,
                               ylabel=ylabel, **kwargs)
        else:
            plot_bar_panel(ax, data, ct, ylabel=ylabel)

        if row_i == 0:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=4)
        else:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=2)

    # 行标签
    fig.text(-0.01, 1 - (row_i+0.5)/len(row_configs),
             row_label, ha='right', va='center',
             fontsize=8, fontweight='bold', rotation=90,
             transform=fig.transFigure)

fig.suptitle('TF-Region Benchmark: DyGMamba vs Competing Methods\n'
             'Each column = cell type  |  ★ DyGMamba highlighted  |  #N/M = rank',
             fontsize=11, fontweight='bold', y=1.01)

make_legend(fig, [m for m in METHOD_LIST
                  if any(m in d for d in tfr_fscore_all.values())], y=-0.01)

add_reference(fig,
    'References: [F₀.₁] Bravo González-Blas et al. Nature Methods 20:1355-1367 (2023) [SCENIC+]  |  '
    '[Recall] Wang et al. Nature Methods 20:1368-1378 (2023) [Dictys]  |  '
    '[Macro] Huynh-Thu et al. PLOS ONE 5:e12776 (2010) [GENIE3]', y=0.002)

plt.savefig(f"{output_dir}Fig1_TF_Region.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ Fig1_TF_Region.png")

######################################################################
# Figure 2: Region-Gene  (5行 × N_CT列)
# Row0: Spearman Corr Barplot   Row1: F0.1 All Barplot
# Row2: Recall All Barplot      Row3: F0.1 Top-50 Boxplot
# Row4: AUPRC Top-50 Boxplot
######################################################################
print("绘制 Figure 2: Region-Gene...")
fig = plt.figure(figsize=(2.8*N, 27))
gs  = gridspec.GridSpec(5, N, figure=fig,
                        hspace=0.55, wspace=0.38)

# Spearman 和 AUPRC 转换为 scalar（均值）
rg_spearman_mean = {
    ct: {m: float(np.mean(np.abs(v)))
         for m, v in d.items() if len(v)>0}
    for ct, d in rg_spearman.items()
}
rg_auprc_mean = {
    ct: {m: float(np.mean(v))
         for m, v in d.items() if len(v)>0}
    for ct, d in rg_auprc_per.items()
}

rg_row_configs = [
    (rg_spearman_mean,  'bar_scalar', {}, 'Spearman |ρ| Bar',    'Cicero, Mol Cell 2018'),
    (rg_fscore_per,     'bar_per',   {}, 'F₀.₁ All Barplot',   'FigR, Cell Genomics 2022'),
    (rg_recall_per,     'bar_per',   {}, 'Recall All Barplot',  'SHARE-seq, Cell 2020'),
    (rg_fscore_per,     'box',       {'top_n':50}, 'F₀.₁ Top-50 Boxplot', 'SCENIC+, Nat Methods 2023'),
    (rg_auprc_per,      'box',       {'top_n':50}, 'AUPRC Top-50 Boxplot','LINGER, Nat Biotechnol 2025'),
]

for row_i, (data, ptype, kwargs, row_label, ref) in enumerate(rg_row_configs):
    for col_i, ct in enumerate(avail_ct):
        ax = fig.add_subplot(gs[row_i, col_i])
        ylabel = row_label.split(' ')[0] if col_i == 0 else ''

        if ptype == 'bar_scalar':
            plot_bar_panel(ax, data, ct, ylabel=ylabel)
        elif ptype == 'bar_per':
            plot_bar_panel_from_per(ax, data, ct, ylabel=ylabel)
        elif ptype == 'box':
            plot_boxplot_panel(ax, data, ct, ylabel=ylabel, **kwargs)

        if row_i == 0:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=4)
        else:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=2)

fig.suptitle('Region-Gene Benchmark: DyGMamba vs Competing Methods',
             fontsize=11, fontweight='bold', y=1.01)
make_legend(fig, [m for m in METHOD_LIST
                  if any(m in d for d in rg_spearman_mean.values())], y=-0.01)
add_reference(fig,
    'References: [Spearman ρ] Pliner et al. Molecular Cell 71:858-871 (2018) [Cicero]  |  '
    '[F₀.₁/Recall] Kartha et al. Cell Genomics 2:100237 (2022) [FigR]  |  '
    '[AUPRC] Yuan & Duren Nature Biotechnology 43:247-257 (2025) [LINGER]', y=0.002)

plt.savefig(f"{output_dir}Fig2_Region_Gene.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ Fig2_Region_Gene.png")

######################################################################
# Figure 3: TF-Gene  (4行 × N_CT列)
# Row0: Correlation Top-50 Boxplot   Row1: Correlation Macro Bar
# Row2: F-score All Barplot          Row3: Precision All Barplot
######################################################################
print("绘制 Figure 3: TF-Gene...")
fig = plt.figure(figsize=(2.8*N, 22))
gs  = gridspec.GridSpec(4, N, figure=fig,
                        hspace=0.55, wspace=0.38)

# Macro correlation (mean per method per CT)
tfg_corr_macro = {
    ct: {m: float(np.mean(np.abs(v)))
         for m, v in d.items() if len(v)>0}
    for ct, d in tfg_corr_per.items()
}
# abs values
tfg_corr_abs = {
    ct: {m: np.abs(v) for m, v in d.items()}
    for ct, d in tfg_corr_per.items()
}

tfg_row_configs = [
    (tfg_corr_abs,    'box',      {'top_n':50}, 'Correlation Top-50 Box', 'CellOracle, Nature 2023'),
    (tfg_corr_macro,  'bar_scalar',{},           'Correlation Macro Bar',  'CellOracle, Nature 2023'),
    (tfg_fscore_per,  'bar_per',  {},            'F-score All Barplot',    'BEELINE, Nat Methods 2020'),
    (tfg_prec_per,    'bar_per',  {},            'Precision All Barplot',  'Omony, Brief Bioinform 2019'),
]

for row_i, (data, ptype, kwargs, row_label, ref) in enumerate(tfg_row_configs):
    for col_i, ct in enumerate(avail_ct):
        ax = fig.add_subplot(gs[row_i, col_i])
        ylabel = row_label.split(' ')[0] if col_i == 0 else ''

        if ptype == 'bar_scalar':
            plot_bar_panel(ax, data, ct, ylabel=ylabel)
        elif ptype == 'bar_per':
            plot_bar_panel_from_per(ax, data, ct, ylabel=ylabel)
        elif ptype == 'box':
            plot_boxplot_panel(ax, data, ct, ylabel=ylabel, **kwargs)

        if row_i == 0:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=4)
        else:
            ax.set_title(ct, fontsize=9, fontweight='bold', pad=2)

fig.suptitle('TF-Gene Benchmark: DyGMamba vs Competing Methods',
             fontsize=11, fontweight='bold', y=1.01)
make_legend(fig, [m for m in METHOD_LIST
                  if any(m in d for d in tfg_corr_macro.values())], y=-0.01)
add_reference(fig,
    'References: [Correlation] Kamimoto et al. Nature 614:742-751 (2023) [CellOracle]  |  '
    '[F-score] Pratapa et al. Nature Methods 17:147-154 (2020) [BEELINE]  |  '
    '[Precision] Omony et al. Brief Bioinformatics 20:812-823 (2019)', y=0.002)

plt.savefig(f"{output_dir}Fig3_TF_Gene.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ Fig3_TF_Gene.png")

######################################################################
# Figure 4: TF-Recovery  (1行 × N_CT列)
######################################################################
print("绘制 Figure 4: TF-Recovery...")
fig, axes = plt.subplots(1, N, figsize=(3.2*N, 5.5), sharey=False)
axes = np.array(axes).flatten()

for col_i, ct in enumerate(avail_ct):
    ax = axes[col_i]

    if ct not in tf_recovery:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#aaa', fontsize=9)
        ax.set_title(ct, fontsize=9, fontweight='bold'); continue

    methods_present = [m for m in METHOD_LIST if m in tf_recovery[ct]]
    auc_dict = {}

    for method in methods_present:
        vals = tf_recovery[ct][method]
        x    = np.arange(1, len(vals)+1)
        lw   = 2.5 if method == FOCAL else 1.2
        alpha= 1.0 if method == FOCAL else 0.6
        zo   = 10  if method == FOCAL else 1
        ax.plot(x, vals, color=COLORS.get(method,'#999'),
                linewidth=lw, alpha=alpha, zorder=zo,
                label=method)
        if len(vals) > 1:
            auc_dict[method] = float(np.trapz(vals, x))

    ax.set_title(ct, fontsize=9, fontweight='bold')
    ax.set_xlabel('Top-N Ranked TFs', fontsize=7.5)
    if col_i == 0:
        ax.set_ylabel('Cumulative TFs Recovered', fontsize=7.5)
    ax.spines[['top','right']].set_visible(False)

    # DyGMamba AUC vs best competitor
    if FOCAL in auc_dict and len(auc_dict) > 1:
        dyg_auc   = auc_dict[FOCAL]
        best_comp = max({m:v for m,v in auc_dict.items() if m!=FOCAL},
                        key=lambda m: auc_dict[m], default=None)
        if best_comp:
            delta_pct = (dyg_auc - auc_dict[best_comp]) / \
                        max(auc_dict[best_comp], 1e-9) * 100
            sign = '+' if delta_pct >= 0 else ''
            color = '#006400' if delta_pct >= 0 else '#8B0000'
            ax.text(0.97, 0.05,
                    f'AUC {sign}{delta_pct:.1f}%\nvs {best_comp[:6]}',
                    ha='right', va='bottom', transform=ax.transAxes,
                    fontsize=7, color=color, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.25', facecolor='white',
                              edgecolor=color, alpha=0.8, lw=1.2))

    rank = sorted(auc_dict, key=auc_dict.get, reverse=True).index(FOCAL)+1 \
           if FOCAL in auc_dict else None
    if rank is not None:
        add_rank_badge(ax, rank, len(auc_dict))

fig.suptitle('TF-Recovery Benchmark: DyGMamba vs Competing Methods\n'
             '(AUC% = improvement over best competitor)',
             fontsize=11, fontweight='bold')
make_legend(fig, methods_present if tf_recovery else METHOD_LIST, y=-0.15)
add_reference(fig,
    'Reference: Bravo González-Blas et al. Nature Methods 20:1355-1367 (2023) [SCENIC+]  |  '
    'Huynh-Thu et al. PLOS ONE 5:e12776 (2010) [GENIE3]', y=0.0)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(f"{output_dir}Fig4_TF_Recovery.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ Fig4_TF_Recovery.png")

######################################################################
# Figure 5: 综合排名热图 (所有指标 × 所有细胞系，DyGMamba 排名)
######################################################################
print("绘制 Figure 5: 综合排名热图...")

all_metrics = {
    'TF-Reg F₀.₁(Top50)':  ('box_top50', tfr_fscore_all),
    'TF-Reg F₀.₁(All)':    ('box_all',   tfr_fscore_all),
    'TF-Reg Recall(Top50)': ('box_top50', tfr_recall_all),
    'TF-Reg Macro F₀.₁':   ('scalar',    tfr_macro),
    'TF-Reg Macro Prec':    ('scalar',    tfr_macro_prec),
    'RG Spearman |ρ|':      ('scalar',    rg_spearman_mean),
    'RG F₀.₁(All)':         ('per_mean',  rg_fscore_per),
    'RG Recall(All)':        ('per_mean',  rg_recall_per),
    'RG F₀.₁(Top50)':       ('box_top50', rg_fscore_per),
    'RG AUPRC(Top50)':       ('box_top50', rg_auprc_per),
    'TFG Corr(Top50)':       ('box_top50', tfg_corr_abs),
    'TFG Corr Macro':        ('scalar',    tfg_corr_macro),
    'TFG F-score(All)':      ('per_mean',  tfg_fscore_per),
    'TFG Precision(All)':    ('per_mean',  tfg_prec_per),
}

def get_method_mean(mtype, data, ct):
    if ct not in data: return {}
    if mtype == 'scalar':
        return {m: v for m,v in data[ct].items()}
    elif mtype == 'box_top50':
        return {m: float(np.mean(np.sort(v)[::-1][:min(50,len(v))]))
                for m,v in data[ct].items() if len(v)>0}
    elif mtype in ('box_all','per_mean'):
        return {m: float(np.mean(v)) for m,v in data[ct].items() if len(v)>0}
    return {}

# 构建排名矩阵：行=指标，列=细胞系
rank_matrix = pd.DataFrame(index=list(all_metrics.keys()),
                            columns=avail_ct, dtype=float)
win_matrix  = pd.DataFrame(index=list(all_metrics.keys()),
                            columns=avail_ct, dtype=float)

for metric_name, (mtype, data) in all_metrics.items():
    for ct in avail_ct:
        means = get_method_mean(mtype, data, ct)
        if not means or FOCAL not in means: continue
        sorted_m = sorted(means, key=means.get, reverse=True)
        rank = sorted_m.index(FOCAL)+1 if FOCAL in sorted_m else len(sorted_m)
        rank_matrix.loc[metric_name, ct] = rank
        win_matrix.loc[metric_name, ct]  = 1 if rank == 1 else 0

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle('DyGMamba Comprehensive Ranking Summary\n'
             '(Left: rank across metrics/cell types | Right: win rate per metric)',
             fontsize=12, fontweight='bold')

# 左图：排名热图
rank_plot = rank_matrix.astype(float)
vmax = len(METHOD_LIST)
sns.heatmap(rank_plot, ax=axes[0],
            cmap='RdYlGn_r', vmin=1, vmax=vmax,
            annot=True, fmt='.0f', annot_kws={'size':8.5},
            linewidths=0.4, linecolor='white',
            cbar_kws={'label':'Rank (1=best)','shrink':0.75})
axes[0].set_title('DyGMamba Rank per Metric × Cell Type\n(1=best, green; worst=red)',
                  fontsize=10, fontweight='bold')
axes[0].tick_params(axis='x', rotation=40, labelsize=8.5)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)
axes[0].set_xlabel('Cell Type', fontsize=9)
axes[0].set_ylabel('Metric', fontsize=9)

# 右图：胜率（排名第一的比例）× 每个指标
win_rates = win_matrix.astype(float).mean(axis=1).sort_values(ascending=True)
colors_bar = ['#006400' if v >= 0.5 else '#8B0000'
              for v in win_rates.values]
bars = axes[1].barh(np.arange(len(win_rates)), win_rates.values,
                    color=colors_bar, alpha=0.82,
                    edgecolor='white', linewidth=0.8)
axes[1].set_yticks(np.arange(len(win_rates)))
axes[1].set_yticklabels(win_rates.index, fontsize=8.5)
axes[1].axvline(x=0.5, color='gray', linestyle='--',
                linewidth=1.5, alpha=0.7, label='50% line')
axes[1].set_xlabel('Win Rate (rank #1 fraction across cell types)',
                   fontsize=9)
axes[1].set_title('DyGMamba Win Rate (Rank #1) per Metric\n(green ≥ 50%)',
                  fontsize=10, fontweight='bold')
for i, (name, val) in enumerate(win_rates.items()):
    axes[1].text(val + 0.02, i, f'{val:.0%}',
                 va='center', fontsize=8,
                 color='#006400' if val>=0.5 else '#8B0000',
                 fontweight='bold')
axes[1].set_xlim(0, 1.15)
axes[1].legend(fontsize=8)
axes[1].spines[['top','right']].set_visible(False)

# 总体统计
# 替换这两行
total_rank1 = int(win_matrix.astype(float).fillna(0).values.sum())
total_cells  = int((~rank_matrix.isna()).values.sum())
fig.text(0.5, 0.0,
         f'DyGMamba achieved Rank #1 in {total_rank1}/{total_cells} '
         f'metric×cell-type combinations  ({total_rank1/max(total_cells,1)*100:.1f}%)',
         ha='center', fontsize=9, fontweight='bold', color=COLORS[FOCAL])

plt.tight_layout(rect=[0,0.04,1,1])
plt.savefig(f"{output_dir}Fig5_Ranking_Summary.png",
            dpi=180, bbox_inches='tight')
plt.close()
print("✓ Fig5_Ranking_Summary.png")

######################################################################
# 输出数字摘要
######################################################################
print(f"\n{'='*60}")
print("DyGMamba 数字摘要")
print(f"{'='*60}")
print(f"总体: Rank #1 in {total_rank1}/{total_cells} "
      f"metric×CT combinations ({total_rank1/max(total_cells,1)*100:.1f}%)")
print("\n指标维度 (win rate ≥ 50% 的指标):")
for name, wr in win_rates.sort_values(ascending=False).items():
    if wr >= 0.5:
        print(f"  ✓ {name:30s}: {wr:.0%}")
print("\n参考文献:")
refs = [
    "[TF-Region F₀.₁]  Bravo González-Blas et al. Nature Methods 20:1355-1367 (2023) — SCENIC+",
    "[TF-Region Recall] Wang et al. Nature Methods 20:1368-1378 (2023) — Dictys",
    "[TF-Region Macro]  Huynh-Thu et al. PLOS ONE 5:e12776 (2010) — GENIE3",
    "[Region-Gene ρ]    Pliner et al. Molecular Cell 71:858-871 (2018) — Cicero",
    "[Region-Gene F₀.₁] Kartha et al. Cell Genomics 2:100237 (2022) — FigR",
    "[Region-Gene AUPRC] Yuan & Duren Nature Biotechnology 43:247-257 (2025) — LINGER",
    "[TF-Gene Corr]     Kamimoto et al. Nature 614:742-751 (2023) — CellOracle",
    "[TF-Gene F-score]  Pratapa et al. Nature Methods 17:147-154 (2020) — BEELINE",
    "[TF-Recovery]      Bravo González-Blas et al. Nature Methods 20:1355-1367 (2023)",
]
for r in refs:
    print(f"  {r}")

print(f"\n图已保存到: {output_dir}")

加载数据...
可用细胞系: ['A549', 'GM12878', 'H1', 'HELA', 'HepG2', 'IMR90', 'K562', 'MCF7', 'SK']  (9个)
绘制 Figure 1: TF-Region...
✓ Fig1_TF_Region.png
绘制 Figure 2: Region-Gene...
✓ Fig2_Region_Gene.png
绘制 Figure 3: TF-Gene...
✓ Fig3_TF_Gene.png
绘制 Figure 4: TF-Recovery...
✓ Fig4_TF_Recovery.png
绘制 Figure 5: 综合排名热图...
✓ Fig5_Ranking_Summary.png

DyGMamba 数字摘要
总体: Rank #1 in 56/105 metric×CT combinations (53.3%)

指标维度 (win rate ≥ 50% 的指标):
  ✓ RG F₀.₁(All)                  : 100%
  ✓ RG Recall(All)                : 100%
  ✓ TF-Reg Recall(Top50)          : 100%
  ✓ TF-Reg F₀.₁(Top50)            : 89%
  ✓ TF-Reg Macro F₀.₁             : 89%
  ✓ TF-Reg F₀.₁(All)              : 89%
  ✓ TFG Corr(Top50)               : 78%

参考文献:
  [TF-Region F₀.₁]  Bravo González-Blas et al. Nature Methods 20:1355-1367 (2023) — SCENIC+
  [TF-Region Recall] Wang et al. Nature Methods 20:1368-1378 (2023) — Dictys
  [TF-Region Macro]  Huynh-Thu et al. PLOS ONE 5:e12776 (2010) — GENIE3
  [Region-Gene ρ]    Pliner et al. M

# V4

In [6]:
"""
paper_benchmark_figure.py
A4 论文级 benchmark 图
包含:
  Main Figure (A4 portrait): 4层综合结果
  Supplementary Figure: 各层详细子图
  TF-Recovery 修复: 从 tf_set + ground_truth 重建曲线
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, Rectangle
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

######################################################################
# 全局样式（论文级）
######################################################################
plt.rcParams.update({
    'font.family':        'Arial',
    'font.size':          7,
    'axes.titlesize':     8,
    'axes.labelsize':     7,
    'xtick.labelsize':    6.5,
    'ytick.labelsize':    6.5,
    'legend.fontsize':    6.5,
    'axes.linewidth':     0.6,
    'xtick.major.width':  0.5,
    'ytick.major.width':  0.5,
    'xtick.major.size':   2.5,
    'ytick.major.size':   2.5,
    'pdf.fonttype':       42,    # 嵌入字体（Nature 要求）
    'svg.fonttype':       'none',
})

######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/paper_figures/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878","HepG2","IMR90","K562","MCF7","A549","H1","HELA","SK"]
METHOD_LIST = ["DyGMamba","CellOracle","FigR","GLUE","LINGER","GRaNIE","Pando"]
FOCAL       = "DyGMamba"
TOP_N       = 50

# 论文配色（色盲友好）
COLORS = {
    "DyGMamba":  "#7B2D8B",   # 深紫
    "GLUE":      "#009B77",   # 翠绿
    "FigR":      "#2E4A7C",   # 深蓝
    "CellOracle":"#E87040",   # 橙
    "LINGER":    "#6B7DC0",   # 淡蓝紫
    "GRaNIE":    "#5BAD92",   # 青绿
    "Pando":     "#C0392B",   # 红
}

A4_W, A4_H = 8.27, 11.69   # inches

######################################################################
# 数据读取
######################################################################
def load_excel(ct, sheet):
    path = f"{data_root}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    if not os.path.exists(path): return None
    try:
        xl = pd.ExcelFile(path)
        if sheet not in xl.sheet_names: return None
        df = xl.parse(sheet)
        if 'Method' in df.columns:
            df['Method'] = df['Method'].astype(str).str.strip()
        return df
    except Exception:
        return None

def load_per(sheet, col):
    """返回 {ct: {method: np.array}}"""
    result = {}
    for ct in CELL_TYPES:
        df = load_excel(ct, sheet)
        if df is None or 'Method' not in df.columns or col not in df.columns:
            continue
        d = {}
        for m in METHOD_LIST:
            v = df[df['Method']==m][col].dropna().values.astype(float)
            if len(v): d[m] = v
        if d: result[ct] = d
    return result

def load_scalar(sheet, col):
    """返回 {ct: {method: float}}"""
    result = {}
    for ct in CELL_TYPES:
        df = load_excel(ct, sheet)
        if df is None or 'Method' not in df.columns or col not in df.columns:
            continue
        d = {r['Method']: float(r[col]) for _, r in df[['Method',col]].dropna().iterrows()}
        if d: result[ct] = d
    return result

print("加载数据...")

# TF-Region
tfr_fscore  = load_per('TF_region_per', 'fscore')
tfr_recall  = load_per('TF_region_per', 'Recall')
tfr_macro_f = load_scalar('TF_region_all', 'F_score')

# Region-Gene
rg_fscore   = load_per('Region_gene_per_precision', 'F_score')
rg_recall   = load_per('Region_gene_per_precision', 'Recall')
rg_spearman = load_per('Region_gene_per_corr', 'Abs_Spearman_Rho')
for ct in rg_spearman:
    rg_spearman[ct] = {m: np.abs(v) for m,v in rg_spearman[ct].items()}

# TF-Gene
tfg_corr    = load_per('TF_gene_per_corr', 'Correlation')
tfg_fscore  = load_per('TF_gene_per_precision', 'F_score')
tfg_prec    = load_per('TF_gene_per_precision', 'Precision')
for ct in tfg_corr:
    tfg_corr[ct] = {m: np.abs(v) for m,v in tfg_corr[ct].items()}

avail_ct = sorted(set(
    list(tfr_fscore.keys()) +
    list(rg_fscore.keys()) +
    list(tfg_corr.keys())
))
print(f"可用细胞系: {avail_ct}")

######################################################################
# TF-Recovery 修复：从 tf_set + ground_truth 重建曲线
######################################################################
print("重建 TF-Recovery 曲线...")

def reconstruct_recovery_curves(ct, top_n=50):
    """
    从 TF_recovery_set (Excel) + count_region_df.pkl (本地) 重建恢复曲线
    """
    # ── 1. 读取各方法的 TF 集合 ──────────────────────────────────
    set_df = load_excel(ct, 'TF_recovery_set')
    if set_df is None:
        print(f"  [跳过] {ct}: 找不到 TF_recovery_set sheet")
        return {}

    # ── 2. 读取 ground truth 排序（从 pkl，不从 Excel）────────────
    pkl_path = f"{data_root}{ct}/benchmarkV7/count_region_df.pkl"
    if not os.path.exists(pkl_path):
        print(f"  [跳过] {ct}: 找不到 {pkl_path}")
        return {}

    try:
        count_df = pd.read_pickle(pkl_path)
        # 按 PeakCount 降序排列，取 TF 列
        tf_col = 'TF' if 'TF' in count_df.columns else count_df.columns[0]
        gt_tfs = count_df.sort_values(
            by='PeakCount' if 'PeakCount' in count_df.columns
            else count_df.columns[-1],
            ascending=False
        )[tf_col].dropna().tolist()
    except Exception as e:
        print(f"  [错误] {ct} 读取 count_region_df.pkl: {e}")
        return {}

    # ── 3. 重建每个方法的累积恢复曲线 ───────────────────────────
    curves  = {}
    max_rank = min(top_n, len(gt_tfs))

    for m in set_df.columns:
        method_tfs = set(set_df[m].dropna().astype(str).tolist())
        if not method_tfs:
            continue

        x = np.arange(1, max_rank + 1, dtype=float)
        y = np.zeros(max_rank, dtype=float)
        cumulative = 0
        for i in range(max_rank):
            if gt_tfs[i] in method_tfs:
                cumulative += 1
            y[i] = cumulative

        # 归一化 AUC
        perfect_y   = np.minimum(np.arange(1, max_rank+1, dtype=float),
                                  float(len(method_tfs)))
        raw_auc     = float(np.trapz(y, x))
        perfect_auc = float(np.trapz(perfect_y, x))
        norm_auc    = raw_auc / perfect_auc if perfect_auc > 0 else 0.0

        curves[m] = (x, y, norm_auc)
        print(f"    {m:>12s}: {len(method_tfs)} TFs, "
              f"max_recovered={int(y[-1])}, norm_AUC={norm_auc:.3f}")

    return curves

# 重建所有细胞系的恢复曲线
tf_recovery_curves = {}
for ct in CELL_TYPES:
    curves = reconstruct_recovery_curves(ct)
    if curves:
        tf_recovery_curves[ct] = curves
        print(f"  ✓ {ct}: {list(curves.keys())}")

######################################################################
# 工具函数
######################################################################
def methods_in(data, ct):
    if ct not in data: return []
    return [m for m in METHOD_LIST if m in data[ct]]

def focal_rank_in(data, ct, top_n=None):
    if ct not in data or FOCAL not in data[ct]: return None, None
    if isinstance(list(data[ct].values())[0], np.ndarray):
        if top_n:
            means = {m: np.mean(np.sort(v)[::-1][:min(top_n,len(v))])
                     for m,v in data[ct].items() if len(v)>0}
        else:
            means = {m: np.mean(v) for m,v in data[ct].items() if len(v)>0}
    else:
        means = {m: v for m,v in data[ct].items()}
    ranked = sorted(means, key=means.get, reverse=True)
    r = ranked.index(FOCAL)+1 if FOCAL in ranked else None
    return r, len(ranked)

def panel_label(ax, label, x=-0.18, y=1.08):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top', ha='left')

def clean_ax(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(length=2)

def rank_badge(ax, rank, total, green_thresh=2):
    if rank is None: return
    c = '#1B7837' if rank <= green_thresh else '#888'
    ax.text(0.98, 0.98, f'#{rank}',
            transform=ax.transAxes, fontsize=6.5,
            fontweight='bold', color=c, ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=c, alpha=0.85, lw=0.8))

def draw_boxplot(ax, data, ct, metric_col=None, top_n=None,
                 ylabel='', show_legend=False):
    """通用箱线图，数据为 {method: array}"""
    if ct not in data:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        clean_ax(ax); return

    d = data[ct]
    methods = [m for m in METHOD_LIST if m in d and len(d[m])>0]
    if not methods: clean_ax(ax); return

    if top_n:
        d = {m: np.sort(v)[::-1][:min(top_n,len(v))] for m,v in d.items()}

    records = [{'M':m,'V':float(v)} for m in methods for v in d[m]]
    df = pd.DataFrame(records)

    sns.boxplot(data=df, x='M', y='V', order=methods,
                palette={m: COLORS.get(m,'#999') for m in methods},
                hue='M', legend=False,
                ax=ax, width=0.55, linewidth=0.6,
                flierprops={'marker':'o','markersize':1.5,
                            'markerfacecolor':'#999','alpha':0.4},
                medianprops={'color':'white','linewidth':1.2})

    if FOCAL in methods:
        fi = methods.index(FOCAL)
        ax.axvspan(fi-0.42, fi+0.42, alpha=0.08,
                   color=COLORS[FOCAL], zorder=0)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([m[:7] for m in methods],
                       rotation=35, ha='right', fontsize=6)
    ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)
    ax.set_xlabel('')
    clean_ax(ax)

    rk, tot = focal_rank_in(data if not top_n else
                            {ct:{m:d[m] for m in methods}}, ct)
    rank_badge(ax, rk, tot)

def draw_bar(ax, data, ct, ylabel='', show_val=True):
    """scalar 柱状图"""
    if ct not in data:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        clean_ax(ax); return
    d = data[ct]
    methods = [m for m in METHOD_LIST if m in d]
    if not methods: clean_ax(ax); return

    x = np.arange(len(methods))
    vals = [d[m] for m in methods]
    vmax = max(vals) if vals else 1

    bars = ax.bar(x, vals,
                  color=[COLORS.get(m,'#999') for m in methods],
                  edgecolor=[COLORS[FOCAL] if m==FOCAL else 'white'
                             for m in methods],
                  linewidth=[1.8 if m==FOCAL else 0.3 for m in methods],
                  width=0.65, alpha=0.88, zorder=3)

    if FOCAL in methods:
        fi = methods.index(FOCAL)
        ax.axvspan(fi-0.4, fi+0.4, alpha=0.07,
                   color=COLORS[FOCAL], zorder=0)
        if show_val:
            ax.text(fi, vals[fi] + vmax*0.04, f'{vals[fi]:.3f}',
                    ha='center', va='bottom', fontsize=5.5,
                    color=COLORS[FOCAL], fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels([m[:7] for m in methods],
                       rotation=35, ha='right', fontsize=6)
    ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)
    ax.set_xlabel('')
    ax.set_ylim(0, vmax * 1.28)
    clean_ax(ax)

    ranked = sorted(methods, key=lambda m: d[m], reverse=True)
    rk = ranked.index(FOCAL)+1 if FOCAL in ranked else None
    rank_badge(ax, rk, len(ranked))

def draw_bar_per(ax, data, ct, ylabel='', top_n=None):
    """per-entry 均值柱状图"""
    if ct not in data:
        ax.text(0.5,0.5,'N/A',ha='center',va='center',
                transform=ax.transAxes,color='#bbb',fontsize=7)
        clean_ax(ax); return
    d = data[ct]
    methods = [m for m in METHOD_LIST if m in d and len(d[m])>0]
    if not methods: clean_ax(ax); return
    if top_n:
        scalar = {m: np.mean(np.sort(v)[::-1][:min(top_n,len(v))])
                  for m,v in d.items() if len(v)>0}
    else:
        scalar = {m: np.mean(v) for m,v in d.items() if len(v)>0}
    draw_bar(ax, {ct: scalar}, ct, ylabel=ylabel)

######################################################################
# ══════════════════════════════════════════════════════════════════
# MAIN FIGURE (A4 portrait, 论文主图)
# Layout: 4 rows
#   Row 0: TF-Region  (heatmap + macro boxplot)
#   Row 1: Region-Gene (3 key metrics × 9 CTs)
#   Row 2: TF-Gene     (2 key metrics × 9 CTs)
#   Row 3: TF-Recovery (curves × 9 CTs)
# ══════════════════════════════════════════════════════════════════
######################################################################
print("\n绘制主图 (A4)...")

N = len(avail_ct)
fig = plt.figure(figsize=(A4_W, A4_H))
fig.patch.set_facecolor('white')

# 总 GridSpec: 4行，比例 [2.8, 2.2, 2.2, 2.5]
outer_gs = gridspec.GridSpec(
    4, 1, figure=fig,
    height_ratios=[2.8, 2.2, 2.2, 2.5],
    hspace=0.52,
    left=0.09, right=0.97,
    top=0.96, bottom=0.05
)

# ─── Row 0: TF-Region ─────────────────────────────────────────────
gs0 = gridspec.GridSpecFromSubplotSpec(
    2, N, subplot_spec=outer_gs[0],
    hspace=0.55, wspace=0.35
)

for col_i, ct in enumerate(avail_ct):
    # 上: F0.1 Top-50 boxplot
    ax = fig.add_subplot(gs0[0, col_i])
    draw_boxplot(ax, tfr_fscore, ct, top_n=50,
                 ylabel='F₀.₁' if col_i==0 else '')
    if col_i == 0:
        panel_label(ax, 'A', x=-0.45)
        ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    else:
        ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)

    # 下: Macro F0.1 bar
    ax2 = fig.add_subplot(gs0[1, col_i])
    draw_bar(ax2, tfr_macro_f, ct,
             ylabel='Macro F₀.₁' if col_i==0 else '',
             show_val=(col_i < 3))

# Row 0 标签
fig.text(0.005, 0.88, 'TF-Region', va='center', ha='left',
         fontsize=8, fontweight='bold', rotation=90,
         color='#333')

# ─── Row 1: Region-Gene ───────────────────────────────────────────
gs1 = gridspec.GridSpecFromSubplotSpec(
    3, N, subplot_spec=outer_gs[1],
    hspace=0.55, wspace=0.35
)

rg_rows = [
    (rg_fscore,   'F₀.₁',     'B', None),
    (rg_recall,   'Recall',   None, None),
    (rg_spearman, '|ρ|',      None, None),
]
for row_i, (data, ylabel, plabel, _) in enumerate(rg_rows):
    for col_i, ct in enumerate(avail_ct):
        ax = fig.add_subplot(gs1[row_i, col_i])
        draw_bar_per(ax, data, ct,
                     ylabel=ylabel if col_i==0 else '')
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
            if col_i == 0 and plabel:
                panel_label(ax, plabel, x=-0.45)

fig.text(0.005, 0.62, 'Region-Gene', va='center', ha='left',
         fontsize=8, fontweight='bold', rotation=90, color='#333')

# ─── Row 2: TF-Gene ───────────────────────────────────────────────
gs2 = gridspec.GridSpecFromSubplotSpec(
    2, N, subplot_spec=outer_gs[2],
    hspace=0.55, wspace=0.35
)

tfg_rows = [
    (tfg_corr,   '|Corr|', 'C', 50),
    (tfg_fscore, 'F-score', None, None),
]
for row_i, (data, ylabel, plabel, tn) in enumerate(tfg_rows):
    for col_i, ct in enumerate(avail_ct):
        ax = fig.add_subplot(gs2[row_i, col_i])
        if tn:
            draw_boxplot(ax, data, ct, top_n=tn,
                         ylabel=ylabel if col_i==0 else '')
        else:
            draw_bar_per(ax, data, ct,
                         ylabel=ylabel if col_i==0 else '')
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
            if col_i == 0 and plabel:
                panel_label(ax, plabel, x=-0.45)

fig.text(0.005, 0.38, 'TF-Gene', va='center', ha='left',
         fontsize=8, fontweight='bold', rotation=90, color='#333')

# ─── Row 3: TF-Recovery ───────────────────────────────────────────
gs3 = gridspec.GridSpecFromSubplotSpec(
    1, N, subplot_spec=outer_gs[3],
    wspace=0.35
)

avail_rec = [ct for ct in avail_ct if ct in tf_recovery_curves]

for col_i, ct in enumerate(avail_ct):
    ax = fig.add_subplot(gs3[0, col_i])

    if ct not in tf_recovery_curves:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        clean_ax(ax); continue

    curves = tf_recovery_curves[ct]
    methods = [m for m in METHOD_LIST if m in curves]

    # 先画非 DyGMamba
    for m in methods:
        if m == FOCAL: continue
        x, y, auc_v = curves[m]
        ax.plot(x, y, color=COLORS.get(m,'#999'),
                linewidth=0.8, alpha=0.55, zorder=1)

    # 最后画 DyGMamba（突出）
    if FOCAL in curves:
        x, y, auc_v = curves[FOCAL]
        ax.plot(x, y, color=COLORS[FOCAL],
                linewidth=2.0, alpha=1.0, zorder=10,
                label=f'DyGMamba\n(AUC={auc_v:.2f})')
        ax.fill_between(x, y, alpha=0.08,
                        color=COLORS[FOCAL], zorder=0)

        # AUC 对比
        other_aucs = {m: curves[m][2] for m in methods if m!=FOCAL}
        if other_aucs:
            best_m = max(other_aucs, key=other_aucs.get)
            delta  = (auc_v - other_aucs[best_m]) / \
                     max(other_aucs[best_m], 1e-6) * 100
            color_d = '#1B7837' if delta >= 0 else '#C0392B'
            sign    = '+' if delta >= 0 else ''
            ax.text(0.97, 0.05, f'{sign}{delta:.0f}%',
                    ha='right', va='bottom',
                    transform=ax.transAxes,
                    fontsize=6, color=color_d,
                    fontweight='bold')

    ax.set_xlim(0, TOP_N)
    ax.set_ylim(bottom=0)
    ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    if col_i == 0:
        ax.set_ylabel('TFs Recovered', fontsize=6.5, labelpad=2)
        panel_label(ax, 'D', x=-0.45)
    ax.set_xlabel('TF Rank' if col_i==N//2 else '', fontsize=6.5)
    clean_ax(ax)

fig.text(0.005, 0.12, 'TF-Recovery', va='center', ha='left',
         fontsize=8, fontweight='bold', rotation=90, color='#333')

# ─── 全局图例 ──────────────────────────────────────────────────────
present_methods = [m for m in METHOD_LIST
                   if any(m in d for d in tfr_fscore.values())]
legend_handles = [
    Patch(facecolor=COLORS.get(m,'#999'), label=m,
          edgecolor=COLORS[FOCAL] if m==FOCAL else 'none',
          linewidth=1.8 if m==FOCAL else 0)
    for m in present_methods
]
fig.legend(handles=legend_handles,
           loc='lower center',
           ncol=len(present_methods),
           fontsize=6.5,
           bbox_to_anchor=(0.5, 0.0),
           frameon=True,
           framealpha=0.9,
           edgecolor='#ccc',
           title='Methods  (■ = DyGMamba outlined)',
           title_fontsize=6.5,
           handlelength=1.2,
           handletextpad=0.4,
           columnspacing=0.8)

# 标题
fig.text(0.5, 0.975,
         'DyGMamba Benchmark: TF-Region, Region-Gene, TF-Gene & TF-Recovery',
         ha='center', va='top',
         fontsize=9.5, fontweight='bold')
fig.text(0.5, 0.963,
         'A–D: each column = cell type; #N = DyGMamba rank; % = AUC improvement over best competitor',
         ha='center', va='top', fontsize=7, color='#555')

plt.savefig(f"{output_dir}Main_Figure_A4.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Main_Figure_A4.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Main_Figure_A4.pdf/png")

######################################################################
# ══════════════════════════════════════════════════════════════════
# SUPPLEMENTARY FIGURE 1: TF-Region 详细 (A4 landscape)
# ══════════════════════════════════════════════════════════════════
######################################################################
print("绘制 Supplementary Figure 1: TF-Region 详细...")

fig, axes = plt.subplots(
    3, N, figsize=(A4_H, A4_W*1.1),  # landscape
    gridspec_kw={'hspace':0.55,'wspace':0.35}
)
fig.patch.set_facecolor('white')

sup1_rows = [
    (tfr_fscore,  'F₀.₁ Top-50\n(Boxplot)', True,  50),
    (tfr_recall,  'Recall Top-50\n(Boxplot)', False, 50),
    (tfr_macro_f, 'Macro F₀.₁\n(Barplot)',   False, None),
]
for row_i, (data, row_title, is_box_array, tn) in enumerate(sup1_rows):
    for col_i, ct in enumerate(avail_ct):
        ax = axes[row_i, col_i]
        ylabel = row_title.split('\n')[0] if col_i == 0 else ''
        if is_box_array or (tn and isinstance(list(data.values())[0]
                             if data else {}, dict)):
            draw_boxplot(ax, data, ct, top_n=tn, ylabel=ylabel)
        else:
            if ct in data and isinstance(list(data[ct].values())[0]
                                         if data.get(ct) else 0, np.ndarray):
                draw_bar_per(ax, data, ct, ylabel=ylabel)
            else:
                draw_bar(ax, data, ct, ylabel=ylabel)
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)

    # 行标签
    axes[row_i, 0].set_ylabel(row_title.split('\n')[0],
                               fontsize=7, labelpad=3)
    fig.text(-0.01, 1-(row_i+0.5)/3, row_title,
             ha='right', va='center', fontsize=7.5,
             fontweight='bold', rotation=90,
             transform=axes[row_i,0].transAxes)

# 图例
handles = [Patch(facecolor=COLORS.get(m,'#999'), label=m)
           for m in present_methods]
fig.legend(handles=handles, loc='lower center', ncol=len(present_methods),
           fontsize=6.5, bbox_to_anchor=(0.5,-0.02),
           frameon=True, edgecolor='#ccc',
           handlelength=1.0, columnspacing=0.7)

fig.suptitle('Supplementary Figure 1: TF-Region Benchmark\n'
             'F₀.₁ (β=0.1), Recall across cell types\n'
             'Ref: Bravo González-Blas et al. Nature Methods 20:1355 (2023)',
             fontsize=8.5, fontweight='bold', y=1.02)

plt.savefig(f"{output_dir}Supp_Fig1_TF_Region.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig1_TF_Region.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig1_TF_Region.pdf/png")

######################################################################
# SUPPLEMENTARY FIGURE 2: Region-Gene 详细
######################################################################
print("绘制 Supplementary Figure 2: Region-Gene 详细...")

fig, axes = plt.subplots(
    4, N, figsize=(A4_H, A4_W*1.4),
    gridspec_kw={'hspace':0.55,'wspace':0.35}
)
fig.patch.set_facecolor('white')

rg_rows_sup = [
    (rg_spearman, 'Spearman |ρ|',  'bar_per'),
    (rg_fscore,   'F₀.₁',          'bar_per'),
    (rg_recall,   'Recall',         'bar_per'),
    (rg_fscore,   'F₀.₁ Top-50',   'box_top50'),
]
for row_i, (data, ylabel, ptype) in enumerate(rg_rows_sup):
    for col_i, ct in enumerate(avail_ct):
        ax = axes[row_i, col_i]
        yl = ylabel if col_i == 0 else ''
        if ptype == 'bar_per':
            draw_bar_per(ax, data, ct, ylabel=yl)
        elif ptype == 'box_top50':
            draw_boxplot(ax, data, ct, top_n=50, ylabel=yl)
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)

handles = [Patch(facecolor=COLORS.get(m,'#999'), label=m)
           for m in present_methods]
fig.legend(handles=handles, loc='lower center', ncol=len(present_methods),
           fontsize=6.5, bbox_to_anchor=(0.5,-0.01),
           frameon=True, edgecolor='#ccc',
           handlelength=1.0, columnspacing=0.7)

fig.suptitle('Supplementary Figure 2: Region-Gene Benchmark\n'
             'Spearman ρ, F₀.₁, Recall, AUPRC across cell types\n'
             'Ref: Pliner et al. Mol Cell 71:858 (2018); '
             'Kartha et al. Cell Genomics 2:100237 (2022)',
             fontsize=8.5, fontweight='bold', y=1.01)

plt.savefig(f"{output_dir}Supp_Fig2_Region_Gene.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig2_Region_Gene.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig2_Region_Gene.pdf/png")

######################################################################
# SUPPLEMENTARY FIGURE 3: TF-Gene 详细
######################################################################
print("绘制 Supplementary Figure 3: TF-Gene 详细...")

fig, axes = plt.subplots(
    3, N, figsize=(A4_H, A4_W*1.1),
    gridspec_kw={'hspace':0.55,'wspace':0.35}
)
fig.patch.set_facecolor('white')

tfg_rows_sup = [
    (tfg_corr,   '|Correlation|\nTop-50 Boxplot', 'box_top50'),
    (tfg_fscore, 'F-score\nBarplot',               'bar_per'),
    (tfg_prec,   'Precision\nBarplot',             'bar_per'),
]
for row_i, (data, ylabel, ptype) in enumerate(tfg_rows_sup):
    for col_i, ct in enumerate(avail_ct):
        ax = axes[row_i, col_i]
        yl = ylabel.split('\n')[0] if col_i == 0 else ''
        if ptype == 'box_top50':
            draw_boxplot(ax, data, ct, top_n=50, ylabel=yl)
        else:
            draw_bar_per(ax, data, ct, ylabel=yl)
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)

handles = [Patch(facecolor=COLORS.get(m,'#999'), label=m)
           for m in present_methods]
fig.legend(handles=handles, loc='lower center', ncol=len(present_methods),
           fontsize=6.5, bbox_to_anchor=(0.5,-0.02),
           frameon=True, edgecolor='#ccc',
           handlelength=1.0, columnspacing=0.7)

fig.suptitle('Supplementary Figure 3: TF-Gene Benchmark\n'
             'Correlation (Top-50), F-score, Precision across cell types\n'
             'Ref: Kamimoto et al. Nature 614:742 (2023); '
             'Pratapa et al. Nature Methods 17:147 (2020)',
             fontsize=8.5, fontweight='bold', y=1.02)

plt.savefig(f"{output_dir}Supp_Fig3_TF_Gene.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig3_TF_Gene.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig3_TF_Gene.pdf/png")

######################################################################
# SUPPLEMENTARY FIGURE 4: TF-Recovery (修复后，一行9个细胞系)
######################################################################
print("绘制 Supplementary Figure 4: TF-Recovery...")

fig, axes = plt.subplots(
    1, N, figsize=(A4_H, 3.0),
    gridspec_kw={'wspace':0.38}
)
fig.patch.set_facecolor('white')

auc_summary = {}   # {ct: {method: auc}}

for col_i, ct in enumerate(avail_ct):
    ax = axes[col_i]

    if ct not in tf_recovery_curves:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        ax.set_title(ct, fontsize=7.5, fontweight='bold')
        clean_ax(ax); continue

    curves  = tf_recovery_curves[ct]
    methods = [m for m in METHOD_LIST if m in curves]
    auc_summary[ct] = {}

    # 画所有方法
    for m in methods:
        x, y, auc_v = curves[m]
        auc_summary[ct][m] = auc_v
        is_focal = (m == FOCAL)
        ax.plot(x, y,
                color=COLORS.get(m,'#999'),
                linewidth=1.8 if is_focal else 0.8,
                alpha=1.0  if is_focal else 0.5,
                zorder=10  if is_focal else 1,
                label=f'{m} ({auc_v:.2f})')
        if is_focal:
            ax.fill_between(x, y, alpha=0.1,
                            color=COLORS[FOCAL], zorder=0)

    ax.set_xlim(0, TOP_N)
    ax.set_ylim(bottom=0)
    ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    if col_i == 0:
        ax.set_ylabel('TFs Recovered', fontsize=7, labelpad=2)
    ax.set_xlabel('TF Rank', fontsize=6.5)
    clean_ax(ax)

    # AUC 提升标注
    if FOCAL in auc_summary[ct] and len(auc_summary[ct]) > 1:
        dyg_auc = auc_summary[ct][FOCAL]
        best_v  = max(v for m,v in auc_summary[ct].items() if m!=FOCAL)
        delta   = (dyg_auc - best_v) / max(best_v, 1e-6) * 100
        c = '#1B7837' if delta >= 0 else '#C0392B'
        ax.text(0.97, 0.06,
                f'{"+" if delta>=0 else ""}{delta:.0f}%\nvs best',
                ha='right', va='bottom', transform=ax.transAxes,
                fontsize=5.5, color=c, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                          edgecolor=c, alpha=0.8, lw=0.7))

# AUC 图例（放在最后一个子图旁）
handles_rec = [
    plt.Line2D([0],[0], color=COLORS.get(m,'#999'), linewidth=1.5, label=m)
    for m in present_methods
]
fig.legend(handles=handles_rec, loc='lower center',
           ncol=len(present_methods),
           fontsize=6.5, bbox_to_anchor=(0.5, -0.12),
           frameon=True, edgecolor='#ccc',
           handlelength=1.5, columnspacing=0.8)

fig.suptitle('Supplementary Figure 4: TF-Recovery Benchmark\n'
             '(Curves show cumulative ground-truth TFs recovered by rank; '
             '%  = AUC improvement over best competitor)\n'
             'Ref: Bravo González-Blas et al. Nature Methods 20:1355 (2023) [SCENIC+]',
             fontsize=8.5, fontweight='bold', y=1.08)

plt.savefig(f"{output_dir}Supp_Fig4_TF_Recovery.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig4_TF_Recovery.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig4_TF_Recovery.pdf/png")

######################################################################
# 输出文件清单
######################################################################
print(f"\n{'='*55}")
print("输出文件清单")
print(f"{'='*55}")
files = [
    ("Main_Figure_A4.pdf/png",      "论文主图，A4 portrait，包含4层结果"),
    ("Supp_Fig1_TF_Region.pdf/png", "补充图1：TF-Region详细，landscape"),
    ("Supp_Fig2_Region_Gene.pdf/png","补充图2：Region-Gene详细，landscape"),
    ("Supp_Fig3_TF_Gene.pdf/png",   "补充图3：TF-Gene详细，landscape"),
    ("Supp_Fig4_TF_Recovery.pdf/png","补充图4：TF-Recovery曲线修复版"),
]
for fn, desc in files:
    exists = "✓" if os.path.exists(f"{output_dir}{fn.split('/')[0]}") else "✗"
    print(f"  {exists} {fn:40s}: {desc}")
print(f"\n路径: {output_dir}")

加载数据...
可用细胞系: ['A549', 'GM12878', 'H1', 'HELA', 'HepG2', 'IMR90', 'K562', 'MCF7', 'SK']
重建 TF-Recovery 曲线...
        DyGMamba: 31 TFs, max_recovered=7, norm_AUC=0.119
            GLUE: 1 TFs, max_recovered=0, norm_AUC=0.000
      CellOracle: 31 TFs, max_recovered=7, norm_AUC=0.119
            FigR: 4 TFs, max_recovered=0, norm_AUC=0.000
          GRaNIE: 17 TFs, max_recovered=6, norm_AUC=0.135
           Pando: 22 TFs, max_recovered=5, norm_AUC=0.108
  ✓ GM12878: ['DyGMamba', 'GLUE', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
        DyGMamba: 31 TFs, max_recovered=5, norm_AUC=0.096
            GLUE: 1 TFs, max_recovered=1, norm_AUC=0.153
      CellOracle: 31 TFs, max_recovered=5, norm_AUC=0.096
            FigR: 4 TFs, max_recovered=2, norm_AUC=0.245
          GRaNIE: 15 TFs, max_recovered=4, norm_AUC=0.100
           Pando: 19 TFs, max_recovered=5, norm_AUC=0.133
  ✓ HepG2: ['DyGMamba', 'GLUE', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
        DyGMamba: 22 TFs, max_recovered=2, norm_AU

findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

✓ Main_Figure_A4.pdf/png
绘制 Supplementary Figure 1: TF-Region 详细...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

✓ Supp_Fig1_TF_Region.pdf/png
绘制 Supplementary Figure 2: Region-Gene 详细...


...

✓ Supp_Fig2_Region_Gene.pdf/png
绘制 Supplementary Figure 3: TF-Gene 详细...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

✓ Supp_Fig3_TF_Gene.pdf/png
绘制 Supplementary Figure 4: TF-Recovery...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f

✓ Supp_Fig4_TF_Recovery.pdf/png

输出文件清单
  ✓ Main_Figure_A4.pdf/png                  : 论文主图，A4 portrait，包含4层结果
  ✓ Supp_Fig1_TF_Region.pdf/png             : 补充图1：TF-Region详细，landscape
  ✓ Supp_Fig2_Region_Gene.pdf/png           : 补充图2：Region-Gene详细，landscape
  ✓ Supp_Fig3_TF_Gene.pdf/png               : 补充图3：TF-Gene详细，landscape
  ✓ Supp_Fig4_TF_Recovery.pdf/png           : 补充图4：TF-Recovery曲线修复版

路径: /home/wuyan/dygmamba_project/data/benchmark_summary/paper_figures/


## Test

In [5]:
# 诊断 TF-Recovery 数据格式
for ct in ["A549", "GM12878"]:
    print(f"\n=== {ct} ===")
    
    set_df = load_excel(ct, 'TF_recovery_set')
    print(f"TF_recovery_set:")
    if set_df is not None:
        print(f"  shape: {set_df.shape}")
        print(f"  columns: {list(set_df.columns)}")
        print(f"  前3行:\n{set_df.head(3)}")
    else:
        print("  None — sheet 不存在")
    
    rank_df = load_excel(ct, 'TF_recovery_rank')
    print(f"TF_recovery_rank:")
    if rank_df is not None:
        print(f"  shape: {rank_df.shape}")
        print(f"  columns: {list(rank_df.columns)}")
        print(f"  前5行:\n{rank_df.head(5)}")
    else:
        print("  None — sheet 不存在")
    
    # 同时查看 Excel 里所有 sheet 名
    import pandas as pd
    path = f"{data_root}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    xl = pd.ExcelFile(path)
    print(f"  所有 sheets: {xl.sheet_names}")


=== A549 ===
TF_recovery_set:
  shape: (28, 6)
  columns: ['DyGMamba', 'GLUE', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  前3行:
  DyGMamba  GLUE CellOracle    FigR  GRaNIE   Pando
0   ZNF343   NaN     ZNF343   SNAI2  ZNF343    RARA
1     EGR1   NaN       EGR1    RARA    ELF4     SP3
2     CREM   NaN       CREM  ZNF343  PKNOX1  ZNF317
TF_recovery_rank:
  None — sheet 不存在
  所有 sheets: ['TF_recovery_set', 'TF_region_per', 'TF_region_all', 'Region_gene_total_corr', 'Region_gene_per_corr', 'Region_gene_total_precision', 'Region_gene_per_precision', 'TF_gene_per_corr', 'TF_gene_per_precision', 'TF_gene_total_precision']

=== GM12878 ===
TF_recovery_set:
  shape: (31, 6)
  columns: ['DyGMamba', 'GLUE', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  前3行:
  DyGMamba GLUE CellOracle    FigR  GRaNIE   Pando
0   ZBTB33  SP2     ZBTB33    EGR2   BACH1  ZBTB33
1     ZNF8  NaN       ZNF8  ZNF701  ZBTB33    ZNF8
2     IRF9  NaN       IRF9    IRF3   NFKB1    IRF9
TF_recovery_rank:
  None — sheet 不存在
  所有 s

# V5

In [7]:
"""
paper_benchmark_v2.py
修复版：
1. 统一横坐标（所有方法始终显示，无数据则空）
2. x轴用色块代替旋转文字
3. 无标注（仅柱顶数字）
4. 新增 Region-Gene Top-50 相关系数、TF-Gene Top-50 相关系数
5. 行标签不叠加
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, FancyBboxPatch
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

######################################################################
# 全局样式
######################################################################
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      7,
    'axes.titlesize': 8,
    'axes.labelsize': 7,
    'xtick.labelsize':6,
    'ytick.labelsize':6,
    'axes.linewidth': 0.5,
    'xtick.major.width':0.4,
    'ytick.major.width':0.4,
    'xtick.major.size':2,
    'ytick.major.size':2,
    'pdf.fonttype':   42,
})

######################################################################
# 参数
######################################################################
data_root  = "/home/wuyan/dygmamba_project/data/cell_line/"
output_dir = "/home/wuyan/dygmamba_project/data/benchmark_summary/paper_v2/"
os.makedirs(output_dir, exist_ok=True)

CELL_TYPES  = ["GM12878","HepG2","IMR90","K562","MCF7","A549","H1","HELA","SK"]
METHOD_LIST = ["DyGMamba","CellOracle","FigR","GLUE","LINGER","GRaNIE","Pando"]
FOCAL       = "DyGMamba"
TOP_N       = 50

COLORS = {
    "DyGMamba":  "#7B2D8B",
    "GLUE":      "#009B77",
    "FigR":      "#2E4A7C",
    "CellOracle":"#E87040",
    "LINGER":    "#6B7DC0",
    "GRaNIE":    "#5BAD92",
    "Pando":     "#C0392B",
}
A4_W, A4_H = 8.27, 11.69

######################################################################
# 数据读取
######################################################################
def load_excel(ct, sheet):
    path = f"{data_root}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    if not os.path.exists(path): return None
    try:
        xl = pd.ExcelFile(path)
        if sheet not in xl.sheet_names: return None
        df = xl.parse(sheet)
        if 'Method' in df.columns:
            df['Method'] = df['Method'].astype(str).str.strip()
        return df
    except Exception:
        return None

def load_per(sheet, col):
    """→ {ct: {method: np.array}}，所有 METHOD_LIST 方法都有 key（无数据则空数组）"""
    result = {}
    for ct in CELL_TYPES:
        df = load_excel(ct, sheet)
        d = {}
        for m in METHOD_LIST:
            if df is not None and 'Method' in df.columns and col in df.columns:
                v = df[df['Method']==m][col].dropna().values.astype(float)
            else:
                v = np.array([])
            d[m] = v
        if any(len(v)>0 for v in d.values()):
            result[ct] = d
    return result

def load_scalar(sheet, col):
    """→ {ct: {method: float or None}}"""
    result = {}
    for ct in CELL_TYPES:
        df = load_excel(ct, sheet)
        d = {}
        for m in METHOD_LIST:
            if df is not None and 'Method' in df.columns and col in df.columns:
                sub = df[df['Method']==m][col].dropna()
                d[m] = float(sub.iloc[0]) if len(sub)>0 else None
            else:
                d[m] = None
        if any(v is not None for v in d.values()):
            result[ct] = d
    return result

def reconstruct_recovery(ct, top_n=50):
    set_df   = load_excel(ct, 'TF_recovery_set')
    pkl_path = f"{data_root}{ct}/benchmarkV7/count_region_df.pkl"
    if set_df is None or not os.path.exists(pkl_path):
        return {}
    try:
        count_df = pd.read_pickle(pkl_path)
        tf_col   = 'TF' if 'TF' in count_df.columns else count_df.columns[0]
        pk_col   = 'PeakCount' if 'PeakCount' in count_df.columns else count_df.columns[-1]
        gt_tfs   = count_df.sort_values(pk_col, ascending=False)[tf_col].dropna().tolist()
    except Exception:
        return {}

    curves = {}
    max_rank = min(top_n, len(gt_tfs))
    for m in METHOD_LIST:
        if m not in set_df.columns:
            curves[m] = None; continue
        method_tfs = set(set_df[m].dropna().astype(str).tolist())
        if not method_tfs:
            curves[m] = None; continue
        x = np.arange(1, max_rank+1, dtype=float)
        y = np.zeros(max_rank, dtype=float)
        cum = 0
        for i in range(max_rank):
            if gt_tfs[i] in method_tfs: cum += 1
            y[i] = cum
        perfect_y   = np.minimum(np.arange(1, max_rank+1, dtype=float), float(len(method_tfs)))
        raw_auc     = float(np.trapz(y, x))
        perfect_auc = float(np.trapz(perfect_y, x))
        norm_auc    = raw_auc / perfect_auc if perfect_auc > 0 else 0.0
        curves[m]   = (x, y, norm_auc)
    return curves

print("加载数据...")
tfr_fscore   = load_per('TF_region_per', 'fscore')
tfr_recall   = load_per('TF_region_per', 'Recall')
tfr_macro_f  = load_scalar('TF_region_all', 'F_score')
rg_fscore    = load_per('Region_gene_per_precision', 'F_score')
rg_recall    = load_per('Region_gene_per_precision', 'Recall')
rg_spearman  = load_per('Region_gene_per_corr', 'Abs_Spearman_Rho')
for ct in rg_spearman:
    rg_spearman[ct] = {m: np.abs(v) for m,v in rg_spearman[ct].items()}
tfg_corr     = load_per('TF_gene_per_corr', 'Correlation')
tfg_fscore   = load_per('TF_gene_per_precision', 'F_score')
tfg_prec     = load_per('TF_gene_per_precision', 'Precision')
for ct in tfg_corr:
    tfg_corr[ct] = {m: np.abs(v) for m,v in tfg_corr[ct].items()}

avail_ct = sorted(set(
    list(tfr_fscore.keys())+list(rg_fscore.keys())+list(tfg_corr.keys())
))
N = len(avail_ct)
print(f"可用: {avail_ct}")

tf_recovery_all = {}
for ct in CELL_TYPES:
    c = reconstruct_recovery(ct)
    if c: tf_recovery_all[ct] = c

######################################################################
# 绘图核心工具
######################################################################
SWATCH_H  = 0.06   # 色块高度（axes 坐标）
SWATCH_Y  = -0.22  # 色块 Y 位置（axes 坐标）

def draw_color_swatches(ax, methods=METHOD_LIST):
    """在 x 轴下方绘制方法颜色色块（替代旋转文字标签）"""
    n = len(methods)
    for i, m in enumerate(methods):
        rect = FancyBboxPatch(
            (i - 0.38, SWATCH_Y), 0.76, SWATCH_H,
            boxstyle="round,pad=0.01",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
            facecolor=COLORS.get(m,'#999'),
            edgecolor='white', linewidth=0.3,
            zorder=5
        )
        ax.add_patch(rect)
    ax.set_xticks([])          # 不显示文字标签
    ax.tick_params(bottom=False)

def clean_ax(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(length=2, pad=1)

def draw_bar_unified(ax, data_dict, ct, ylabel='', show_val=False):
    """
    统一柱状图：所有 METHOD_LIST 方法均显示（无数据则高度=0，颜色灰色）
    data_dict: {ct: {method: float or None}}
    """
    ax.set_xlim(-0.5, len(METHOD_LIST)-0.5)
    vals  = []
    colors= []
    edges = []
    lws   = []
    has_data = False

    for m in METHOD_LIST:
        v = None
        if ct in data_dict:
            v = data_dict[ct].get(m, None)
        if v is not None and not np.isnan(float(v)):
            vals.append(float(v)); colors.append(COLORS.get(m,'#999'))
            has_data = True
        else:
            vals.append(0); colors.append('#E0E0E0')  # 灰色=无数据
        edges.append(COLORS[FOCAL] if m==FOCAL else 'white')
        lws.append(1.5 if m==FOCAL else 0.2)

    if not has_data:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        draw_color_swatches(ax)
        clean_ax(ax)
        if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)
        return

    x = np.arange(len(METHOD_LIST))
    ax.bar(x, vals, color=colors, edgecolor=edges, linewidth=lws,
           width=0.7, alpha=0.90, zorder=3)

    # DyGMamba 背景
    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi-0.4, fi+0.4, alpha=0.06, color=COLORS[FOCAL], zorder=0)

    # 柱顶数字（仅 DyGMamba 和最高值）
    vmax = max(v for v in vals if v > 0) if any(v>0 for v in vals) else 1
    if show_val:
        best_i = int(np.argmax(vals))
        for i, v in enumerate(vals):
            if v > 0 and (i == fi or i == best_i):
                ax.text(i, v + vmax*0.03, f'{v:.3f}',
                        ha='center', va='bottom', fontsize=5.5,
                        color=COLORS.get(METHOD_LIST[i],'#444'),
                        fontweight='bold' if i==fi else 'normal')

    ax.set_ylim(0, vmax * 1.32)
    draw_color_swatches(ax)
    clean_ax(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)

def draw_bar_per_unified(ax, data_dict, ct,
                         ylabel='', top_n=None, show_val=False):
    """per-entry → scalar 均值，然后调用 draw_bar_unified"""
    scalar = {}
    if ct in data_dict:
        for m in METHOD_LIST:
            v = data_dict[ct].get(m, np.array([]))
            if len(v) > 0:
                if top_n:
                    sv = np.sort(v)[::-1][:min(top_n, len(v))]
                    scalar[m] = float(np.mean(sv))
                else:
                    scalar[m] = float(np.mean(v))
    draw_bar_unified(ax, {ct: scalar}, ct,
                     ylabel=ylabel, show_val=show_val)

def draw_box_unified(ax, data_dict, ct,
                     ylabel='', top_n=None):
    """
    统一箱线图：所有方法均在 x 轴显示，无数据方法留白
    """
    ax.set_xlim(-0.5, len(METHOD_LIST)-0.5)
    has_any = False

    for i, m in enumerate(METHOD_LIST):
        v = np.array([])
        if ct in data_dict:
            v = data_dict[ct].get(m, np.array([]))
        if top_n and len(v) > 0:
            v = np.sort(v)[::-1][:min(top_n, len(v))]

        if len(v) == 0:
            # 空方法：仅画一条灰线示意
            ax.plot([i-0.3, i+0.3], [0, 0],
                    color='#D0D0D0', linewidth=0.5, zorder=1)
            continue

        has_any = True
        bp = ax.boxplot(v, positions=[i], widths=0.55,
                        patch_artist=True, manage_ticks=False,
                        whiskerprops={'linewidth':0.7, 'color':'#666'},
                        capprops={'linewidth':0.7, 'color':'#666'},
                        medianprops={'color':'white','linewidth':1.2},
                        flierprops={'marker':'o','markersize':1.5,
                                    'markerfacecolor':'#999','alpha':0.35,
                                    'linewidth':0},
                        boxprops={'linewidth':0.6})
        for patch in bp['boxes']:
            patch.set_facecolor(COLORS.get(m,'#999'))
            patch.set_alpha(0.88)
            if m == FOCAL:
                patch.set_edgecolor(COLORS[FOCAL])
                patch.set_linewidth(1.5)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)

    # DyGMamba 背景
    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi-0.42, fi+0.42, alpha=0.06,
               color=COLORS[FOCAL], zorder=0)

    draw_color_swatches(ax)
    clean_ax(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)

def draw_recovery_unified(ax, curves_dict, ct, ylabel=''):
    """TF-Recovery 曲线，统一样式，无标注"""
    has_any = False
    for m in METHOD_LIST:
        c = curves_dict.get(ct, {}).get(m, None) if ct in curves_dict else None
        if c is None:
            continue
        x, y, _ = c
        has_any = True
        is_f = (m == FOCAL)
        ax.plot(x, y,
                color=COLORS.get(m,'#999'),
                linewidth=1.8 if is_f else 0.7,
                alpha=1.0   if is_f else 0.5,
                zorder=10   if is_f else 1)
        if is_f:
            ax.fill_between(x, y, alpha=0.08,
                            color=COLORS[FOCAL], zorder=0)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)

    ax.set_xlim(0, TOP_N)
    ax.set_ylim(bottom=0)
    # x轴：直接显示刻度，不用色块（曲线图）
    ax.set_xlabel('TF Rank', fontsize=6, labelpad=1)
    clean_ax(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)

######################################################################
# 图例（全局，放在图底部）
######################################################################
def make_global_legend(fig, y_pos=-0.01):
    handles = []
    for m in METHOD_LIST:
        h = Patch(facecolor=COLORS.get(m,'#999'), label=m,
                  edgecolor=COLORS[FOCAL] if m==FOCAL else 'none',
                  linewidth=1.5 if m==FOCAL else 0)
        handles.append(h)
    # 灰色=无数据说明
    handles.append(Patch(facecolor='#E0E0E0', label='No data',
                         edgecolor='none'))
    fig.legend(handles=handles, loc='lower center',
               ncol=len(handles), fontsize=6.5,
               bbox_to_anchor=(0.5, y_pos),
               frameon=True, framealpha=0.9, edgecolor='#ccc',
               handlelength=1.0, handletextpad=0.3, columnspacing=0.6)

######################################################################
# ══════════════════════════════════════════════════
# SUPPLEMENTARY FIG 1: TF-Region (A4 landscape)
# 行: F0.1 Top50 box | F0.1 All box |
#     Recall Top50 box | Macro F0.1 bar
# ══════════════════════════════════════════════════
######################################################################
print("绘制 Supp Fig 1: TF-Region...")

fig = plt.figure(figsize=(A4_H, A4_W * 1.15))
fig.patch.set_facecolor('white')

ROW_LABELS_S1 = [
    'F₀.₁ Top-50\nBoxplot',
    'F₀.₁ All\nBoxplot',
    'Recall Top-50\nBoxplot',
    'Macro F₀.₁\nBarplot',
]
n_rows = len(ROW_LABELS_S1)
gs = gridspec.GridSpec(n_rows, N,
                       figure=fig,
                       left=0.10, right=0.97,
                       top=0.93, bottom=0.10,
                       hspace=0.72, wspace=0.25)

for col_i, ct in enumerate(avail_ct):
    for row_i in range(n_rows):
        ax = fig.add_subplot(gs[row_i, col_i])
        yl = ROW_LABELS_S1[row_i].split('\n')[0] if col_i==0 else ''

        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
            draw_box_unified(ax, tfr_fscore, ct, ylabel=yl, top_n=50)
        elif row_i == 1:
            draw_box_unified(ax, tfr_fscore, ct, ylabel=yl, top_n=None)
        elif row_i == 2:
            draw_box_unified(ax, tfr_recall, ct, ylabel=yl, top_n=50)
        elif row_i == 3:
            draw_bar_unified(ax, tfr_macro_f, ct, ylabel=yl, show_val=True)

# 行标签（左侧，不叠加）
for row_i, label in enumerate(ROW_LABELS_S1):
    # 计算行中心位置
    ax_ref = fig.add_subplot(gs[row_i, 0])
    pos = ax_ref.get_position()
    y_center = pos.y0 + pos.height / 2
    fig.text(0.01, y_center, label,
             ha='center', va='center', fontsize=7,
             fontweight='bold', rotation=90,
             transform=fig.transFigure)
    ax_ref.remove()

make_global_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 1 — TF-Region Benchmark',
             fontsize=9, fontweight='bold', y=0.97)
fig.text(0.5, 0.94,
         'Ref: Bravo González-Blas et al. Nature Methods 20:1355 (2023) | '
         'Wang et al. Nature Methods 20:1368 (2023)',
         ha='center', fontsize=6.5, color='#555')

plt.savefig(f"{output_dir}Supp_Fig1_TF_Region.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig1_TF_Region.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig1_TF_Region.pdf/png")

######################################################################
# SUPPLEMENTARY FIG 2: Region-Gene (A4 landscape)
# 行: Spearman All bar | Spearman Top50 box |
#     F0.1 All bar     | F0.1 Top50 box     |
#     Recall All bar
######################################################################
print("绘制 Supp Fig 2: Region-Gene...")

fig = plt.figure(figsize=(A4_H, A4_W * 1.45))
fig.patch.set_facecolor('white')

ROW_LABELS_S2 = [
    'Spearman |ρ|\nAll Barplot',
    'Spearman |ρ|\nTop-50 Boxplot',
    'F₀.₁\nAll Barplot',
    'F₀.₁\nTop-50 Boxplot',
    'Recall\nAll Barplot',
]
n_rows2 = len(ROW_LABELS_S2)
gs2 = gridspec.GridSpec(n_rows2, N,
                        figure=fig,
                        left=0.10, right=0.97,
                        top=0.94, bottom=0.07,
                        hspace=0.72, wspace=0.25)

for col_i, ct in enumerate(avail_ct):
    for row_i in range(n_rows2):
        ax = fig.add_subplot(gs2[row_i, col_i])
        yl = ROW_LABELS_S2[row_i].split('\n')[0] if col_i==0 else ''

        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
            draw_bar_per_unified(ax, rg_spearman, ct,
                                 ylabel=yl, show_val=True)
        elif row_i == 1:
            draw_box_unified(ax, rg_spearman, ct,
                             ylabel=yl, top_n=50)
        elif row_i == 2:
            draw_bar_per_unified(ax, rg_fscore, ct,
                                 ylabel=yl, show_val=True)
        elif row_i == 3:
            draw_box_unified(ax, rg_fscore, ct,
                             ylabel=yl, top_n=50)
        elif row_i == 4:
            draw_bar_per_unified(ax, rg_recall, ct,
                                 ylabel=yl, show_val=True)

# 行标签
for row_i, label in enumerate(ROW_LABELS_S2):
    ax_ref = fig.add_subplot(gs2[row_i, 0])
    pos    = ax_ref.get_position()
    y_c    = pos.y0 + pos.height / 2
    fig.text(0.01, y_c, label,
             ha='center', va='center', fontsize=7,
             fontweight='bold', rotation=90,
             transform=fig.transFigure)
    ax_ref.remove()

make_global_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 2 — Region-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.975)
fig.text(0.5, 0.955,
         'Ref: Pliner et al. Mol Cell 71:858 (2018) | '
         'Kartha et al. Cell Genomics 2:100237 (2022) | '
         'Yuan & Duren Nature Biotechnology 43:247 (2025)',
         ha='center', fontsize=6.5, color='#555')

plt.savefig(f"{output_dir}Supp_Fig2_Region_Gene.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig2_Region_Gene.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig2_Region_Gene.pdf/png")

######################################################################
# SUPPLEMENTARY FIG 3: TF-Gene (A4 landscape)
# 行: Corr Top50 box | Corr All bar |
#     F-score All bar | Precision All bar
######################################################################
print("绘制 Supp Fig 3: TF-Gene...")

fig = plt.figure(figsize=(A4_H, A4_W * 1.15))
fig.patch.set_facecolor('white')

ROW_LABELS_S3 = [
    'Correlation\nTop-50 Boxplot',
    'Correlation\nAll Barplot',
    'F-score\nAll Barplot',
    'Precision\nAll Barplot',
]
n_rows3 = len(ROW_LABELS_S3)
gs3 = gridspec.GridSpec(n_rows3, N,
                        figure=fig,
                        left=0.10, right=0.97,
                        top=0.93, bottom=0.10,
                        hspace=0.72, wspace=0.25)

for col_i, ct in enumerate(avail_ct):
    for row_i in range(n_rows3):
        ax = fig.add_subplot(gs3[row_i, col_i])
        yl = ROW_LABELS_S3[row_i].split('\n')[0] if col_i==0 else ''

        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
            draw_box_unified(ax, tfg_corr, ct,
                             ylabel=yl, top_n=50)
        elif row_i == 1:
            draw_bar_per_unified(ax, tfg_corr, ct,
                                 ylabel=yl, show_val=True)
        elif row_i == 2:
            draw_bar_per_unified(ax, tfg_fscore, ct,
                                 ylabel=yl, show_val=True)
        elif row_i == 3:
            draw_bar_per_unified(ax, tfg_prec, ct,
                                 ylabel=yl, show_val=True)

for row_i, label in enumerate(ROW_LABELS_S3):
    ax_ref = fig.add_subplot(gs3[row_i, 0])
    pos    = ax_ref.get_position()
    y_c    = pos.y0 + pos.height / 2
    fig.text(0.01, y_c, label,
             ha='center', va='center', fontsize=7,
             fontweight='bold', rotation=90,
             transform=fig.transFigure)
    ax_ref.remove()

make_global_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 3 — TF-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.97)
fig.text(0.5, 0.94,
         'Ref: Kamimoto et al. Nature 614:742 (2023) | '
         'Pratapa et al. Nature Methods 17:147 (2020)',
         ha='center', fontsize=6.5, color='#555')

plt.savefig(f"{output_dir}Supp_Fig3_TF_Gene.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig3_TF_Gene.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig3_TF_Gene.pdf/png")

######################################################################
# SUPPLEMENTARY FIG 4: TF-Recovery (宽幅，1行×N列)
######################################################################
print("绘制 Supp Fig 4: TF-Recovery...")

fig, axes = plt.subplots(
    1, N, figsize=(A4_H, 2.8),
    gridspec_kw={'wspace': 0.30}
)
fig.patch.set_facecolor('white')
axes = np.array(axes).flatten()

for col_i, ct in enumerate(avail_ct):
    ax = axes[col_i]
    yl = 'TFs Recovered' if col_i==0 else ''
    ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    draw_recovery_unified(ax, tf_recovery_all, ct, ylabel=yl)

# 曲线专用图例（线型）
line_handles = [
    plt.Line2D([0],[0], color=COLORS.get(m,'#999'),
               linewidth=1.8 if m==FOCAL else 0.9,
               linestyle='-', label=m)
    for m in METHOD_LIST
]
fig.legend(handles=line_handles, loc='lower center',
           ncol=len(METHOD_LIST), fontsize=6.5,
           bbox_to_anchor=(0.5, -0.12),
           frameon=True, edgecolor='#ccc',
           handlelength=1.5, columnspacing=0.8)

fig.suptitle('Supplementary Figure 4 — TF-Recovery Benchmark\n'
             '(Cumulative ground-truth TFs recovered by rank)',
             fontsize=8.5, fontweight='bold', y=1.05)
fig.text(0.5, -0.04,
         'Ref: Bravo González-Blas et al. Nature Methods 20:1355 (2023) [SCENIC+]',
         ha='center', fontsize=6.5, color='#555')

plt.savefig(f"{output_dir}Supp_Fig4_TF_Recovery.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Supp_Fig4_TF_Recovery.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Supp_Fig4_TF_Recovery.pdf/png")

######################################################################
# MAIN FIGURE (A4 portrait)
# Row A: TF-Region  — F0.1 Top-50 box + Macro F0.1 bar
# Row B: Region-Gene — Spearman Top-50 box + F0.1 bar
# Row C: TF-Gene    — Corr Top-50 box + F-score bar
# Row D: TF-Recovery — curves
######################################################################
print("绘制 Main Figure (A4)...")

fig = plt.figure(figsize=(A4_W, A4_H))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(
    4, 1, figure=fig,
    height_ratios=[2.5, 2.5, 2.5, 2.2],
    left=0.10, right=0.97,
    top=0.955, bottom=0.06,
    hspace=0.55
)

panel_labels = ['A','B','C','D']
layer_titles = [
    'TF-Region',
    'Region-Gene',
    'TF-Gene',
    'TF-Recovery',
]

# 每层的子行数和内容定义
layer_configs = [
    # (sub_rows, [(data, plot_type, top_n, ylabel_short)])
    (2, [(tfr_fscore, 'box', 50, 'F₀.₁ Top-50'),
         (tfr_macro_f,'bar', None,'Macro F₀.₁')]),
    (2, [(rg_spearman,'box', 50, '|ρ| Top-50'),
         (rg_fscore,  'bar_per', None, 'F₀.₁')]),
    (2, [(tfg_corr,   'box', 50, '|Corr| Top-50'),
         (tfg_fscore, 'bar_per', None, 'F-score')]),
    (1, [(tf_recovery_all, 'recovery', None, 'TFs Recovered')]),
]

for layer_i, (sub_rows, row_defs) in enumerate(layer_configs):
    inner = gridspec.GridSpecFromSubplotSpec(
        sub_rows, N,
        subplot_spec=outer[layer_i],
        hspace=0.55, wspace=0.25
    )

    for sub_i, (data, ptype, tn, short_y) in enumerate(row_defs):
        for col_i, ct in enumerate(avail_ct):
            ax = fig.add_subplot(inner[sub_i, col_i])

            # 列标题（仅第一子行）
            if sub_i == 0:
                ax.set_title(ct, fontsize=7, fontweight='bold', pad=2)

            yl = short_y if col_i == 0 else ''

            if ptype == 'box':
                draw_box_unified(ax, data, ct, ylabel=yl, top_n=tn)
            elif ptype == 'bar':
                draw_bar_unified(ax, data, ct,
                                 ylabel=yl, show_val=True)
            elif ptype == 'bar_per':
                draw_bar_per_unified(ax, data, ct,
                                     ylabel=yl, show_val=True)
            elif ptype == 'recovery':
                draw_recovery_unified(ax, data, ct, ylabel=yl)

    # 层标签（左侧，每层一个）
    # 取该层第一子行第一列 ax 的位置
    ax0 = fig.add_subplot(inner[0, 0])
    pos0 = ax0.get_position()
    if sub_rows > 1:
        ax1 = fig.add_subplot(inner[sub_rows-1, 0])
        pos1 = ax1.get_position()
        y_c  = (pos0.y0 + pos0.height + pos1.y0) / 2
        ax1.remove()
    else:
        y_c = pos0.y0 + pos0.height / 2
    ax0.remove()

    # Panel 字母
    fig.text(0.015, y_c + 0.02,
             panel_labels[layer_i],
             ha='center', va='bottom',
             fontsize=11, fontweight='bold',
             transform=fig.transFigure)
    # 层名
    fig.text(0.015, y_c - 0.01,
             layer_titles[layer_i],
             ha='center', va='top',
             fontsize=7, fontweight='bold',
             rotation=90, color='#333',
             transform=fig.transFigure)

# 全局图例
make_global_legend(fig, y_pos=0.01)

fig.suptitle(
    'DyGMamba Benchmark — TF-Region, Region-Gene, TF-Gene & TF-Recovery',
    fontsize=9, fontweight='bold', y=0.975)
fig.text(
    0.5, 0.962,
    'Each column = cell type  |  color patches on x-axis indicate methods  |  '
    'grey bar = no data for that method',
    ha='center', fontsize=6.5, color='#555')

plt.savefig(f"{output_dir}Main_Figure_A4.pdf",
            dpi=300, bbox_inches='tight', format='pdf')
plt.savefig(f"{output_dir}Main_Figure_A4.png",
            dpi=300, bbox_inches='tight')
plt.close()
print("✓ Main_Figure_A4.pdf/png")

print(f"\n所有图已保存至: {output_dir}")

加载数据...
可用: ['A549', 'GM12878', 'H1', 'HELA', 'HepG2', 'IMR90', 'K562', 'MCF7', 'SK']
绘制 Supp Fig 1: TF-Region...
✓ Supp_Fig1_TF_Region.pdf/png
绘制 Supp Fig 2: Region-Gene...
✓ Supp_Fig2_Region_Gene.pdf/png
绘制 Supp Fig 3: TF-Gene...
✓ Supp_Fig3_TF_Gene.pdf/png
绘制 Supp Fig 4: TF-Recovery...
✓ Supp_Fig4_TF_Recovery.pdf/png
绘制 Main Figure (A4)...
✓ Main_Figure_A4.pdf/png

所有图已保存至: /home/wuyan/dygmamba_project/data/benchmark_summary/paper_v2/


# V6

In [10]:
"""
paper_benchmark_final.py
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
论文级 Benchmark 可视化 — 最终版
输出文件：
  Main_Figure_A4.pdf/png        论文主图 (A4 portrait)
  Supp_Fig1_TF_Region.pdf/png   补充图1 (landscape)
  Supp_Fig2_Region_Gene.pdf/png 补充图2 (landscape)
  Supp_Fig3_TF_Gene.pdf/png     补充图3 (landscape)
  Supp_Fig4_TF_Recovery.pdf/png 补充图4 (landscape)

设计原则：
  - 所有方法横坐标统一 (无数据显示灰色空柱/空线)
  - x 轴色块代替旋转文字标签
  - 无任何图内标注文字 (无 #rank、无 % 、无数值)
  - 含 Region-Gene Top-50 相关系数、TF-Gene Top-50 相关系数
  - TF-Recovery: 上行曲线 + 下行 AUC 柱状图

评估指标参考文献：
  TF-Region  F₀.₁/Recall  Bravo González-Blas et al. Nat Methods 20:1355 (2023) [SCENIC+]
                            Wang et al. Nat Methods 20:1368 (2023) [Dictys]
  Region-Gene Spearman ρ   Pliner et al. Mol Cell 71:858 (2018) [Cicero]
              F₀.₁/Recall  Kartha et al. Cell Genomics 2:100237 (2022) [FigR]
              AUPRC         Yuan & Duren Nat Biotechnol 43:247 (2025) [LINGER]
  TF-Gene    Correlation   Kamimoto et al. Nature 614:742 (2023) [CellOracle]
             F-score/Prec  Pratapa et al. Nat Methods 17:147 (2020) [BEELINE]
  TF-Recovery              Bravo González-Blas et al. Nat Methods 20:1355 (2023)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────
# 全局样式（publication-ready）
# ─────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          7,
    'axes.titlesize':     8,
    'axes.labelsize':     7,
    'xtick.labelsize':    6,
    'ytick.labelsize':    6,
    'axes.linewidth':     0.5,
    'xtick.major.width':  0.4,
    'ytick.major.width':  0.4,
    'xtick.major.size':   2.0,
    'ytick.major.size':   2.0,
    'pdf.fonttype':       42,   # embed fonts (Nature/Cell requirement)
    'svg.fonttype':       'none',
})

# ─────────────────────────────────────────────────
# 参数 — 修改这里适配你的路径
# ─────────────────────────────────────────────────
DATA_ROOT  = "/home/wuyan/dygmamba_project/data/cell_line/"
OUTPUT_DIR = "/home/wuyan/dygmamba_project/data/benchmark_summary/paper_final/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CELL_TYPES  = ["GM12878", "HepG2", "IMR90", "K562",
               "MCF7",   "A549",  "H1",   "HELA",  "SK"]
METHOD_LIST = ["DyGMamba", "CellOracle", "FigR",
               "GLUE",     "LINGER",     "GRaNIE", "Pando"]
FOCAL  = "DyGMamba"
TOP_N  = 50

# 色盲友好配色
COLORS = {
    "DyGMamba":  "#7B2D8B",
    "CellOracle":"#E87040",
    "FigR":      "#2E4A7C",
    "GLUE":      "#009B77",
    "LINGER":    "#6B7DC0",
    "GRaNIE":    "#5BAD92",
    "Pando":     "#C0392B",
}
NO_DATA_COLOR = "#DCDCDC"   # 灰色 = 无数据

A4_W, A4_H = 8.27, 11.69   # A4 inches

# ══════════════════════════════════════════════════
# 数据读取
# ══════════════════════════════════════════════════

def _read_excel(ct: str, sheet: str):
    path = f"{DATA_ROOT}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    if not os.path.exists(path):
        return None
    try:
        xl = pd.ExcelFile(path)
        if sheet not in xl.sheet_names:
            return None
        df = xl.parse(sheet)
        if 'Method' in df.columns:
            df['Method'] = df['Method'].astype(str).str.strip()
        return df
    except Exception:
        return None


def load_per(sheet: str, col: str) -> dict:
    """
    返回 {ct: {method: np.array}}
    每个 METHOD_LIST 方法均有 key（无数据 → 空数组）
    """
    result = {}
    for ct in CELL_TYPES:
        df = _read_excel(ct, sheet)
        d  = {}
        for m in METHOD_LIST:
            if (df is not None
                    and 'Method' in df.columns
                    and col in df.columns):
                v = df[df['Method'] == m][col].dropna().values.astype(float)
            else:
                v = np.array([])
            d[m] = v
        if any(len(v) > 0 for v in d.values()):
            result[ct] = d
    return result


def load_scalar(sheet: str, col: str) -> dict:
    """
    返回 {ct: {method: float or None}}
    """
    result = {}
    for ct in CELL_TYPES:
        df = _read_excel(ct, sheet)
        d  = {}
        for m in METHOD_LIST:
            if (df is not None
                    and 'Method' in df.columns
                    and col in df.columns):
                sub = df[df['Method'] == m][col].dropna()
                d[m] = float(sub.iloc[0]) if len(sub) > 0 else None
            else:
                d[m] = None
        if any(v is not None for v in d.values()):
            result[ct] = d
    return result


def reconstruct_recovery(ct: str, top_n: int = TOP_N) -> dict:
    """
    从 TF_recovery_set (Excel) + count_region_df.pkl 重建曲线
    返回 {method: (x_arr, y_arr, norm_auc) or None}
    """
    set_df   = _read_excel(ct, 'TF_recovery_set')
    pkl_path = f"{DATA_ROOT}{ct}/benchmarkV7/count_region_df.pkl"

    if set_df is None or not os.path.exists(pkl_path):
        return {m: None for m in METHOD_LIST}

    try:
        count_df = pd.read_pickle(pkl_path)
        tf_col   = 'TF'         if 'TF'         in count_df.columns else count_df.columns[0]
        pk_col   = 'PeakCount'  if 'PeakCount'  in count_df.columns else count_df.columns[-1]
        gt_tfs   = (count_df
                    .sort_values(pk_col, ascending=False)[tf_col]
                    .dropna().tolist())
    except Exception:
        return {m: None for m in METHOD_LIST}

    max_rank = min(top_n, len(gt_tfs))
    curves   = {}
    for m in METHOD_LIST:
        if m not in set_df.columns:
            curves[m] = None
            continue
        tfs = set(set_df[m].dropna().astype(str).tolist())
        if not tfs:
            curves[m] = None
            continue
        x   = np.arange(1, max_rank + 1, dtype=float)
        y   = np.zeros(max_rank, dtype=float)
        cum = 0
        for i in range(max_rank):
            if gt_tfs[i] in tfs:
                cum += 1
            y[i] = cum
        perfect     = np.minimum(np.arange(1, max_rank + 1, dtype=float), float(len(tfs)))
        raw_auc     = float(np.trapz(y, x))
        perfect_auc = float(np.trapz(perfect, x))
        norm_auc    = raw_auc / perfect_auc if perfect_auc > 0 else 0.0
        curves[m]   = (x, y, norm_auc)
    return curves


# ── 加载所有指标 ──────────────────────────────────
print("加载数据...")

# TF-Region
TFR_FSCORE   = load_per   ('TF_region_per', 'fscore')
TFR_RECALL   = load_per   ('TF_region_per', 'Recall')
TFR_MACRO_F  = load_scalar('TF_region_all', 'F_score')

# Region-Gene
for _d in [TFR_FSCORE, TFR_RECALL]:   # abs（预防性）
    for ct in _d:
        _d[ct] = {m: np.abs(v) for m, v in _d[ct].items()}

RG_SPEARMAN  = load_per('Region_gene_per_corr',      'Abs_Spearman_Rho')
RG_FSCORE    = load_per('Region_gene_per_precision', 'F_score')
RG_RECALL    = load_per('Region_gene_per_precision', 'Recall')

for ct in RG_SPEARMAN:
    RG_SPEARMAN[ct] = {m: np.abs(v) for m, v in RG_SPEARMAN[ct].items()}

# TF-Gene
TFG_CORR     = load_per('TF_gene_per_corr',      'Correlation')
TFG_FSCORE   = load_per('TF_gene_per_precision', 'F_score')
TFG_PREC     = load_per('TF_gene_per_precision', 'Precision')

for ct in TFG_CORR:
    TFG_CORR[ct] = {m: np.abs(v) for m, v in TFG_CORR[ct].items()}

# TF-Recovery
TF_REC_CURVES = {}
for ct in CELL_TYPES:
    c = reconstruct_recovery(ct)
    if any(v is not None for v in c.values()):
        TF_REC_CURVES[ct] = c
        print(f"  ✓ {ct} Recovery: "
              f"{[m for m,v in c.items() if v is not None]}")

# AUC scalar
TF_REC_AUC = {}
for ct in CELL_TYPES:
    if ct not in TF_REC_CURVES:
        continue
    d = {}
    for m in METHOD_LIST:
        c = TF_REC_CURVES[ct].get(m)
        d[m] = float(c[2]) if c is not None else None
    TF_REC_AUC[ct] = d

AVAIL_CT = sorted(set(
    list(TFR_FSCORE.keys()) +
    list(RG_FSCORE.keys())  +
    list(TFG_CORR.keys())
))
N = len(AVAIL_CT)
print(f"\n可用细胞系 ({N}): {AVAIL_CT}")

# ══════════════════════════════════════════════════
# 绘图工具函数
# ══════════════════════════════════════════════════

SWATCH_H =  0.07   # 色块高度（axes 坐标）
SWATCH_Y = -0.24   # 色块距 x 轴距离（axes 坐标）


def _swatches(ax):
    """x 轴下方绘制方法颜色色块，取代旋转文字"""
    for i, m in enumerate(METHOD_LIST):
        rect = FancyBboxPatch(
            (i - 0.39, SWATCH_Y), 0.78, SWATCH_H,
            boxstyle="round,pad=0.01",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
            facecolor=COLORS.get(m, '#999'),
            edgecolor='white', linewidth=0.3,
            zorder=5
        )
        ax.add_patch(rect)
    ax.set_xticks([])
    ax.tick_params(bottom=False)


def _clean(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=2, pad=1)


def draw_bar(ax, scalar_dict: dict, ct: str, ylabel: str = ''):
    """
    柱状图（scalar 值）
    scalar_dict: {ct: {method: float or None}}
    无数据方法 → 灰色零高柱
    """
    ax.set_xlim(-0.5, len(METHOD_LIST) - 0.5)

    vals, fc, ec, lw = [], [], [], []
    has_any = False
    for m in METHOD_LIST:
        v = scalar_dict.get(ct, {}).get(m, None)
        if v is not None and not np.isnan(float(v)):
            vals.append(float(v))
            fc.append(COLORS.get(m, '#999'))
            has_any = True
        else:
            vals.append(0.0)
            fc.append(NO_DATA_COLOR)
        ec.append(COLORS[FOCAL] if m == FOCAL else 'white')
        lw.append(1.5        if m == FOCAL else 0.2)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        _swatches(ax); _clean(ax)
        if ylabel:
            ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)
        return

    x    = np.arange(len(METHOD_LIST))
    vmax = max(v for v in vals if v > 0) if any(v > 0 for v in vals) else 1

    ax.bar(x, vals, color=fc, edgecolor=ec, linewidth=lw,
           width=0.70, alpha=0.90, zorder=3)

    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi - 0.42, fi + 0.42,
               alpha=0.07, color=COLORS[FOCAL], zorder=0)

    ax.set_ylim(0, vmax * 1.28)
    _swatches(ax)
    _clean(ax)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


def draw_bar_per(ax, per_dict: dict, ct: str,
                 ylabel: str = '', top_n = None):
    """per-entry 均值 → scalar → draw_bar"""
    scalar = {}
    if ct in per_dict:
        for m in METHOD_LIST:
            v = per_dict[ct].get(m, np.array([]))
            if len(v) > 0:
                if top_n:
                    sv = np.sort(v)[::-1][:min(top_n, len(v))]
                    scalar[m] = float(np.mean(sv))
                else:
                    scalar[m] = float(np.mean(v))
    draw_bar(ax, {ct: scalar}, ct, ylabel=ylabel)


def draw_box(ax, per_dict: dict, ct: str,
             ylabel: str = '', top_n = None):
    """
    箱线图，所有方法均显示（无数据 → 灰色短线占位）
    """
    ax.set_xlim(-0.5, len(METHOD_LIST) - 0.5)
    has_any = False

    for i, m in enumerate(METHOD_LIST):
        v = per_dict.get(ct, {}).get(m, np.array([]))
        if top_n and len(v) > 0:
            v = np.sort(v)[::-1][:min(top_n, len(v))]

        if len(v) == 0:
            ax.plot([i - 0.28, i + 0.28], [0, 0],
                    color='#C8C8C8', linewidth=0.6, zorder=1)
            continue

        has_any = True
        bp = ax.boxplot(
            v, positions=[i], widths=0.58,
            patch_artist=True,
            manage_ticks=False,
            whiskerprops={'linewidth': 0.6, 'color': '#555'},
            capprops   ={'linewidth': 0.6, 'color': '#555'},
            medianprops ={'color': 'white', 'linewidth': 1.2},
            flierprops  ={'marker': 'o', 'markersize': 1.5,
                          'markerfacecolor': '#999',
                          'alpha': 0.35, 'linewidth': 0},
            boxprops    ={'linewidth': 0.6}
        )
        for patch in bp['boxes']:
            patch.set_facecolor(COLORS.get(m, '#999'))
            patch.set_alpha(0.88)
            if m == FOCAL:
                patch.set_edgecolor(COLORS[FOCAL])
                patch.set_linewidth(1.6)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)

    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi - 0.42, fi + 0.42,
               alpha=0.07, color=COLORS[FOCAL], zorder=0)

    _swatches(ax)
    _clean(ax)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


def draw_recovery_curve(ax, ct: str, ylabel: str = ''):
    """TF-Recovery 曲线（无任何文字标注）"""
    has_any = False
    for m in METHOD_LIST:
        c = TF_REC_CURVES.get(ct, {}).get(m, None)
        if c is None:
            continue
        x, y, _ = c
        has_any  = True
        is_focal = (m == FOCAL)
        ax.plot(x, y,
                color=COLORS.get(m, '#999'),
                linewidth=1.8 if is_focal else 0.7,
                alpha=1.0    if is_focal else 0.50,
                zorder=10    if is_focal else 1)
        if is_focal:
            ax.fill_between(x, y, alpha=0.09,
                            color=COLORS[FOCAL], zorder=0)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)

    ax.set_xlim(0, TOP_N)
    ax.set_ylim(bottom=0)
    ax.set_xlabel('TF Rank', fontsize=6, labelpad=1)
    _clean(ax)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


# ── 通用图例构建 ──────────────────────────────────
def _patch_legend(fig, y_pos: float = 0.01, ncol = None):
    handles = [
        Patch(facecolor=COLORS.get(m, '#999'), label=m,
              edgecolor=COLORS[FOCAL] if m == FOCAL else 'none',
              linewidth=1.5 if m == FOCAL else 0)
        for m in METHOD_LIST
    ]
    handles.append(Patch(facecolor=NO_DATA_COLOR,
                         label='No data', edgecolor='none'))
    fig.legend(
        handles=handles,
        loc='lower center',
        ncol=ncol or len(handles),
        fontsize=6.5,
        bbox_to_anchor=(0.5, y_pos),
        frameon=True, framealpha=0.92, edgecolor='#ccc',
        handlelength=1.0, handletextpad=0.3, columnspacing=0.6
    )


def _line_legend(fig, y_pos: float = -0.04):
    handles = [
        plt.Line2D([0], [0],
                   color=COLORS.get(m, '#999'),
                   linewidth=1.8 if m == FOCAL else 0.9,
                   label=m)
        for m in METHOD_LIST
    ]
    handles.append(
        plt.Line2D([0], [0], color='#C8C8C8',
                   linewidth=0.6, label='No data')
    )
    fig.legend(
        handles=handles,
        loc='lower center',
        ncol=len(handles),
        fontsize=6.5,
        bbox_to_anchor=(0.5, y_pos),
        frameon=True, framealpha=0.92, edgecolor='#ccc',
        handlelength=1.5, columnspacing=0.7
    )


# ── 行标签辅助（不叠加）────────────────────────────
def _row_label(fig, gs_inner, row_i: int, total_rows: int,
               label: str, x_frac: float = 0.015):
    """在 figure 坐标系中写行标签，避免与子图重叠"""
    # 用虚拟 subplot 获取位置，然后立刻移除
    tmp = fig.add_subplot(gs_inner[row_i, 0])
    pos = tmp.get_position()
    tmp.remove()
    y_c = pos.y0 + pos.height / 2
    fig.text(x_frac, y_c, label,
             ha='center', va='center',
             fontsize=7, fontweight='bold', rotation=90,
             transform=fig.transFigure)


# ══════════════════════════════════════════════════
# SUPPLEMENTARY FIGURE 1: TF-Region
# Rows: F₀.₁ Top-50 Box | F₀.₁ All Box |
#       Recall Top-50 Box | Macro F₀.₁ Bar
# ══════════════════════════════════════════════════
print("\n绘制 Supp Fig 1 — TF-Region ...")

S1_ROWS = [
    ('F₀.₁ Top-50\nBoxplot',    'box',     TFR_FSCORE,  TOP_N),
    ('F₀.₁ All\nBoxplot',       'box',     TFR_FSCORE,  None),
    ('Recall Top-50\nBoxplot',  'box',     TFR_RECALL,  TOP_N),
    ('Macro F₀.₁\nBarplot',     'bar_sc',  TFR_MACRO_F, None),
]
NR1 = len(S1_ROWS)

fig = plt.figure(figsize=(A4_H, A4_W * 1.15))
fig.patch.set_facecolor('white')
gs1 = gridspec.GridSpec(NR1, N, figure=fig,
                        left=0.10, right=0.97,
                        top=0.93, bottom=0.10,
                        hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S1_ROWS):
        ax  = fig.add_subplot(gs1[row_i, col_i])
        yl  = S1_ROWS[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar(ax, data, ct, ylabel=yl)

for row_i, (label, *_) in enumerate(S1_ROWS):
    _row_label(fig, gs1, row_i, NR1, label, x_frac=0.015)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 1 — TF-Region Benchmark',
             fontsize=9, fontweight='bold', y=0.97)
fig.text(0.5, 0.945,
         'Ref: Bravo González-Blas et al. Nat Methods 20:1355 (2023)  |  '
         'Wang et al. Nat Methods 20:1368 (2023)',
         ha='center', fontsize=6.5, color='#555')

for ext in ('pdf', 'png'):
    plt.savefig(f"{OUTPUT_DIR}Supp_Fig1_TF_Region.{ext}",
                dpi=300, bbox_inches='tight', format=ext)
plt.close()
print("✓ Supp_Fig1_TF_Region")


# ══════════════════════════════════════════════════
# SUPPLEMENTARY FIGURE 2: Region-Gene
# Rows: Spearman All Bar | Spearman Top-50 Box |
#       F₀.₁ All Bar    | F₀.₁ Top-50 Box     |
#       Recall All Bar
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 2 — Region-Gene ...")

S2_ROWS = [
    ('Spearman |ρ|\nAll Barplot',     'bar_per', RG_SPEARMAN, None),
    ('Spearman |ρ|\nTop-50 Boxplot',  'box',     RG_SPEARMAN, TOP_N),
    ('F₀.₁\nAll Barplot',             'bar_per', RG_FSCORE,   None),
    ('F₀.₁\nTop-50 Boxplot',          'box',     RG_FSCORE,   TOP_N),
    ('Recall\nAll Barplot',           'bar_per', RG_RECALL,   None),
]
NR2 = len(S2_ROWS)

fig = plt.figure(figsize=(A4_H, A4_W * 1.45))
fig.patch.set_facecolor('white')
gs2 = gridspec.GridSpec(NR2, N, figure=fig,
                        left=0.10, right=0.97,
                        top=0.94, bottom=0.07,
                        hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S2_ROWS):
        ax = fig.add_subplot(gs2[row_i, col_i])
        yl = S2_ROWS[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar_per(ax, data, ct, ylabel=yl)

for row_i, (label, *_) in enumerate(S2_ROWS):
    _row_label(fig, gs2, row_i, NR2, label, x_frac=0.015)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 2 — Region-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.975)
fig.text(0.5, 0.955,
         'Ref: Pliner et al. Mol Cell 71:858 (2018)  |  '
         'Kartha et al. Cell Genomics 2:100237 (2022)  |  '
         'Yuan & Duren Nat Biotechnol 43:247 (2025)',
         ha='center', fontsize=6.5, color='#555')

for ext in ('pdf', 'png'):
    plt.savefig(f"{OUTPUT_DIR}Supp_Fig2_Region_Gene.{ext}",
                dpi=300, bbox_inches='tight', format=ext)
plt.close()
print("✓ Supp_Fig2_Region_Gene")


# ══════════════════════════════════════════════════
# SUPPLEMENTARY FIGURE 3: TF-Gene
# Rows: Corr Top-50 Box | Corr All Bar |
#       F-score All Bar | Precision All Bar
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 3 — TF-Gene ...")

S3_ROWS = [
    ('Correlation\nTop-50 Boxplot', 'box',     TFG_CORR,   TOP_N),
    ('Correlation\nAll Barplot',    'bar_per', TFG_CORR,   None),
    ('F-score\nAll Barplot',        'bar_per', TFG_FSCORE, None),
    ('Precision\nAll Barplot',      'bar_per', TFG_PREC,   None),
]
NR3 = len(S3_ROWS)

fig = plt.figure(figsize=(A4_H, A4_W * 1.15))
fig.patch.set_facecolor('white')
gs3 = gridspec.GridSpec(NR3, N, figure=fig,
                        left=0.10, right=0.97,
                        top=0.93, bottom=0.10,
                        hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S3_ROWS):
        ax = fig.add_subplot(gs3[row_i, col_i])
        yl = S3_ROWS[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar_per(ax, data, ct, ylabel=yl)

for row_i, (label, *_) in enumerate(S3_ROWS):
    _row_label(fig, gs3, row_i, NR3, label, x_frac=0.015)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 3 — TF-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.97)
fig.text(0.5, 0.945,
         'Ref: Kamimoto et al. Nature 614:742 (2023)  |  '
         'Pratapa et al. Nat Methods 17:147 (2020)',
         ha='center', fontsize=6.5, color='#555')

for ext in ('pdf', 'png'):
    plt.savefig(f"{OUTPUT_DIR}Supp_Fig3_TF_Gene.{ext}",
                dpi=300, bbox_inches='tight', format=ext)
plt.close()
print("✓ Supp_Fig3_TF_Gene")


# ══════════════════════════════════════════════════
# SUPPLEMENTARY FIGURE 4: TF-Recovery
# Row 0: 恢复曲线
# Row 1: Normalized AUC 柱状图
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 4 — TF-Recovery ...")

fig = plt.figure(figsize=(A4_H, 4.8))
fig.patch.set_facecolor('white')
gs4 = gridspec.GridSpec(
    2, N, figure=fig,
    height_ratios=[2.0, 1.3],
    left=0.08, right=0.97,
    top=0.87, bottom=0.14,
    hspace=0.65, wspace=0.28
)

for col_i, ct in enumerate(AVAIL_CT):
    # 上行：曲线
    ax_c = fig.add_subplot(gs4[0, col_i])
    ax_c.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    draw_recovery_curve(
        ax_c, ct,
        ylabel='TFs Recovered' if col_i == 0 else ''
    )

    # 下行：AUC 柱状图
    ax_a = fig.add_subplot(gs4[1, col_i])
    draw_bar(
        ax_a, TF_REC_AUC, ct,
        ylabel='Norm. AUC' if col_i == 0 else ''
    )

# 行标签
_row_label(fig, gs4, 0, 2, 'Recovery\nCurve',    x_frac=0.012)
_row_label(fig, gs4, 1, 2, 'Norm.\nAUC Bar',     x_frac=0.012)

# 曲线行用线型图例，AUC 行用色块图例 → 统一写一个线型图例
_line_legend(fig, y_pos=0.0)

fig.suptitle('Supplementary Figure 4 — TF-Recovery Benchmark\n'
             'Top: cumulative recovered TFs by rank  |  '
             'Bottom: normalized AUC',
             fontsize=8.5, fontweight='bold', y=1.0)
fig.text(0.5, -0.06,
         'Ref: Bravo González-Blas et al. Nat Methods 20:1355 (2023) [SCENIC+]',
         ha='center', fontsize=6.5, color='#555')

for ext in ('pdf', 'png'):
    plt.savefig(f"{OUTPUT_DIR}Supp_Fig4_TF_Recovery.{ext}",
                dpi=300, bbox_inches='tight', format=ext)
plt.close()
print("✓ Supp_Fig4_TF_Recovery")


# ══════════════════════════════════════════════════
# MAIN FIGURE (A4 portrait)
# Panel A: TF-Region    — F₀.₁ Top-50 Box + Macro F₀.₁ Bar
# Panel B: Region-Gene  — Spearman Top-50 Box + F₀.₁ Bar
# Panel C: TF-Gene      — Corr Top-50 Box + F-score Bar
# Panel D: TF-Recovery  — Curve + AUC Bar
# ══════════════════════════════════════════════════
print("绘制 Main Figure (A4) ...")

fig = plt.figure(figsize=(A4_W, A4_H))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(
    4, 1, figure=fig,
    height_ratios=[2.4, 2.4, 2.4, 3.2],
    left=0.10, right=0.97,
    top=0.955, bottom=0.055,
    hspace=0.50
)

PANEL_LETTERS = ['A', 'B', 'C', 'D']
LAYER_NAMES   = ['TF-Region', 'Region-Gene', 'TF-Gene', 'TF-Recovery']

MAIN_LAYERS = [
    # (sub_rows, [(data, ptype, top_n, ylabel)])
    (2, [
        (TFR_FSCORE,  'box',     TOP_N, 'F₀.₁ Top-50'),
        (TFR_MACRO_F, 'bar_sc',  None,  'Macro F₀.₁'),
    ]),
    (2, [
        (RG_SPEARMAN, 'box',     TOP_N, '|ρ| Top-50'),
        (RG_FSCORE,   'bar_per', None,  'F₀.₁'),
    ]),
    (2, [
        (TFG_CORR,    'box',     TOP_N, '|Corr| Top-50'),
        (TFG_FSCORE,  'bar_per', None,  'F-score'),
    ]),
    (2, [
        (None,        'rec_curve', None, 'TFs Recovered'),
        (TF_REC_AUC,  'bar_sc',   None,  'Norm. AUC'),
    ]),
]

for layer_i, (sub_rows, row_defs) in enumerate(MAIN_LAYERS):
    inner = gridspec.GridSpecFromSubplotSpec(
        sub_rows, N,
        subplot_spec=outer[layer_i],
        hspace=0.65, wspace=0.22
    )

    for sub_i, (data, ptype, tn, short_y) in enumerate(row_defs):
        for col_i, ct in enumerate(AVAIL_CT):
            ax = fig.add_subplot(inner[sub_i, col_i])

            yl = short_y if col_i == 0 else ''
            if sub_i == 0:
                ax.set_title(ct, fontsize=6.8, fontweight='bold', pad=2)

            if ptype == 'box':
                draw_box(ax, data, ct, ylabel=yl, top_n=tn)
            elif ptype == 'bar_sc':
                draw_bar(ax, data, ct, ylabel=yl)
            elif ptype == 'bar_per':
                draw_bar_per(ax, data, ct, ylabel=yl)
            elif ptype == 'rec_curve':
                draw_recovery_curve(ax, ct, ylabel=yl)

    # Panel 字母（左上角）
    tmp = fig.add_subplot(inner[0, 0])
    pos = tmp.get_position()
    tmp.remove()
    fig.text(0.005, pos.y0 + pos.height + 0.005,
             PANEL_LETTERS[layer_i],
             ha='left', va='bottom',
             fontsize=11, fontweight='bold',
             transform=fig.transFigure)

    # 层名（左侧竖排）
    if sub_rows > 1:
        tmp2 = fig.add_subplot(inner[sub_rows - 1, 0])
        pos2 = tmp2.get_position()
        tmp2.remove()
        y_c = (pos.y0 + pos.height + pos2.y0) / 2
    else:
        y_c = pos.y0 + pos.height / 2

    fig.text(0.022, y_c,
             LAYER_NAMES[layer_i],
             ha='center', va='center',
             fontsize=7, fontweight='bold', rotation=90,
             color='#333',
             transform=fig.transFigure)

# 全局图例
_patch_legend(fig, y_pos=0.008)

fig.suptitle(
    'DyGMamba Benchmark — TF-Region, Region-Gene, TF-Gene & TF-Recovery',
    fontsize=9, fontweight='bold', y=0.974)
fig.text(
    0.5, 0.960,
    'Color patches on x-axis indicate methods  |  grey = no data  |  '
    'boxplot: Top-50  |  barplot: mean',
    ha='center', fontsize=6.5, color='#555')

for ext in ('pdf', 'png'):
    plt.savefig(f"{OUTPUT_DIR}Main_Figure_A4.{ext}",
                dpi=300, bbox_inches='tight', format=ext)
plt.close()
print("✓ Main_Figure_A4")


# ══════════════════════════════════════════════════
# 完成
# ══════════════════════════════════════════════════
print(f"\n{'━'*55}")
print("输出文件：")
for fn in ['Main_Figure_A4',
           'Supp_Fig1_TF_Region',
           'Supp_Fig2_Region_Gene',
           'Supp_Fig3_TF_Gene',
           'Supp_Fig4_TF_Recovery']:
    for ext in ('pdf', 'png'):
        fp = f"{OUTPUT_DIR}{fn}.{ext}"
        ok = "✓" if os.path.exists(fp) else "✗"
        print(f"  {ok} {fn}.{ext}")
print(f"\n路径: {OUTPUT_DIR}")
print(f"{'━'*55}")

加载数据...
  ✓ GM12878 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ HepG2 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ IMR90 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ K562 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ MCF7 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ A549 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ H1 Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ HELA Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ SK Recovery: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']

可用细胞系 (9): ['A549', 'GM12878', 'H1', 'HELA', 'HepG2', 'IMR90', 'K562', 'MCF7', 'SK']

绘制 Supp Fig 1 — TF-Region ...
✓ Supp_Fig1_TF_Region
绘制 Supp Fig 2 — Region-Gene ...
✓ Supp_Fig2_Region_Gene
绘制 Supp Fig 3 — TF-Gene ...
✓ Supp_Fig3_TF_Gene
绘制 Supp Fig 4 — TF-Recovery ...
✓ Supp_Fig4_TF_Reco

# V7

In [12]:
"""
paper_benchmark_final_v3.py
Python 3.9 兼容版
修复：
  1. Correlation 增加 Top-50 和 All 两版 Box + Bar
  2. TF-Recovery AUC = raw_auc / max_rank（与曲线一致，DyGMamba 最高）
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch, FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────
# 全局样式
# ─────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'font.size':          7,
    'axes.titlesize':     8,
    'axes.labelsize':     7,
    'xtick.labelsize':    6,
    'ytick.labelsize':    6,
    'axes.linewidth':     0.5,
    'xtick.major.width':  0.4,
    'ytick.major.width':  0.4,
    'xtick.major.size':   2.0,
    'ytick.major.size':   2.0,
    'pdf.fonttype':       42,
    'svg.fonttype':       'none',
})

# ─────────────────────────────────────────────────
# 参数
# ─────────────────────────────────────────────────
DATA_ROOT  = "/home/wuyan/dygmamba_project/data/cell_line/"
OUTPUT_DIR = "/home/wuyan/dygmamba_project/data/benchmark_summary/paper_v3/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CELL_TYPES  = ["GM12878", "HepG2", "IMR90", "K562",
               "MCF7",   "A549",  "H1",   "HELA",  "SK"]
METHOD_LIST = ["DyGMamba", "CellOracle", "FigR",
               "GLUE",     "LINGER",     "GRaNIE", "Pando"]
FOCAL       = "DyGMamba"
TOP_N       = 50

COLORS = {
    "DyGMamba":  "#7B2D8B",
    "CellOracle":"#E87040",
    "FigR":      "#2E4A7C",
    "GLUE":      "#009B77",
    "LINGER":    "#6B7DC0",
    "GRaNIE":    "#5BAD92",
    "Pando":     "#C0392B",
}
NO_DATA_COLOR = "#DCDCDC"
A4_W, A4_H    = 8.27, 11.69

# ══════════════════════════════════════════════════
# 数据读取
# ══════════════════════════════════════════════════
def _read_excel(ct, sheet):
    path = f"{DATA_ROOT}{ct}/benchmarkV7/benchmark_all_results.xlsx"
    if not os.path.exists(path):
        return None
    try:
        xl = pd.ExcelFile(path)
        if sheet not in xl.sheet_names:
            return None
        df = xl.parse(sheet)
        if 'Method' in df.columns:
            df['Method'] = df['Method'].astype(str).str.strip()
        return df
    except Exception:
        return None


def load_per(sheet, col):
    """→ {ct: {method: np.array}}，无数据方法返回空数组"""
    result = {}
    for ct in CELL_TYPES:
        df = _read_excel(ct, sheet)
        d  = {}
        for m in METHOD_LIST:
            if (df is not None
                    and 'Method' in df.columns
                    and col in df.columns):
                v = df[df['Method'] == m][col].dropna().values.astype(float)
            else:
                v = np.array([])
            d[m] = v
        if any(len(v) > 0 for v in d.values()):
            result[ct] = d
    return result


def load_scalar(sheet, col):
    """→ {ct: {method: float or None}}"""
    result = {}
    for ct in CELL_TYPES:
        df = _read_excel(ct, sheet)
        d  = {}
        for m in METHOD_LIST:
            if (df is not None
                    and 'Method' in df.columns
                    and col in df.columns):
                sub = df[df['Method'] == m][col].dropna()
                d[m] = float(sub.iloc[0]) if len(sub) > 0 else None
            else:
                d[m] = None
        if any(v is not None for v in d.values()):
            result[ct] = d
    return result


def reconstruct_recovery(ct, top_n=TOP_N):
    """
    从 TF_recovery_set + count_region_df.pkl 重建曲线
    AUC = raw_auc / max_rank  ← 修复：所有方法同一分母，与曲线高低一致
    """
    set_df   = _read_excel(ct, 'TF_recovery_set')
    pkl_path = f"{DATA_ROOT}{ct}/benchmarkV7/count_region_df.pkl"

    if set_df is None or not os.path.exists(pkl_path):
        return {m: None for m in METHOD_LIST}

    try:
        count_df = pd.read_pickle(pkl_path)
        tf_col   = 'TF'        if 'TF'        in count_df.columns else count_df.columns[0]
        pk_col   = 'PeakCount' if 'PeakCount' in count_df.columns else count_df.columns[-1]
        gt_tfs   = (count_df
                    .sort_values(pk_col, ascending=False)[tf_col]
                    .dropna().tolist())
    except Exception:
        return {m: None for m in METHOD_LIST}

    max_rank = min(top_n, len(gt_tfs))
    curves   = {}

    for m in METHOD_LIST:
        if m not in set_df.columns:
            curves[m] = None
            continue
        tfs = set(set_df[m].dropna().astype(str).tolist())
        if not tfs:
            curves[m] = None
            continue

        x   = np.arange(1, max_rank + 1, dtype=float)
        y   = np.zeros(max_rank, dtype=float)
        cum = 0
        for i in range(max_rank):
            if gt_tfs[i] in tfs:
                cum += 1
            y[i] = cum

        raw_auc = float(np.trapz(y, x))

        # ── 修复核心：所有方法用同一分母 max_rank ──
        # raw_auc / max_rank = 平均每 rank 步恢复的 TF 数
        # 与曲线的高低完全一致（曲线面积大 → AUC 大）
        norm_auc = raw_auc / max_rank

        curves[m] = (x, y, norm_auc)

    return curves


# ── 加载所有指标 ──────────────────────────────────
print("加载数据...")

TFR_FSCORE   = load_per   ('TF_region_per', 'fscore')
TFR_RECALL   = load_per   ('TF_region_per', 'Recall')
TFR_MACRO_F  = load_scalar('TF_region_all', 'F_score')

RG_SPEARMAN  = load_per('Region_gene_per_corr',      'Abs_Spearman_Rho')
RG_FSCORE    = load_per('Region_gene_per_precision', 'F_score')
RG_RECALL    = load_per('Region_gene_per_precision', 'Recall')

TFG_CORR     = load_per('TF_gene_per_corr',      'Correlation')
TFG_FSCORE   = load_per('TF_gene_per_precision', 'F_score')
TFG_PREC     = load_per('TF_gene_per_precision', 'Precision')

# 取绝对值
for _d in [TFR_FSCORE, TFR_RECALL, RG_SPEARMAN, TFG_CORR]:
    for ct in _d:
        _d[ct] = {m: np.abs(v) for m, v in _d[ct].items()}

# TF-Recovery
TF_REC_CURVES = {}
for ct in CELL_TYPES:
    c = reconstruct_recovery(ct)
    if any(v is not None for v in c.values()):
        TF_REC_CURVES[ct] = c
        _present = [m for m, v in c.items() if v is not None]
        print(f"  ✓ {ct}: {_present}")

TF_REC_AUC = {}
for ct in CELL_TYPES:
    if ct not in TF_REC_CURVES:
        continue
    d = {}
    for m in METHOD_LIST:
        c = TF_REC_CURVES[ct].get(m)
        d[m] = float(c[2]) if c is not None else None
    TF_REC_AUC[ct] = d

AVAIL_CT = sorted(set(
    list(TFR_FSCORE.keys()) +
    list(RG_FSCORE.keys())  +
    list(TFG_CORR.keys())
))
N = len(AVAIL_CT)
print(f"\n可用细胞系 ({N}): {AVAIL_CT}")

# ══════════════════════════════════════════════════
# 绘图工具
# ══════════════════════════════════════════════════
SWATCH_H =  0.07
SWATCH_Y = -0.24


def _swatches(ax):
    """x 轴下方色块（替代旋转文字）"""
    for i, m in enumerate(METHOD_LIST):
        rect = FancyBboxPatch(
            (i - 0.39, SWATCH_Y), 0.78, SWATCH_H,
            boxstyle="round,pad=0.01",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
            facecolor=COLORS.get(m, '#999'),
            edgecolor='white', linewidth=0.3,
            zorder=5
        )
        ax.add_patch(rect)
    ax.set_xticks([])
    ax.tick_params(bottom=False)


def _clean(ax):
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(length=2, pad=1)


def draw_bar(ax, scalar_dict, ct, ylabel=''):
    """scalar 柱状图，统一 METHOD_LIST 横坐标"""
    ax.set_xlim(-0.5, len(METHOD_LIST) - 0.5)
    vals, fc, ec, lw = [], [], [], []
    has_any = False

    for m in METHOD_LIST:
        v = scalar_dict.get(ct, {}).get(m, None)
        if v is not None and not np.isnan(float(v)):
            vals.append(float(v))
            fc.append(COLORS.get(m, '#999'))
            has_any = True
        else:
            vals.append(0.0)
            fc.append(NO_DATA_COLOR)
        ec.append(COLORS[FOCAL] if m == FOCAL else 'white')
        lw.append(1.5 if m == FOCAL else 0.2)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
        _swatches(ax); _clean(ax)
        if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)
        return

    vmax = max(v for v in vals if v > 0) if any(v > 0 for v in vals) else 1
    x    = np.arange(len(METHOD_LIST))
    ax.bar(x, vals, color=fc, edgecolor=ec, linewidth=lw,
           width=0.70, alpha=0.90, zorder=3)

    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi - 0.42, fi + 0.42,
               alpha=0.07, color=COLORS[FOCAL], zorder=0)
    ax.set_ylim(0, vmax * 1.28)
    _swatches(ax); _clean(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


def draw_bar_per(ax, per_dict, ct, ylabel='', top_n=None):
    """per-entry 均值 → scalar → draw_bar"""
    scalar = {}
    if ct in per_dict:
        for m in METHOD_LIST:
            v = per_dict[ct].get(m, np.array([]))
            if len(v) > 0:
                if top_n:
                    sv = np.sort(v)[::-1][:min(top_n, len(v))]
                    scalar[m] = float(np.mean(sv))
                else:
                    scalar[m] = float(np.mean(v))
    draw_bar(ax, {ct: scalar}, ct, ylabel=ylabel)


def draw_box(ax, per_dict, ct, ylabel='', top_n=None):
    """箱线图，所有方法统一显示"""
    ax.set_xlim(-0.5, len(METHOD_LIST) - 0.5)
    has_any = False

    for i, m in enumerate(METHOD_LIST):
        v = per_dict.get(ct, {}).get(m, np.array([]))
        if top_n and len(v) > 0:
            v = np.sort(v)[::-1][:min(top_n, len(v))]
        if len(v) == 0:
            ax.plot([i - 0.28, i + 0.28], [0, 0],
                    color='#C8C8C8', linewidth=0.6, zorder=1)
            continue
        has_any = True
        bp = ax.boxplot(
            v, positions=[i], widths=0.58,
            patch_artist=True, manage_ticks=False,
            whiskerprops={'linewidth': 0.6, 'color': '#555'},
            capprops   ={'linewidth': 0.6, 'color': '#555'},
            medianprops ={'color': 'white', 'linewidth': 1.2},
            flierprops  ={'marker': 'o', 'markersize': 1.5,
                          'markerfacecolor': '#999',
                          'alpha': 0.35, 'linewidth': 0},
            boxprops    ={'linewidth': 0.6}
        )
        for patch in bp['boxes']:
            patch.set_facecolor(COLORS.get(m, '#999'))
            patch.set_alpha(0.88)
            if m == FOCAL:
                patch.set_edgecolor(COLORS[FOCAL])
                patch.set_linewidth(1.6)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)

    fi = METHOD_LIST.index(FOCAL)
    ax.axvspan(fi - 0.42, fi + 0.42,
               alpha=0.07, color=COLORS[FOCAL], zorder=0)
    _swatches(ax); _clean(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


def draw_recovery_curve(ax, ct, ylabel=''):
    """TF-Recovery 曲线"""
    has_any = False
    for m in METHOD_LIST:
        c = TF_REC_CURVES.get(ct, {}).get(m, None)
        if c is None:
            continue
        x, y, _ = c
        has_any  = True
        is_f     = (m == FOCAL)
        ax.plot(x, y,
                color=COLORS.get(m, '#999'),
                linewidth=1.8 if is_f else 0.7,
                alpha=1.0    if is_f else 0.50,
                zorder=10    if is_f else 1)
        if is_f:
            ax.fill_between(x, y, alpha=0.09, color=COLORS[FOCAL], zorder=0)

    if not has_any:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                transform=ax.transAxes, color='#bbb', fontsize=7)
    ax.set_xlim(0, TOP_N)
    ax.set_ylim(bottom=0)
    ax.set_xlabel('TF Rank', fontsize=6, labelpad=1)
    _clean(ax)
    if ylabel: ax.set_ylabel(ylabel, fontsize=6.5, labelpad=2)


# ── 图例 ─────────────────────────────────────────
def _patch_legend(fig, y_pos=0.01, ncol=None):
    handles = [
        Patch(facecolor=COLORS.get(m, '#999'), label=m,
              edgecolor=COLORS[FOCAL] if m == FOCAL else 'none',
              linewidth=1.5 if m == FOCAL else 0)
        for m in METHOD_LIST
    ]
    handles.append(Patch(facecolor=NO_DATA_COLOR,
                         label='No data', edgecolor='none'))
    fig.legend(handles=handles, loc='lower center',
               ncol=ncol or len(handles), fontsize=6.5,
               bbox_to_anchor=(0.5, y_pos),
               frameon=True, framealpha=0.92, edgecolor='#ccc',
               handlelength=1.0, handletextpad=0.3, columnspacing=0.6)


def _line_legend(fig, y_pos=-0.04):
    handles = [
        plt.Line2D([0], [0], color=COLORS.get(m, '#999'),
                   linewidth=1.8 if m == FOCAL else 0.9, label=m)
        for m in METHOD_LIST
    ]
    handles.append(plt.Line2D([0], [0], color='#C8C8C8',
                               linewidth=0.6, label='No data'))
    fig.legend(handles=handles, loc='lower center',
               ncol=len(handles), fontsize=6.5,
               bbox_to_anchor=(0.5, y_pos),
               frameon=True, framealpha=0.92, edgecolor='#ccc',
               handlelength=1.5, columnspacing=0.7)


def _row_label(fig, gs_inner, row_i, label, x_frac=0.015):
    tmp = fig.add_subplot(gs_inner[row_i, 0])
    pos = tmp.get_position()
    tmp.remove()
    fig.text(x_frac, pos.y0 + pos.height / 2, label,
             ha='center', va='center', fontsize=7,
             fontweight='bold', rotation=90,
             transform=fig.transFigure)


def _save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f"{OUTPUT_DIR}{name}.{ext}",
                    dpi=300, bbox_inches='tight', format=ext)
    plt.close(fig)
    print(f"✓ {name}")


# ══════════════════════════════════════════════════
# SUPP FIG 1: TF-Region
# F₀.₁ Top-50 Box | F₀.₁ All Box |
# Recall Top-50 Box | Macro F₀.₁ Bar
# ══════════════════════════════════════════════════
print("\n绘制 Supp Fig 1 — TF-Region ...")

S1 = [
    ('F₀.₁ Top-50\nBoxplot',   'box',    TFR_FSCORE,  TOP_N),
    ('F₀.₁ All\nBoxplot',      'box',    TFR_FSCORE,  None),
    ('Recall Top-50\nBoxplot', 'box',    TFR_RECALL,  TOP_N),
    ('Macro F₀.₁\nBarplot',    'bar_sc', TFR_MACRO_F, None),
]
fig = plt.figure(figsize=(A4_H, A4_W * 1.15))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(len(S1), N, figure=fig,
                       left=0.10, right=0.97, top=0.93, bottom=0.10,
                       hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S1):
        ax = fig.add_subplot(gs[row_i, col_i])
        yl = S1[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar(ax, data, ct, ylabel=yl)

for row_i, (label, *_) in enumerate(S1):
    _row_label(fig, gs, row_i, label)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 1 — TF-Region Benchmark',
             fontsize=9, fontweight='bold', y=0.97)
fig.text(0.5, 0.945,
         'Ref: Bravo González-Blas et al. Nat Methods 20:1355 (2023)  |  '
         'Wang et al. Nat Methods 20:1368 (2023)',
         ha='center', fontsize=6.5, color='#555')
_save(fig, 'Supp_Fig1_TF_Region')


# ══════════════════════════════════════════════════
# SUPP FIG 2: Region-Gene
# Spearman Top-50 Box | Spearman All Box |
# Spearman Top-50 Bar | Spearman All Bar |
# F₀.₁ All Bar | Recall All Bar
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 2 — Region-Gene ...")

S2 = [
    ('Spearman |ρ|\nTop-50 Boxplot',  'box',     RG_SPEARMAN, TOP_N),
    ('Spearman |ρ|\nAll Boxplot',     'box',     RG_SPEARMAN, None),
    ('Spearman |ρ|\nTop-50 Barplot',  'bar_per', RG_SPEARMAN, TOP_N),
    ('Spearman |ρ|\nAll Barplot',     'bar_per', RG_SPEARMAN, None),
    ('F₀.₁\nAll Barplot',             'bar_per', RG_FSCORE,   None),
    ('Recall\nAll Barplot',           'bar_per', RG_RECALL,   None),
]
fig = plt.figure(figsize=(A4_H, A4_W * 1.75))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(len(S2), N, figure=fig,
                       left=0.10, right=0.97, top=0.95, bottom=0.05,
                       hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S2):
        ax = fig.add_subplot(gs[row_i, col_i])
        yl = S2[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar_per(ax, data, ct, ylabel=yl, top_n=tn)

for row_i, (label, *_) in enumerate(S2):
    _row_label(fig, gs, row_i, label)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 2 — Region-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.975)
fig.text(0.5, 0.960,
         'Ref: Pliner et al. Mol Cell 71:858 (2018)  |  '
         'Kartha et al. Cell Genomics 2:100237 (2022)  |  '
         'Yuan & Duren Nat Biotechnol 43:247 (2025)',
         ha='center', fontsize=6.5, color='#555')
_save(fig, 'Supp_Fig2_Region_Gene')


# ══════════════════════════════════════════════════
# SUPP FIG 3: TF-Gene
# Corr Top-50 Box | Corr All Box |
# Corr Top-50 Bar | Corr All Bar |
# F-score All Bar | Precision All Bar
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 3 — TF-Gene ...")

S3 = [
    ('Correlation\nTop-50 Boxplot',  'box',     TFG_CORR,   TOP_N),
    ('Correlation\nAll Boxplot',     'box',     TFG_CORR,   None),
    ('Correlation\nTop-50 Barplot',  'bar_per', TFG_CORR,   TOP_N),
    ('Correlation\nAll Barplot',     'bar_per', TFG_CORR,   None),
    ('F-score\nAll Barplot',         'bar_per', TFG_FSCORE, None),
    ('Precision\nAll Barplot',       'bar_per', TFG_PREC,   None),
]
fig = plt.figure(figsize=(A4_H, A4_W * 1.75))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(len(S3), N, figure=fig,
                       left=0.10, right=0.97, top=0.95, bottom=0.05,
                       hspace=0.80, wspace=0.22)

for col_i, ct in enumerate(AVAIL_CT):
    for row_i, (_, ptype, data, tn) in enumerate(S3):
        ax = fig.add_subplot(gs[row_i, col_i])
        yl = S3[row_i][0].split('\n')[0] if col_i == 0 else ''
        if row_i == 0:
            ax.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
        if ptype == 'box':
            draw_box(ax, data, ct, ylabel=yl, top_n=tn)
        else:
            draw_bar_per(ax, data, ct, ylabel=yl, top_n=tn)

for row_i, (label, *_) in enumerate(S3):
    _row_label(fig, gs, row_i, label)

_patch_legend(fig, y_pos=0.01)
fig.suptitle('Supplementary Figure 3 — TF-Gene Benchmark',
             fontsize=9, fontweight='bold', y=0.975)
fig.text(0.5, 0.960,
         'Ref: Kamimoto et al. Nature 614:742 (2023)  |  '
         'Pratapa et al. Nat Methods 17:147 (2020)',
         ha='center', fontsize=6.5, color='#555')
_save(fig, 'Supp_Fig3_TF_Gene')


# ══════════════════════════════════════════════════
# SUPP FIG 4: TF-Recovery
# Row 0: 曲线
# Row 1: AUC 柱状图（= raw_auc / max_rank，与曲线一致）
# ══════════════════════════════════════════════════
print("绘制 Supp Fig 4 — TF-Recovery ...")

fig = plt.figure(figsize=(A4_H, 4.8))
fig.patch.set_facecolor('white')
gs4 = gridspec.GridSpec(2, N, figure=fig,
                        height_ratios=[2.0, 1.3],
                        left=0.08, right=0.97,
                        top=0.87, bottom=0.14,
                        hspace=0.65, wspace=0.28)

for col_i, ct in enumerate(AVAIL_CT):
    ax_c = fig.add_subplot(gs4[0, col_i])
    ax_c.set_title(ct, fontsize=7.5, fontweight='bold', pad=3)
    draw_recovery_curve(ax_c, ct,
                        ylabel='TFs Recovered' if col_i == 0 else '')

    ax_a = fig.add_subplot(gs4[1, col_i])
    draw_bar(ax_a, TF_REC_AUC, ct,
             ylabel='AUC / Rank' if col_i == 0 else '')

_row_label(fig, gs4, 0, 'Recovery\nCurve',  x_frac=0.012)
_row_label(fig, gs4, 1, 'AUC\nBarplot',     x_frac=0.012)

_line_legend(fig, y_pos=0.0)
fig.suptitle('Supplementary Figure 4 — TF-Recovery Benchmark\n'
             'Top: cumulative TFs recovered by rank  |  '
             'Bottom: AUC = ∫curve / max_rank',
             fontsize=8.5, fontweight='bold', y=1.0)
fig.text(0.5, -0.06,
         'Ref: Bravo González-Blas et al. Nat Methods 20:1355 (2023) [SCENIC+]',
         ha='center', fontsize=6.5, color='#555')
_save(fig, 'Supp_Fig4_TF_Recovery')


# ══════════════════════════════════════════════════
# MAIN FIGURE (A4 portrait)
# A: TF-Region    F₀.₁ Top-50 Box + Macro F₀.₁ Bar
# B: Region-Gene  Spearman Top-50 Box + F₀.₁ Bar
# C: TF-Gene      Corr Top-50 Box + F-score Bar
# D: TF-Recovery  Curve + AUC Bar
# ══════════════════════════════════════════════════
print("绘制 Main Figure (A4) ...")

fig = plt.figure(figsize=(A4_W, A4_H))
fig.patch.set_facecolor('white')

outer = gridspec.GridSpec(4, 1, figure=fig,
                          height_ratios=[2.4, 2.4, 2.4, 3.2],
                          left=0.10, right=0.97,
                          top=0.955, bottom=0.055,
                          hspace=0.50)

PANEL_LETTERS = ['A', 'B', 'C', 'D']
LAYER_NAMES   = ['TF-Region', 'Region-Gene', 'TF-Gene', 'TF-Recovery']

MAIN_LAYERS = [
    (2, [
        (TFR_FSCORE,  'box',      TOP_N, 'F₀.₁ Top-50'),
        (TFR_MACRO_F, 'bar_sc',   None,  'Macro F₀.₁'),
    ]),
    (2, [
        (RG_SPEARMAN, 'box',      TOP_N, '|ρ| Top-50'),
        (RG_FSCORE,   'bar_per',  None,  'F₀.₁'),
    ]),
    (2, [
        (TFG_CORR,    'box',      TOP_N, '|Corr| Top-50'),
        (TFG_FSCORE,  'bar_per',  None,  'F-score'),
    ]),
    (2, [
        (None,        'rec_curve',None,  'TFs Recovered'),
        (TF_REC_AUC,  'bar_sc',   None,  'AUC / Rank'),
    ]),
]

for layer_i, (sub_rows, row_defs) in enumerate(MAIN_LAYERS):
    inner = gridspec.GridSpecFromSubplotSpec(
        sub_rows, N,
        subplot_spec=outer[layer_i],
        hspace=0.65, wspace=0.22
    )

    for sub_i, (data, ptype, tn, short_y) in enumerate(row_defs):
        for col_i, ct in enumerate(AVAIL_CT):
            ax = fig.add_subplot(inner[sub_i, col_i])
            yl = short_y if col_i == 0 else ''
            if sub_i == 0:
                ax.set_title(ct, fontsize=6.8, fontweight='bold', pad=2)

            if ptype == 'box':
                draw_box(ax, data, ct, ylabel=yl, top_n=tn)
            elif ptype == 'bar_sc':
                draw_bar(ax, data, ct, ylabel=yl)
            elif ptype == 'bar_per':
                draw_bar_per(ax, data, ct, ylabel=yl)
            elif ptype == 'rec_curve':
                draw_recovery_curve(ax, ct, ylabel=yl)

    # Panel 字母
    tmp = fig.add_subplot(inner[0, 0])
    pos = tmp.get_position()
    tmp.remove()
    fig.text(0.005, pos.y0 + pos.height + 0.005,
             PANEL_LETTERS[layer_i],
             ha='left', va='bottom', fontsize=11, fontweight='bold',
             transform=fig.transFigure)

    # 层名
    if sub_rows > 1:
        tmp2 = fig.add_subplot(inner[sub_rows - 1, 0])
        pos2 = tmp2.get_position()
        tmp2.remove()
        y_c = (pos.y0 + pos.height + pos2.y0) / 2
    else:
        y_c = pos.y0 + pos.height / 2

    fig.text(0.022, y_c, LAYER_NAMES[layer_i],
             ha='center', va='center', fontsize=7,
             fontweight='bold', rotation=90, color='#333',
             transform=fig.transFigure)

_patch_legend(fig, y_pos=0.008)
# fig.suptitle(
#     'DyGMamba Benchmark — TF-Region, Region-Gene, TF-Gene & TF-Recovery',
#     fontsize=9, fontweight='bold', y=0.974)
# fig.text(0.5, 0.960,
#          'Color patches on x-axis = methods  |  grey = no data  |  '
#          'boxplot: Top-50  |  barplot: mean  |  AUC = ∫curve / max_rank',
#          ha='center', fontsize=6.5, color='#555')
_save(fig, 'Main_Figure_A4')


# ── 完成 ─────────────────────────────────────────
print(f"\n{'━'*55}")
print("输出文件:")
for fn in ['Main_Figure_A4',
           'Supp_Fig1_TF_Region',
           'Supp_Fig2_Region_Gene',
           'Supp_Fig3_TF_Gene',
           'Supp_Fig4_TF_Recovery']:
    for ext in ('pdf', 'png'):
        fp  = f"{OUTPUT_DIR}{fn}.{ext}"
        tag = "✓" if os.path.exists(fp) else "✗"
        print(f"  {tag} {fn}.{ext}")
print(f"\n路径: {OUTPUT_DIR}")
print(f"{'━'*55}")

加载数据...
  ✓ GM12878: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ HepG2: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ IMR90: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ K562: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ MCF7: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ A549: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ H1: ['DyGMamba', 'CellOracle', 'FigR', 'GLUE', 'GRaNIE', 'Pando']
  ✓ HELA: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']
  ✓ SK: ['DyGMamba', 'CellOracle', 'FigR', 'GRaNIE', 'Pando']

可用细胞系 (9): ['A549', 'GM12878', 'H1', 'HELA', 'HepG2', 'IMR90', 'K562', 'MCF7', 'SK']

绘制 Supp Fig 1 — TF-Region ...
✓ Supp_Fig1_TF_Region
绘制 Supp Fig 2 — Region-Gene ...
✓ Supp_Fig2_Region_Gene
绘制 Supp Fig 3 — TF-Gene ...
✓ Supp_Fig3_TF_Gene
绘制 Supp Fig 4 — TF-Recovery ...
✓ Supp_Fig4_TF_Recovery
绘制 Main Figure (A4) ...
✓ Main_Figure_A4

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━